# RTC-SDD Research — E02: CE + Phoneme Contrastive Learning

**Status:** Epoch 1 completed, independently arithmetic-audited, and checkpointed to a private Hugging Face repository.  
**Model:** XLS-R 300M + AASIST (original pretrained initialization, **not** E01 fine-tuned weights).  
**Training:** FP32; weighted CE + `0.1 × PCL`; original paired RTCFake Train and Dev; fixed `0.5` evaluation threshold.

> ⚠️ **RESEARCH ARCHIVE — DO NOT CLICK `RUN ALL`.**
>
> This notebook combines **two distinct historical Kaggle sessions** to preserve
> the complete E02 methodology. Phase A constructs/verifies the paired-data,
> phoneme-cache, PCL, and checkpoint-handoff pipeline. Phase B restores that
> exact handoff on a **separate fresh GPU runtime** and resumes Epoch 1.
> Replaying both phases sequentially is invalid and may conflict with existing
> files, re-create a GPU environment, restart training, or overwrite outputs.
> Only inspect or selectively adapt a cell when deliberately reproducing an
> independent new experiment with fresh paths and verified prerequisites.

## Verified result — Epoch 1

| Evaluation | Macro-F1 |
|---|---:|
| Full labeled Dev (n=15,204) | **0.438914** |
| Online Dev (n=7,465) | **0.436917** |
| Offline Dev (n=7,739) | **0.440790** |
| English Dev | 0.451757 |
| Chinese Dev | 0.429883 |

- Weighted Dev loss: `0.6891188856315054`; final optimizer step: **4,737**.
- Spoof recall: **45.29%**; Bonafide recall: **57.99%**.
- Actual Bonafide prevalence: **18.74%**; predicted Bonafide: **55.33%**.
- Always-predict-Spoof reference: All Dev Macro-F1 **0.448311**, above E02's **0.438914**.
- Historical E01 first-epoch control: Online Macro-F1 **0.362177**, All Dev Macro-F1 **0.342422**.
- E02 improved those E01 metrics but still **does not beat the trivial majority baseline**.
- **No Epoch 2 approval**: evaluate class bias, threshold behavior, and E03 protocol first.

**Artifact preservation:** `last.pt`, `best_online_f1.pt`, `best_dev_loss.pt`,
`dev_epoch_001.json`, and `config.json` were reported uploaded to the private
Hugging Face checkpoint repository at `experiments/e02_ce_pcl_fp32/`.
The `last.pt` local SHA-256 matched the remote receipt (the remote checkpoint
bytes themselves were not independently re-downloaded).

**Publication:** Source files and this output-free notebook belong on GitHub;
private dataset contents, cached per-recording phonemes and binary model weights do not.


---

## Phase A — Initial E02 development and phoneme-cache preparation

**Historical Kaggle session:** development branch, paired-data modules,
phoneme model, aligned cache shards, PCL module, first verified optimizer step,
and portable handoff export. The code below is preserved **verbatim** for
methodology and reproducibility; it is not safe to re-run in the finished session.


*The following section captures the original setup-time plan; the verified result and final decision are at the top of this archive.*

# RTC-SDD Research | E02 — CE + PCL

## Experiment Overview

**Experiment ID:** E02

**Objective:** Evaluate the effect of adding Phoneme-Level
Consistency Learning (PCL) to the Weighted Cross-Entropy baseline.

### Shared Experimental Controls

- **Architecture:** XLS-R 300M + AASIST
- **Initialization:** Original pretrained XLS-R 300M
- **Base Loss:** Weighted Cross-Entropy
- **Augmentation:** RawBoost Algorithm 5
- **Precision:** FP32
- **Batch Size:** 2
- **Gradient Accumulation:** 8
- **Learning Rate:** 1e-6
- **Random Seed:** 1234
- **Dataset:** RTCFake Train/Dev

### E02-Specific Component

- **Phoneme-Level Consistency Learning:** Enabled
- **Environmental DNS Noise:** Disabled

PCL implementation and its impact on sample exposure and effective
batch size must be validated before finalizing the training configuration.

---

## 01 — Repository Setup

Restore the private RTC-SDD Research Framework from GitHub using
Kaggle Secrets.

The repository contains the shared training, evaluation,
configuration, and experiment-management infrastructure.

**Execution:** CPU only. No training or package installation.

In [ ]:
# ============================================================
# RTC-SDD RESEARCH | E02 — CE + PCL
# STEP 01: PRIVATE REPOSITORY SETUP
# CPU ONLY | KEEP
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
import subprocess
import base64
import os

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

REPO_URL = (
    "https://github.com/AhmedLabib1/RTC-SDD-Research.git"
)

DATA_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

print("=" * 65)
print("RTC-SDD RESEARCH | E02 — CE + PCL")
print("=" * 65)


# ------------------------------------------------------------
# 1. Restore repository
# ------------------------------------------------------------

if not REPO_DIR.exists():

    print("\nCloning private GitHub repository...")

    github_token = UserSecretsClient().get_secret(
        "GITHUB_TOKEN"
    )

    assert github_token, "GITHUB_TOKEN unavailable."

    # Pass authentication through the Git process environment.
    # Never embed the token in the clone URL or Git configuration.
    credentials = base64.b64encode(
        f"x-access-token:{github_token}".encode()
    ).decode()

    git_env = os.environ.copy()

    git_env["GIT_CONFIG_COUNT"] = "1"
    git_env["GIT_CONFIG_KEY_0"] = (
        "http.https://github.com/.extraheader"
    )
    git_env["GIT_CONFIG_VALUE_0"] = (
        f"AUTHORIZATION: basic {credentials}"
    )
    git_env["GIT_TERMINAL_PROMPT"] = "0"

    try:
        result = subprocess.run(
            [
                "git",
                "clone",
                "--branch", "main",
                "--single-branch",
                "--depth", "1",
                REPO_URL,
                str(REPO_DIR),
            ],
            env=git_env,
            capture_output=True,
            text=True,
        )
    finally:
        git_env.pop("GIT_CONFIG_VALUE_0", None)
        del github_token, credentials

    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("Private repository cloning failed.")

    print("✅ Repository cloned successfully.")

else:
    assert (REPO_DIR / ".git").is_dir(), (
        "Repository path exists but is not a Git repository."
    )

    print("\n✅ Existing repository detected.")
    print("No automatic pull or reset performed.")


# ------------------------------------------------------------
# 2. Git version
# ------------------------------------------------------------

def git_output(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO_DIR), *args],
        text=True,
    ).strip()


print("\n" + "=" * 65)
print("GITHUB VERSION")
print("=" * 65)

print("Branch     :", git_output("branch", "--show-current"))
print("Commit SHA :", git_output("rev-parse", "HEAD"))
print("Clean      :", git_output("status", "--porcelain") == "")


# ------------------------------------------------------------
# 3. Research framework
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FRAMEWORK CHECK")
print("=" * 65)

required_paths = [
    "README.md",
    "docs/experiments.md",
    "docs/training.md",
    "src/rtcsdd/configs",
    "src/rtcsdd/training",
    "src/rtcsdd/data",
    "src/rtcsdd/eval",
    "src/rtcsdd/experiments",
    "scripts/setup",
]

for relative in required_paths:
    path = REPO_DIR / relative

    assert path.exists(), f"Missing: {relative}"
    print(f"✅ {relative}")


# ------------------------------------------------------------
# 4. E02 dataset availability
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("RTCFake DATASET CHECK")
print("=" * 65)

required_data = [
    "train_label.txt",
    "dev_label.txt",
    "train_offline.pack",
    "train_online.pack",
    "dev_offline.pack",
    "dev_online.pack",
    "train_offline_online_pairs.csv",
    "dev_offline_online_pairs.csv",
]

assert DATA_ROOT.is_dir(), "RTCFake dataset missing."

for filename in required_data:
    path = DATA_ROOT / filename

    assert path.is_file(), f"Missing: {filename}"
    print(f"✅ {filename}")


print("\n" + "=" * 65)
print("E02 — INITIAL SETUP COMPLETE")
print("=" * 65)

print("✅ Private Research Framework ready")
print("✅ RTCFake Train/Dev available")
print("✅ Paired manifests available")
print("✅ No configuration changes")
print("✅ No training started")
print("✅ No GPU used")

## 02 — Official RTC-SDD Baseline Setup

Restore the pinned official RTC-SDD baseline source code.

The official XLS-R 300M + AASIST implementation will serve as
the shared architecture for E01 and E02.

For E02, we will inspect the model's intermediate frame-level
representations before implementing Phoneme-Level Consistency Learning.

The official upstream code must remain unchanged.

**Execution:** CPU only. No dependencies installed or model weights loaded.

In [ ]:
# ============================================================
# E02 — OFFICIAL RTC-SDD BASELINE SETUP
# CPU ONLY | KEEP
# ============================================================

from pathlib import Path
import subprocess
import sys
import ast

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

SETUP_SCRIPT = (
    REPO_DIR / "scripts/setup/bootstrap_official.py"
)

OFFICIAL_DIR = (
    REPO_DIR / "third_party/official_rtc_sdd"
)

MODEL_FILE = (
    OFFICIAL_DIR / "xlsr_aasist/model/model.py"
)

assert SETUP_SCRIPT.is_file(), "Official setup script missing."

print("=" * 65)
print("E02 — OFFICIAL BASELINE SETUP")
print("=" * 65)

subprocess.run(
    [sys.executable, str(SETUP_SCRIPT)],
    cwd=str(REPO_DIR),
    check=True,
)

assert MODEL_FILE.is_file(), "Official XLS-R+AASIST model missing."


# ------------------------------------------------------------
# Inspect available model interfaces without importing it
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("OFFICIAL MODEL INTERFACE")
print("=" * 65)

source = MODEL_FILE.read_text(encoding="utf-8")
tree = ast.parse(source)

for node in tree.body:

    if not isinstance(node, ast.ClassDef):
        continue

    if node.name not in {"SSLModel", "Model"}:
        continue

    print(f"\nClass: {node.name}")

    for method in node.body:

        if not isinstance(method, (ast.FunctionDef, ast.AsyncFunctionDef)):
            continue

        args = [
            arg.arg
            for arg in method.args.args
        ]

        print(f"  - {method.name}({', '.join(args)})")


print("\n" + "=" * 65)
print("✅ Official baseline restored")
print("✅ Source commit verified by framework")
print("✅ Model interfaces inspected")
print("✅ No upstream files modified")
print("✅ No dependencies installed")
print("✅ No GPU used")

## 03 — E02 Development Branch

**Experiment:** E02 — Weighted CE + Phoneme-guided Consistency Learning

### Objective

Create an isolated Git development branch for implementing E02.

### Development Strategy

- Preserve the validated E01 baseline.
- Keep the official RTC-SDD implementation unchanged.
- Implement reusable paired-data and PCL components in the research framework.
- Validate all changes before merging them into `main`.
- Do not push experimental code until its tests pass.

**Execution:** CPU only.

**Expected result:** Clean `experiment/e02-pcl` branch.

No training or model downloads are performed.

In [ ]:
# ============================================================
# E02 — CE + PCL
# STEP 08: Create Isolated Development Branch
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

TARGET_BRANCH = "experiment/e02-pcl"

EXPECTED_BASE_COMMIT = (
    "7f2b210ddfaff286a0d497a1c49e4f39a4ae3c6c"
)


def git(*args):
    result = subprocess.run(
        ["git", *args],
        cwd=REPO_DIR,
        text=True,
        capture_output=True,
        check=True,
    )
    return result.stdout.strip()


print("=" * 65)
print("E02 — DEVELOPMENT BRANCH SETUP")
print("=" * 65)


# ------------------------------------------------------------
# 1. Verify repository
# ------------------------------------------------------------

assert REPO_DIR.is_dir(), "Research repository not found!"

current_branch = git("branch", "--show-current")
current_commit = git("rev-parse", "HEAD")
status = git("status", "--porcelain")

print("\n[1] REPOSITORY")
print("Current branch:", current_branch)
print("Current commit:", current_commit)
print("Working tree clean:", not bool(status))

assert not status, (
    "Working tree is not clean. "
    "Inspect existing changes before switching branches."
)


# ------------------------------------------------------------
# 2. Verify expected base
# ------------------------------------------------------------

print("\n[2] BASELINE VERIFICATION")

assert current_commit == EXPECTED_BASE_COMMIT, (
    "Unexpected repository commit. "
    "Review the current source version before proceeding."
)

print("[PASS] Expected E01 framework base")


# ------------------------------------------------------------
# 3. Create or select E02 branch
# ------------------------------------------------------------

print("\n[3] BRANCH SETUP")

if current_branch == TARGET_BRANCH:
    print("[PASS] E02 branch already active")

else:
    assert current_branch == "main", (
        f"Unexpected active branch: {current_branch}"
    )

    existing = git("branch", "--list", TARGET_BRANCH)

    if existing:
        print("[INFO] Existing E02 branch found")
        print(git("switch", TARGET_BRANCH))
    else:
        print(git("switch", "-c", TARGET_BRANCH))
        print("[PASS] E02 branch created")


# ------------------------------------------------------------
# 4. Final verification
# ------------------------------------------------------------

final_branch = git("branch", "--show-current")
final_commit = git("rev-parse", "HEAD")
final_status = git("status", "--porcelain")

assert final_branch == TARGET_BRANCH
assert final_commit == EXPECTED_BASE_COMMIT
assert not final_status

print("\n" + "=" * 65)
print("E02 — BRANCH VERIFICATION")
print("=" * 65)

print("Branch :", final_branch)
print("Commit :", final_commit)
print("Clean  :", not bool(final_status))

print("\n✅ E02 development branch ready.")
print("✅ E01 baseline preserved.")
print("✅ No source files modified.")
print("✅ No changes pushed to GitHub.")
print("✅ No training started.")
print("✅ No GPU used.")

## 04 — Paired Data Module

**Experiment:** E02 — CE + PCL

### Objective

Implement reusable offline–online pair resolution for RTCFake.

### Requirements

- Read official Train/Dev protocols.
- Read offline–online pair manifests.
- Normalize split-prefixed CSV IDs.
- Match pairs against protocol records.
- Validate labels and languages.
- Reject duplicate or missing references.
- Preserve unpaired offline samples for CE training.
- Verify complete protocol coverage.

### Expected Training Data

| Category | Count |
|---|---:|
| Complete offline–online pairs | 37,125 |
| Unpaired offline samples | 1,535 |
| Total CE samples | 75,785 |

The implementation must not modify the private dataset or official RTC-SDD source.

**Execution:** CPU only.

In [ ]:
# ============================================================
# E02 — CE + PCL
# STEP 09: Implement Paired Data Module
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

MODULE_PATH = (
    REPO_DIR / "src/rtcsdd/data/pairs.py"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

EXPECTED_BRANCH = "experiment/e02-pcl"


# ------------------------------------------------------------
# 1. Verify development branch
# ------------------------------------------------------------

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
    text=True,
).strip()

assert branch == EXPECTED_BRANCH, (
    f"Expected {EXPECTED_BRANCH}, found {branch}"
)

print("=" * 65)
print("E02 — PAIRED DATA MODULE IMPLEMENTATION")
print("=" * 65)
print(f"\n[PASS] Development branch: {branch}")


# ------------------------------------------------------------
# 2. Define reusable module
# ------------------------------------------------------------

MODULE_SOURCE = '''\
"""Validated RTCFake offline-online pairing for PCL experiments.

This module only resolves protocol references. Audio loading,
augmentation, batching, and phoneme alignment remain separate.
"""

from __future__ import annotations

import csv
from dataclasses import dataclass
from pathlib import Path
from typing import Sequence

from .protocol import Example, validate_utterance_id


@dataclass(frozen=True)
class PairGroup:
    """One offline recording and its optional online counterpart."""

    offline: Example
    online: Example | None

    @property
    def is_paired(self) -> bool:
        return self.online is not None

    @property
    def members(self) -> tuple[Example, ...]:
        if self.online is None:
            return (self.offline,)
        return (self.offline, self.online)


def _normalize_csv_id(
    raw_value: str,
    *,
    split: str,
    expected_view: str,
    row_number: int,
) -> str:
    """Convert split/view/lang/file.wav to view/lang/file.wav."""

    value = str(raw_value).strip()
    prefix = f"{split}/"

    if not value.startswith(prefix):
        raise ValueError(
            f"Row {row_number}: invalid split prefix"
        )

    normalized = value[len(prefix):]

    try:
        view, _ = validate_utterance_id(
            normalized,
            split=split,
        )
    except ValueError:
        raise ValueError(
            f"Row {row_number}: invalid recording ID"
        ) from None

    if view != expected_view:
        raise ValueError(
            f"Row {row_number}: expected {expected_view} view"
        )

    return normalized


def load_pair_groups(
    csv_path: str | Path,
    protocol_examples: Sequence[Example],
    *,
    split: str,
) -> list[PairGroup]:
    """Resolve and validate every protocol sample exactly once.

    Complete pairs contain two recordings.
    Incomplete pairs contain one offline recording.

    The returned groups preserve CSV row order.
    """

    if split not in {"train", "dev"}:
        raise ValueError(
            "Pair groups are supported only for Train/Dev"
        )

    if not protocol_examples:
        raise ValueError("Protocol examples cannot be empty")

    if any(item.split != split for item in protocol_examples):
        raise ValueError("Protocol split mismatch")

    protocol_index = {
        item.utterance_id: item
        for item in protocol_examples
    }

    if len(protocol_index) != len(protocol_examples):
        raise ValueError("Duplicate protocol IDs")

    groups: list[PairGroup] = []
    seen_ids: set[str] = set()

    with Path(csv_path).open(
        "r",
        encoding="utf-8-sig",
        newline="",
    ) as handle:

        reader = csv.DictReader(handle)

        if not reader.fieldnames:
            raise ValueError("Pair CSV has no header")

        required = {"offline_id", "online_id"}

        if not required.issubset(set(reader.fieldnames)):
            raise ValueError(
                "Pair CSV requires offline_id and online_id"
            )

        for row_number, row in enumerate(reader, start=2):

            if None in row:
                raise ValueError(
                    f"Row {row_number}: unexpected extra columns"
                )

            raw_offline = row.get("offline_id")
            raw_online = row.get("online_id")

            if not raw_offline or not str(raw_offline).strip():
                raise ValueError(
                    f"Row {row_number}: missing offline ID"
                )

            offline_id = _normalize_csv_id(
                raw_offline,
                split=split,
                expected_view="offline",
                row_number=row_number,
            )

            if offline_id in seen_ids:
                raise ValueError(
                    f"Row {row_number}: duplicate offline reference"
                )

            offline = protocol_index.get(offline_id)

            if offline is None:
                raise ValueError(
                    f"Row {row_number}: offline ID absent from protocol"
                )

            seen_ids.add(offline_id)
            online = None

            if raw_online and str(raw_online).strip():

                online_id = _normalize_csv_id(
                    raw_online,
                    split=split,
                    expected_view="online",
                    row_number=row_number,
                )

                if online_id in seen_ids:
                    raise ValueError(
                        f"Row {row_number}: duplicate online reference"
                    )

                online = protocol_index.get(online_id)

                if online is None:
                    raise ValueError(
                        f"Row {row_number}: online ID absent from protocol"
                    )

                if offline.label != online.label:
                    raise ValueError(
                        f"Row {row_number}: paired label mismatch"
                    )

                if offline.language != online.language:
                    raise ValueError(
                        f"Row {row_number}: paired language mismatch"
                    )

                seen_ids.add(online_id)

            groups.append(
                PairGroup(
                    offline=offline,
                    online=online,
                )
            )

    missing = set(protocol_index) - seen_ids

    if missing:
        raise ValueError(
            f"Protocol coverage incomplete: {len(missing)} "
            "recordings were not represented"
        )

    if len(seen_ids) != len(protocol_index):
        raise ValueError("Unexpected protocol coverage mismatch")

    return groups
'''


# ------------------------------------------------------------
# 3. Validate source before writing
# ------------------------------------------------------------

compile(
    MODULE_SOURCE,
    str(MODULE_PATH),
    "exec",
)

print("[PASS] Module syntax validated")


# ------------------------------------------------------------
# 4. Create without overwriting existing changes
# ------------------------------------------------------------

if MODULE_PATH.exists():

    existing = MODULE_PATH.read_text(encoding="utf-8")

    assert existing == MODULE_SOURCE, (
        "pairs.py already exists with different contents. "
        "Inspect it before making changes."
    )

    print("[PASS] Existing identical module preserved")

else:

    assert MODULE_PATH.parent.is_dir()

    MODULE_PATH.write_text(
        MODULE_SOURCE,
        encoding="utf-8",
    )

    print("[PASS] Created:", MODULE_PATH.relative_to(REPO_DIR))


# ------------------------------------------------------------
# 5. Verify against RTCFake protocols
# ------------------------------------------------------------

sys.path.insert(0, str(REPO_DIR / "src"))

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.pairs import load_pair_groups


EXPECTED = {
    "train": {
        "groups": 38660,
        "paired": 37125,
        "unpaired": 1535,
        "samples": 75785,
    },
    "dev": {
        "groups": 7739,
        "paired": 7465,
        "unpaired": 274,
        "samples": 15204,
    },
}

for split in ("train", "dev"):

    records = read_protocol(
        DATASET_ROOT / f"{split}_label.txt",
        split,
    )

    groups = load_pair_groups(
        DATASET_ROOT / f"{split}_offline_online_pairs.csv",
        records,
        split=split,
    )

    paired = sum(group.is_paired for group in groups)
    unpaired = len(groups) - paired
    samples = sum(len(group.members) for group in groups)

    expected = EXPECTED[split]

    assert len(groups) == expected["groups"]
    assert paired == expected["paired"]
    assert unpaired == expected["unpaired"]
    assert samples == expected["samples"]
    assert samples == len(records)

    print(f"\n[{split.upper()}]")
    print(f"Complete pairs    : {paired:,}")
    print(f"Unpaired offline  : {unpaired:,}")
    print(f"Total CE samples  : {samples:,}")
    print("[PASS] Exact protocol coverage")


# ------------------------------------------------------------
# 6. Git status
# ------------------------------------------------------------

status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    text=True,
).strip()

print("\n" + "=" * 65)
print("GIT STATUS")
print("=" * 65)
print(status or "CLEAN")

print("\n✅ Paired Data Module implemented.")
print("✅ Train and Dev integrity verified.")
print("✅ Existing E01 code untouched.")
print("✅ No changes committed or pushed.")
print("✅ No model training started.")
print("✅ No GPU used.")

## 05 — Pair-Preserving Batch Sampler

**Experiment:** E02 — Weighted CE + PCL

### Objective

Create a deterministic batch sampler that keeps paired offline and online
recordings together during training.

### Requirements

- Preserve every training sample exactly once per epoch.
- Keep complete offline–online pairs in the same micro-batch.
- Combine unpaired offline recordings into regular CE batches.
- Maintain micro-batch size of 2, except for the final singleton.
- Support deterministic epoch-level shuffling.
- Support reproducible checkpoint resumption.
- Preserve the E01 optimizer-step budget.

### Implementation

Create `src/rtcsdd/data/paired_batches.py`.

This component builds batch indices only. It does not load audio,
initialize models, or use the GPU.

In [ ]:
# ============================================================
# E02 — CE + PCL
# STEP 10: Implement Pair-Preserving Batch Sampler
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import sys
import math

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

MODULE_PATH = (
    REPO_DIR / "src/rtcsdd/data/paired_batches.py"
)

DATASET_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

EXPECTED_BRANCH = "experiment/e02-pcl"


# ------------------------------------------------------------
# 1. Verify branch and required files
# ------------------------------------------------------------

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
    text=True,
).strip()

assert branch == EXPECTED_BRANCH

assert (
    REPO_DIR / "src/rtcsdd/data/pairs.py"
).is_file(), "STEP 09 module missing!"

print("=" * 65)
print("E02 — PAIR-PRESERVING BATCH SAMPLER")
print("=" * 65)

print(f"\n[PASS] Branch: {branch}")
print("[PASS] Paired data module available")


# ------------------------------------------------------------
# 2. Define reusable batch sampler
# ------------------------------------------------------------

MODULE_SOURCE = '''\
"""Deterministic pair-preserving micro-batches for RTCFake PCL."""

from __future__ import annotations

import random
from collections.abc import Iterator, Sequence

from .pairs import PairGroup
from .protocol import Example


class PairPreservingBatchSampler:
    """Produce index batches without splitting offline-online pairs.

    A complete pair occupies one batch of two recordings.
    Unpaired offline recordings are batched together, two at a time.

    Batch order is deterministic for a given seed and epoch.
    No samples are dropped.
    """

    def __init__(
        self,
        records: Sequence[Example],
        groups: Sequence[PairGroup],
        *,
        seed: int = 1234,
        epoch: int = 0,
    ):

        if not records or not groups:
            raise ValueError("Records and groups cannot be empty")

        if epoch < 0:
            raise ValueError("Epoch must be nonnegative")

        index = {
            record.utterance_id: position
            for position, record in enumerate(records)
        }

        if len(index) != len(records):
            raise ValueError("Duplicate protocol records")

        pair_batches: list[tuple[int, int]] = []
        single_indices: list[int] = []

        pair_ids: set[tuple[str, str]] = set()

        for group in groups:

            offline_idx = index.get(
                group.offline.utterance_id
            )

            if offline_idx is None:
                raise ValueError(
                    "Pair references an unknown offline recording"
                )

            if group.online is None:
                single_indices.append(offline_idx)
                continue

            online_idx = index.get(
                group.online.utterance_id
            )

            if online_idx is None:
                raise ValueError(
                    "Pair references an unknown online recording"
                )

            pair_batches.append(
                (offline_idx, online_idx)
            )

            pair_ids.add(
                (
                    group.offline.utterance_id,
                    group.online.utterance_id,
                )
            )

        # Deterministic RNG independent of global random state.
        rng = random.Random(seed + epoch)

        # Shuffle unpaired recordings before grouping them.
        rng.shuffle(single_indices)

        single_batches = [
            tuple(single_indices[position:position + 2])
            for position in range(0, len(single_indices), 2)
        ]

        batches = pair_batches + single_batches

        # Mix paired batches and CE-only batches.
        rng.shuffle(batches)

        # Every protocol recording must appear exactly once.
        flattened = [
            index_value
            for batch in batches
            for index_value in batch
        ]

        if (
            len(flattened) != len(records)
            or len(set(flattened)) != len(records)
            or set(flattened) != set(range(len(records)))
        ):
            raise ValueError(
                "Batch construction failed exact sample coverage"
            )

        self._batches = tuple(batches)
        self._pair_ids = frozenset(pair_ids)

        self.seed = seed
        self.epoch = epoch
        self.paired_batch_count = len(pair_batches)
        self.ce_only_batch_count = len(single_batches)

    def __len__(self) -> int:
        return len(self._batches)

    def __iter__(self) -> Iterator[list[int]]:
        for batch in self._batches:
            yield list(batch)

    def is_pair_batch(self, utterance_ids: Sequence[str]) -> bool:
        """Check whether IDs form an original ordered pair."""

        return (
            len(utterance_ids) == 2
            and tuple(utterance_ids) in self._pair_ids
        )
'''


# ------------------------------------------------------------
# 3. Syntax validation
# ------------------------------------------------------------

compile(
    MODULE_SOURCE,
    str(MODULE_PATH),
    "exec",
)

print("[PASS] Module syntax validated")


# ------------------------------------------------------------
# 4. Create module safely
# ------------------------------------------------------------

if MODULE_PATH.exists():

    existing = MODULE_PATH.read_text(encoding="utf-8")

    assert existing == MODULE_SOURCE, (
        "Module already exists with different contents."
    )

    print("[PASS] Existing identical module preserved")

else:

    MODULE_PATH.write_text(
        MODULE_SOURCE,
        encoding="utf-8",
    )

    print(
        "[PASS] Created:",
        MODULE_PATH.relative_to(REPO_DIR),
    )


# ------------------------------------------------------------
# 5. Test deterministic batch construction
# ------------------------------------------------------------

sys.path.insert(0, str(REPO_DIR / "src"))

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.pairs import load_pair_groups
from rtcsdd.data.paired_batches import PairPreservingBatchSampler


EXPECTED = {
    "train": {
        "records": 75785,
        "pairs": 37125,
        "singles": 1535,
    },
    "dev": {
        "records": 15204,
        "pairs": 7465,
        "singles": 274,
    },
}

for split in ("train", "dev"):

    records = read_protocol(
        DATASET_ROOT / f"{split}_label.txt",
        split,
    )

    groups = load_pair_groups(
        DATASET_ROOT / f"{split}_offline_online_pairs.csv",
        records,
        split=split,
    )

    sampler = PairPreservingBatchSampler(
        records,
        groups,
        seed=1234,
        epoch=0,
    )

    expected = EXPECTED[split]

    assert len(records) == expected["records"]
    assert sampler.paired_batch_count == expected["pairs"]

    expected_ce_batches = math.ceil(
        expected["singles"] / 2
    )

    assert (
        sampler.ce_only_batch_count
        == expected_ce_batches
    )

    batches = list(sampler)

    # Verify exact one-time exposure of every recording.
    all_indices = [
        index
        for batch in batches
        for index in batch
    ]

    assert len(all_indices) == len(records)
    assert len(set(all_indices)) == len(records)

    # Verify that every identified pair is a real pair.
    detected_pairs = 0

    for batch in batches:

        ids = [
            records[index].utterance_id
            for index in batch
        ]

        if sampler.is_pair_batch(ids):
            assert len(batch) == 2
            detected_pairs += 1

    assert detected_pairs == expected["pairs"]

    # Verify determinism.
    repeated = PairPreservingBatchSampler(
        records,
        groups,
        seed=1234,
        epoch=0,
    )

    assert batches == list(repeated)

    # Verify the next epoch can have a new deterministic order.
    next_epoch = PairPreservingBatchSampler(
        records,
        groups,
        seed=1234,
        epoch=1,
    )

    assert batches != list(next_epoch)

    print(f"\n[{split.upper()}]")
    print(f"Total samples       : {len(records):,}")
    print(f"Paired batches      : {sampler.paired_batch_count:,}")
    print(f"CE-only batches     : {sampler.ce_only_batch_count:,}")
    print(f"Total micro-batches : {len(sampler):,}")

    if split == "train":

        optimizer_steps = math.ceil(len(sampler) / 8)

        print(f"Optimizer steps     : {optimizer_steps:,}")

        assert len(sampler) == 37893
        assert optimizer_steps == 4737

    print("[PASS] All samples appear exactly once")
    print("[PASS] Offline-online pairs preserved")
    print("[PASS] Deterministic epoch shuffling")


# ------------------------------------------------------------
# 6. Git status
# ------------------------------------------------------------

status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    text=True,
).strip()

print("\n" + "=" * 65)
print("GIT STATUS")
print("=" * 65)
print(status or "CLEAN")

print("\n✅ Pair-preserving batch sampler implemented.")
print("✅ Exact sample coverage verified.")
print("✅ E01 optimizer-step budget preserved.")
print("✅ No changes committed or pushed.")
print("✅ No audio decoded.")
print("✅ No training started.")
print("✅ No GPU used.")

## 06 — Phoneme Alignment Setup

### Objective

Prepare the phoneme recognition and alignment pipeline for E02.

### Phoneme Recognition Model

- Model: `facebook/wav2vec2-xlsr-53-espeak-cv-ft`
- Revision: `2c733782da5604684829819a5eb744c193fe9398`
- Purpose: Predict phoneme boundaries for offline and online recordings.
- Training: The phoneme recognition model remains frozen.

### Planned Workflow

1. Verify the phoneme recognition environment.
2. Download the pinned pretrained phoneme model.
3. Validate predictions on representative English and Chinese samples.
4. Match corresponding phoneme segments across offline–online pairs.
5. Verify alignment quality before full-dataset preprocessing.
6. Store reusable alignment metadata for E02 and future robustness experiments.

### Reproducibility

- Keep pretrained model versions pinned.
- Do not modify the original RTC-SDD baseline.
- Do not use Dev recordings to optimize alignment parameters.
- Preserve private recording identifiers and dataset access restrictions.
- Validate the pipeline before processing the full training dataset.

In [ ]:
# ============================================================
# E02 — CE + PCL
# SECTION 06 | CELL 02
# Pinned Phoneme Processor Compatibility
# CPU ONLY | NO MODEL WEIGHTS
# ============================================================

from pathlib import Path
import shutil

from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoProcessor

MODEL_ID = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"

MODEL_REVISION = (
    "2c733782da5604684829819a5eb744c193fe9398"
)

MODEL_META_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

METADATA_FILES = [
    "config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.json",
]

print("=" * 65)
print("E02 — PHONEME PROCESSOR COMPATIBILITY")
print("=" * 65)


# ------------------------------------------------------------
# 1. Download pinned metadata only
# ------------------------------------------------------------

print("\n[1] PINNED MODEL METADATA")

download_path = snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    local_dir=str(MODEL_META_DIR),
    allow_patterns=METADATA_FILES,
    max_workers=2,
)

assert Path(download_path).resolve() == MODEL_META_DIR.resolve()

for filename in METADATA_FILES:

    path = MODEL_META_DIR / filename

    assert path.is_file(), f"Missing: {filename}"

    print(f"[PASS] {filename}")


# ------------------------------------------------------------
# 2. Load model configuration (no weights)
# ------------------------------------------------------------

print("\n[2] MODEL CONFIGURATION")

config = AutoConfig.from_pretrained(
    str(MODEL_META_DIR),
    local_files_only=True,
)

print("Architecture :", config.model_type)
print("Hidden size  :", getattr(config, "hidden_size", None))
print("Vocabulary   :", getattr(config, "vocab_size", None))
print("CTC blank ID :", getattr(config, "pad_token_id", None))


# ------------------------------------------------------------
# 3. Load processor (no neural network)
# ------------------------------------------------------------

print("\n[3] PROCESSOR COMPATIBILITY")

processor = AutoProcessor.from_pretrained(
    str(MODEL_META_DIR),
    local_files_only=True,
    do_phonemize=False,
)

assert processor.tokenizer.do_phonemize is False

print("Processor :", type(processor).__name__)
print("Tokenizer :", type(processor.tokenizer).__name__)

sample_rate = processor.feature_extractor.sampling_rate

print("Sample rate:", sample_rate)

assert sample_rate == 16000, (
    f"Unexpected sample rate: {sample_rate}"
)

assert processor.tokenizer.pad_token_id is not None

print(
    "Tokenizer blank/pad ID:",
    processor.tokenizer.pad_token_id,
)


# ------------------------------------------------------------
# 4. Storage verification
# ------------------------------------------------------------

print("\n[4] STORAGE")

metadata_size = sum(
    path.stat().st_size
    for path in MODEL_META_DIR.rglob("*")
    if path.is_file()
)

print(
    f"Metadata storage: "
    f"{metadata_size / (1024 ** 2):.3f} MiB"
)

disk = shutil.disk_usage("/kaggle/working")

print(
    f"Free disk: "
    f"{disk.free / (1024 ** 3):.2f} GiB"
)


# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PHONEME PROCESSOR STATUS")
print("=" * 65)

print("✅ Pinned model metadata downloaded.")
print("✅ Model configuration loaded.")
print("✅ Phoneme processor initialized.")
print("✅ Sampling rate validated.")
print("✅ No model weights downloaded.")
print("✅ No source files modified.")
print("✅ No training started.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — CE + PCL
# SECTION 06 | CELL 03
# Download and Verify Pinned Phoneme Weights
# CPU ONLY | NO MODEL INITIALIZATION
# ============================================================

from pathlib import Path
from huggingface_hub import (
    HfApi,
    hf_hub_download,
    hf_hub_url,
    get_hf_file_metadata,
)
import hashlib
import re
import shutil

MODEL_ID = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"

MODEL_REVISION = (
    "2c733782da5604684829819a5eb744c193fe9398"
)

MODEL_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

WEIGHT_NAME = "pytorch_model.bin"
WEIGHT_PATH = MODEL_DIR / WEIGHT_NAME

print("=" * 65)
print("E02 — PRETRAINED PHONEME WEIGHTS SETUP")
print("=" * 65)


# ------------------------------------------------------------
# 1. Verify existing model metadata
# ------------------------------------------------------------

required_files = [
    "config.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "vocab.json",
]

print("\n[1] MODEL METADATA")

for filename in required_files:

    assert (MODEL_DIR / filename).is_file(), (
        f"Required metadata missing: {filename}"
    )

    print(f"[PASS] {filename}")


# ------------------------------------------------------------
# 2. Obtain pinned weight metadata
# ------------------------------------------------------------

print("\n[2] OFFICIAL WEIGHT METADATA")

api = HfApi()

info = api.model_info(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    files_metadata=True,
)

assert info.sha == MODEL_REVISION, (
    "Model revision mismatch!"
)

weight_info = next(
    (
        item for item in info.siblings
        if item.rfilename == WEIGHT_NAME
    ),
    None,
)

assert weight_info is not None, (
    "Pinned weight file not found!"
)

expected_size = getattr(weight_info, "size", None)

lfs_metadata = getattr(weight_info, "lfs", None) or {}

if expected_size is None:
    expected_size = lfs_metadata.get("size")

assert isinstance(expected_size, int) and expected_size > 0

expected_sha256 = lfs_metadata.get("sha256")

# Fallback: use the Hub file ETag when it is a SHA256 digest.
if not expected_sha256:

    file_url = hf_hub_url(
        repo_id=MODEL_ID,
        filename=WEIGHT_NAME,
        revision=MODEL_REVISION,
    )

    file_metadata = get_hf_file_metadata(file_url)

    etag = str(file_metadata.etag or "").strip('"')

    if re.fullmatch(r"[0-9a-fA-F]{64}", etag):
        expected_sha256 = etag.lower()

assert expected_sha256, (
    "Expected SHA256 unavailable. "
    "Stop before downloading an unverified checkpoint."
)

expected_sha256 = expected_sha256.lower()

print("Model revision :", MODEL_REVISION)
print("Weight file    :", WEIGHT_NAME)

print(
    f"Expected size  : "
    f"{expected_size / (1024 ** 3):.3f} GiB"
)

print("Expected SHA256:", expected_sha256)


# ------------------------------------------------------------
# 3. Verify storage capacity
# ------------------------------------------------------------

print("\n[3] STORAGE CAPACITY")

disk = shutil.disk_usage("/kaggle/working")

free_gib = disk.free / (1024 ** 3)

print(f"Available disk: {free_gib:.2f} GiB")

# Reserve 4 GiB beyond the weight download.
required_bytes = expected_size + 4 * (1024 ** 3)

assert disk.free >= required_bytes, (
    "Insufficient storage headroom for this download."
)

print("[PASS] Download storage headroom")


# ------------------------------------------------------------
# 4. Download weights only if missing
# ------------------------------------------------------------

print("\n[4] PRETRAINED WEIGHT DOWNLOAD")

if WEIGHT_PATH.is_file():

    print("[INFO] Existing weight file detected")
    print("[INFO] Will verify before reuse")

else:

    print("[DOWNLOAD] Starting pinned checkpoint download...")

    downloaded = hf_hub_download(
        repo_id=MODEL_ID,
        filename=WEIGHT_NAME,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )

    assert Path(downloaded).resolve() == WEIGHT_PATH.resolve()

    print("[PASS] Download completed")


# ------------------------------------------------------------
# 5. Verify file size and SHA256
# ------------------------------------------------------------

print("\n[5] CHECKPOINT INTEGRITY")

assert WEIGHT_PATH.is_file()

actual_size = WEIGHT_PATH.stat().st_size

assert actual_size == expected_size, (
    "Downloaded checkpoint size mismatch!"
)

print("[PASS] File size verified")

sha256 = hashlib.sha256()

with WEIGHT_PATH.open("rb") as handle:

    while True:

        chunk = handle.read(8 * 1024 * 1024)

        if not chunk:
            break

        sha256.update(chunk)

actual_sha256 = sha256.hexdigest()

assert actual_sha256 == expected_sha256, (
    "Downloaded checkpoint SHA256 mismatch!"
)

print("[PASS] SHA256 verified")


# ------------------------------------------------------------
# 6. Final report
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PRETRAINED PHONEME CHECKPOINT READY")
print("=" * 65)

print("Model:", MODEL_ID)
print("Path :", WEIGHT_PATH)

print(
    f"Size : {actual_size / (1024 ** 3):.3f} GiB"
)

print("SHA256:", actual_sha256)

disk = shutil.disk_usage("/kaggle/working")

print(
    f"Remaining disk: "
    f"{disk.free / (1024 ** 3):.2f} GiB"
)

print("\n✅ Original phoneme model weights ready.")
print("✅ Checkpoint integrity verified.")
print("✅ No training started.")
print("✅ No model initialized.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — CE + PCL
# SECTION 06 | CELL 04
# Phoneme Model Loading and CPU Smoke Test
# CPU ONLY | NO TRAINING
# ============================================================

from pathlib import Path
import numpy as np
import torch

from transformers import AutoProcessor, AutoModelForCTC

MODEL_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

WEIGHT_PATH = MODEL_DIR / "pytorch_model.bin"

assert WEIGHT_PATH.is_file(), (
    "Pinned phoneme weights not found!"
)

DEVICE = torch.device("cpu")

print("=" * 65)
print("E02 — PHONEME MODEL CPU SMOKE TEST")
print("=" * 65)


# ------------------------------------------------------------
# 1. Load processor
# ------------------------------------------------------------

print("\n[1] PROCESSOR")

pcl_processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    do_phonemize=False,
)

assert pcl_processor.tokenizer.do_phonemize is False
assert pcl_processor.feature_extractor.sampling_rate == 16000

print("[PASS] Pinned processor ready")


# ------------------------------------------------------------
# 2. Load pretrained model
# ------------------------------------------------------------

print("\n[2] MODEL INITIALIZATION")
print("[INFO] Loading pretrained weights on CPU...")

pcl_phoneme_model = AutoModelForCTC.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)

pcl_phoneme_model.to(DEVICE)
pcl_phoneme_model.eval()

# Frozen phoneme recognizer — inference only.
for parameter in pcl_phoneme_model.parameters():
    parameter.requires_grad_(False)

total_params = sum(
    parameter.numel()
    for parameter in pcl_phoneme_model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in pcl_phoneme_model.parameters()
    if parameter.requires_grad
)

assert trainable_params == 0

print("Device            :", DEVICE)
print("Total parameters  :", f"{total_params:,}")
print("Trainable params  :", trainable_params)
print("Model mode        :", "EVAL")

print("[PASS] Pretrained phoneme model initialized")


# ------------------------------------------------------------
# 3. Create synthetic audio
# ------------------------------------------------------------

print("\n[3] SYNTHETIC AUDIO")

sample_rate = 16000
duration_seconds = 1.0

time_axis = np.arange(
    sample_rate,
    dtype=np.float32,
) / sample_rate

synthetic_audio = (
    0.05 * np.sin(2 * np.pi * 220 * time_axis)
).astype(np.float32)

inputs = pcl_processor(
    synthetic_audio,
    sampling_rate=sample_rate,
    return_tensors="pt",
)

inputs = {
    key: value.to(DEVICE)
    for key, value in inputs.items()
}

print("Audio duration:", duration_seconds, "second")
print("Input shape   :", tuple(inputs["input_values"].shape))


# ------------------------------------------------------------
# 4. CPU forward pass
# ------------------------------------------------------------

print("\n[4] PHONEME INFERENCE")

with torch.inference_mode():

    output = pcl_phoneme_model(**inputs)

    logits = output.logits

assert logits.ndim == 3
assert logits.shape[0] == 1
assert logits.shape[-1] == pcl_phoneme_model.config.vocab_size

assert torch.isfinite(logits).all().item(), (
    "Non-finite phoneme logits detected!"
)

predicted_ids = logits.argmax(dim=-1)

blank_id = pcl_phoneme_model.config.pad_token_id

blank_frames = int(
    (predicted_ids == blank_id).sum().item()
)

total_frames = predicted_ids.numel()

print("Logits shape      :", tuple(logits.shape))
print("CTC frames        :", total_frames)
print("Vocabulary size   :", logits.shape[-1])
print("CTC blank ID      :", blank_id)
print("Blank frames      :", blank_frames)
print("Non-blank frames  :", total_frames - blank_frames)

print("[PASS] Forward pass completed")
print("[PASS] Logits are finite")


# ------------------------------------------------------------
# 5. Final verification
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PHONEME MODEL STATUS")
print("=" * 65)

print("✅ Original pretrained weights loaded.")
print("✅ Model operating in evaluation mode.")
print("✅ All phoneme model parameters frozen.")
print("✅ CPU forward pass completed.")
print("✅ CTC output dimensions validated.")
print("✅ No model training started.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — CE + PCL
# SECTION 06 | CELL 06
# Implement Phoneme Alignment Engine
# CPU ONLY | NO MODEL INFERENCE
# ============================================================

from pathlib import Path
import subprocess
import sys

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

MODULE_PATH = (
    REPO_DIR / "src/rtcsdd/data/phoneme_alignment.py"
)

EXPECTED_BRANCH = "experiment/e02-pcl"

branch = subprocess.check_output(
    ["git", "branch", "--show-current"],
    cwd=REPO_DIR,
    text=True,
).strip()

assert branch == EXPECTED_BRANCH

print("=" * 65)
print("E02 — PHONEME ALIGNMENT ENGINE")
print("=" * 65)
print(f"[PASS] Development branch: {branch}")


# ------------------------------------------------------------
# 1. Define reusable alignment module
# ------------------------------------------------------------

MODULE_SOURCE = '''\
"""CTC phoneme segmentation and monotonic pair alignment.

Pure-Python implementation. No torch dependency or model loading.

CTC frame IDs are converted to contiguous non-blank segments.
Offline and online segments are matched using an ordered
Longest Common Subsequence (LCS) alignment.
"""

from __future__ import annotations

from dataclasses import dataclass
from collections.abc import Collection, Sequence


@dataclass(frozen=True)
class CTCSegment:
    """A predicted token spanning [start_frame, end_frame)."""

    token_id: int
    start_frame: int
    end_frame: int


@dataclass(frozen=True)
class MatchedPhoneme:
    """Corresponding offline-online phoneme segments."""

    token_id: int
    offline: CTCSegment
    online: CTCSegment


def ctc_token_segments(
    frame_token_ids: Sequence[int],
    *,
    blank_id: int,
    excluded_ids: Collection[int] = (),
) -> list[CTCSegment]:
    """Extract CTC segments, retaining their frame boundaries.

    Repeated tokens separated by a blank remain separate segments.
    """

    if not frame_token_ids:
        return []

    excluded = set(excluded_ids)
    tokens = [int(value) for value in frame_token_ids]

    if any(value < 0 for value in tokens):
        raise ValueError("CTC token IDs must be nonnegative")

    segments: list[CTCSegment] = []

    start = 0
    previous = tokens[0]

    for frame in range(1, len(tokens) + 1):

        current = (
            tokens[frame]
            if frame < len(tokens)
            else None
        )

        if current == previous:
            continue

        if (
            previous is not None
            and previous != blank_id
            and previous not in excluded
        ):
            segments.append(
                CTCSegment(
                    token_id=previous,
                    start_frame=start,
                    end_frame=frame,
                )
            )

        previous = current
        start = frame

    return segments


def align_phoneme_segments(
    offline: Sequence[CTCSegment],
    online: Sequence[CTCSegment],
) -> tuple[MatchedPhoneme, ...]:
    """Match identical tokens while preserving temporal order.

    The dynamic program computes an LCS over phoneme token IDs.
    Unmatched insertions and deletions are skipped.
    """

    n = len(offline)
    m = len(online)

    if n == 0 or m == 0:
        return ()

    dp = [
        [0] * (m + 1)
        for _ in range(n + 1)
    ]

    for i in range(n - 1, -1, -1):

        for j in range(m - 1, -1, -1):

            if offline[i].token_id == online[j].token_id:

                dp[i][j] = 1 + dp[i + 1][j + 1]

            else:

                dp[i][j] = max(
                    dp[i + 1][j],
                    dp[i][j + 1],
                )

    matches: list[MatchedPhoneme] = []

    i = 0
    j = 0

    while i < n and j < m:

        if offline[i].token_id == online[j].token_id:

            matches.append(
                MatchedPhoneme(
                    token_id=offline[i].token_id,
                    offline=offline[i],
                    online=online[j],
                )
            )

            i += 1
            j += 1

        elif dp[i + 1][j] >= dp[i][j + 1]:

            i += 1

        else:

            j += 1

    return tuple(matches)


def alignment_similarity(
    matched_count: int,
    offline_count: int,
    online_count: int,
) -> float:
    """Symmetric token-sequence matching score."""

    denominator = offline_count + online_count

    if denominator == 0:
        return 0.0

    if matched_count < 0 or 2 * matched_count > denominator:
        raise ValueError("Invalid alignment statistics")

    return 2.0 * matched_count / denominator
'''


# ------------------------------------------------------------
# 2. Validate source before writing
# ------------------------------------------------------------

compile(
    MODULE_SOURCE,
    str(MODULE_PATH),
    "exec",
)

print("[PASS] Module syntax validated")


# ------------------------------------------------------------
# 3. Create module safely
# ------------------------------------------------------------

if MODULE_PATH.exists():

    existing = MODULE_PATH.read_text(encoding="utf-8")

    assert existing == MODULE_SOURCE, (
        "Existing module contains different contents. "
        "Inspect changes before overwriting."
    )

    print("[PASS] Existing identical module preserved")

else:

    assert MODULE_PATH.parent.is_dir()

    MODULE_PATH.write_text(
        MODULE_SOURCE,
        encoding="utf-8",
    )

    print(
        "[PASS] Created:",
        MODULE_PATH.relative_to(REPO_DIR),
    )


# ------------------------------------------------------------
# 4. Synthetic CTC segmentation tests
# ------------------------------------------------------------

sys.path.insert(0, str(REPO_DIR / "src"))

from rtcsdd.data.phoneme_alignment import (
    ctc_token_segments,
    align_phoneme_segments,
    alignment_similarity,
)

print("\n[1] CTC SEGMENTATION TEST")

frame_ids = [
    0, 7, 7, 0, 7, 0, 3, 3, 0
]

segments = ctc_token_segments(
    frame_ids,
    blank_id=0,
)

assert len(segments) == 3

assert [
    (s.token_id, s.start_frame, s.end_frame)
    for s in segments
] == [
    (7, 1, 3),
    (7, 4, 5),
    (3, 6, 8),
]

print("[PASS] CTC blank removal")
print("[PASS] Repeated phonemes separated correctly")
print("[PASS] Frame boundaries preserved")


# ------------------------------------------------------------
# 5. Synthetic offline-online alignment test
# ------------------------------------------------------------

print("\n[2] MONOTONIC ALIGNMENT TEST")

offline_frames = [
    0, 7, 7, 0, 3, 3, 0, 9, 9, 0
]

online_frames = [
    0, 7, 7, 0, 5, 5, 0, 3, 3, 0, 9, 9, 0
]

offline_segments = ctc_token_segments(
    offline_frames,
    blank_id=0,
)

online_segments = ctc_token_segments(
    online_frames,
    blank_id=0,
)

matches = align_phoneme_segments(
    offline_segments,
    online_segments,
)

matched_tokens = [
    match.token_id
    for match in matches
]

assert matched_tokens == [7, 3, 9]

similarity = alignment_similarity(
    len(matches),
    len(offline_segments),
    len(online_segments),
)

assert abs(similarity - (6 / 7)) < 1e-10

print("Offline tokens :", len(offline_segments))
print("Online tokens  :", len(online_segments))
print("Matched tokens :", len(matches))
print(f"Similarity     : {similarity:.3f}")

print("[PASS] Inserted phoneme handled")
print("[PASS] Matching order preserved")
print("[PASS] Alignment similarity validated")


# ------------------------------------------------------------
# 6. Empty-sequence safety test
# ------------------------------------------------------------

print("\n[3] EMPTY-SEQUENCE SAFETY")

empty = ctc_token_segments(
    [0, 0, 0, 0],
    blank_id=0,
)

assert empty == []
assert align_phoneme_segments(empty, online_segments) == ()
assert alignment_similarity(0, 0, 0) == 0.0

print("[PASS] All-blank CTC output handled")
print("[PASS] Empty alignment handled safely")


# ------------------------------------------------------------
# 7. Git status
# ------------------------------------------------------------

status = subprocess.check_output(
    ["git", "status", "--short"],
    cwd=REPO_DIR,
    text=True,
).strip()

print("\n" + "=" * 65)
print("GIT STATUS")
print("=" * 65)
print(status or "CLEAN")

print("\n✅ Phoneme Alignment Engine implemented.")
print("✅ Synthetic alignment tests passed.")
print("✅ E01 source files untouched.")
print("✅ No changes committed or pushed.")
print("✅ No model inference executed.")
print("✅ No training started.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — GPU PHONEME MODEL SMOKE TEST
# KEEP
# GPU 0 — INFERENCE ONLY
# ============================================================

from pathlib import Path
import hashlib
import time

import numpy as np
import torch

from transformers import AutoProcessor, AutoModelForCTC


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

MODEL_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

WEIGHTS_PATH = MODEL_DIR / "pytorch_model.bin"

EXPECTED_SHA256 = (
    "04366b6c8d24099ef313cf02f0e58d26"
    "f5dddfda16edbfc8eb2c713d94a9f551"
)

DEVICE = torch.device("cuda:0")

assert torch.cuda.is_available(), "CUDA unavailable"
assert WEIGHTS_PATH.is_file(), "Phoneme checkpoint missing"

print("=" * 65)
print("E02 — GPU PHONEME MODEL SMOKE TEST")
print("=" * 65)


# ------------------------------------------------------------
# 2. Checkpoint integrity
# ------------------------------------------------------------

print("\n[1] CHECKPOINT INTEGRITY")

sha = hashlib.sha256()

with WEIGHTS_PATH.open("rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        sha.update(chunk)

actual_sha256 = sha.hexdigest()

assert actual_sha256 == EXPECTED_SHA256, (
    "Checkpoint SHA256 mismatch"
)

print("[PASS] Original checkpoint SHA256 verified")


# ------------------------------------------------------------
# 3. Initialize processor
# ------------------------------------------------------------

print("\n[2] PHONEME PROCESSOR")

phoneme_gpu_processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    do_phonemize=False,
)

assert (
    phoneme_gpu_processor.feature_extractor.sampling_rate
    == 16000
)

print("[PASS] Processor initialized")
print("[PASS] Sample rate: 16000 Hz")


# ------------------------------------------------------------
# 4. Initialize frozen model
# ------------------------------------------------------------

print("\n[3] GPU MODEL INITIALIZATION")

print("[INFO] Loading original weights...")

phoneme_gpu_model = AutoModelForCTC.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    use_safetensors=False,
)

phoneme_gpu_model.requires_grad_(False)
phoneme_gpu_model.eval()

phoneme_gpu_model = phoneme_gpu_model.to(DEVICE)

total_params = sum(
    p.numel() for p in phoneme_gpu_model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in phoneme_gpu_model.parameters()
    if p.requires_grad
)

assert trainable_params == 0
assert not phoneme_gpu_model.training

print("Device           :", next(phoneme_gpu_model.parameters()).device)
print("Total parameters :", f"{total_params:,}")
print("Trainable params :", trainable_params)
print("Model mode       :", "EVAL")

print("[PASS] Frozen model loaded on GPU")


# ------------------------------------------------------------
# 5. Synthetic audio
# ------------------------------------------------------------

print("\n[4] SYNTHETIC AUDIO")

sample_rate = 16000

synthetic_audio = np.zeros(
    sample_rate,
    dtype=np.float32,
)

inputs = phoneme_gpu_processor(
    synthetic_audio,
    sampling_rate=sample_rate,
    return_tensors="pt",
)

input_values = inputs.input_values.to(DEVICE)

print("Audio duration :", "1.0 second")
print("Input shape    :", tuple(input_values.shape))


# ------------------------------------------------------------
# 6. GPU forward pass
# ------------------------------------------------------------

print("\n[5] GPU PHONEME INFERENCE")

torch.cuda.reset_peak_memory_stats(DEVICE)
torch.cuda.synchronize(DEVICE)

start = time.perf_counter()

with torch.inference_mode():
    outputs = phoneme_gpu_model(input_values)
    logits = outputs.logits

torch.cuda.synchronize(DEVICE)

elapsed = time.perf_counter() - start

assert logits.ndim == 3
assert logits.shape[0] == 1
assert logits.shape[-1] == 392
assert torch.isfinite(logits).all().item()

predicted_ids = torch.argmax(logits, dim=-1)

blank_id = phoneme_gpu_model.config.pad_token_id

blank_frames = int(
    (predicted_ids == blank_id).sum().item()
)

nonblank_frames = int(
    (predicted_ids != blank_id).sum().item()
)

print("Logits shape       :", tuple(logits.shape))
print("CTC frames         :", logits.shape[1])
print("Vocabulary size    :", logits.shape[2])
print("CTC blank ID       :", blank_id)
print("Blank frames       :", blank_frames)
print("Non-blank frames   :", nonblank_frames)
print("Inference time     :", f"{elapsed:.3f} seconds")

print("[PASS] GPU forward pass completed")
print("[PASS] All logits finite")


# ------------------------------------------------------------
# 7. GPU memory
# ------------------------------------------------------------

print("\n[6] GPU MEMORY")

allocated_gib = (
    torch.cuda.memory_allocated(DEVICE) / (1024 ** 3)
)

peak_gib = (
    torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3)
)

gpu_name = torch.cuda.get_device_name(DEVICE)

print("GPU              :", gpu_name)
print("Allocated VRAM   :", f"{allocated_gib:.3f} GiB")
print("Peak VRAM        :", f"{peak_gib:.3f} GiB")


# ------------------------------------------------------------
# 8. Summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("GPU PHONEME MODEL STATUS")
print("=" * 65)

print("✅ Original pretrained weights verified.")
print("✅ Phoneme processor initialized.")
print("✅ Model successfully loaded on GPU 0.")
print("✅ All parameters frozen.")
print("✅ GPU forward pass completed.")
print("✅ CTC output dimensions validated.")
print("✅ No model training started.")
print("✅ No dataset preprocessing started.")

In [ ]:
# ============================================================
# E02 — RESUMABLE PHONEME CACHE MODULE
# KEEP
# CPU ONLY
# ============================================================

from pathlib import Path
import subprocess
import tempfile
import importlib.util

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

MODULE_PATH = (
    REPO_DIR / "src/rtcsdd/data/phoneme_cache.py"
)

EXPECTED_COMMIT = (
    "25f5a6ded1db9da26f8fdfb915bd3f71716f7eb9"
)


# ------------------------------------------------------------
# 1. Source-control safety
# ------------------------------------------------------------

def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()


assert git("branch", "--show-current") == "experiment/e02-pcl"
assert git("rev-parse", "HEAD") == EXPECTED_COMMIT
assert not git("status", "--porcelain")

assert not MODULE_PATH.exists(), (
    "Cache module already exists. Refusing to overwrite."
)

print("=" * 65)
print("E02 — RESUMABLE PHONEME CACHE MODULE")
print("=" * 65)

print("[PASS] Correct development branch")
print("[PASS] Clean working tree")
print("[PASS] No existing module to overwrite")


# ------------------------------------------------------------
# 2. Module source
# ------------------------------------------------------------

MODULE_SOURCE = r'''
"""E02 PCL alignment shard cache — schema v1."""

from __future__ import annotations

import gzip
import hashlib
import json
import os
import re
import uuid
from pathlib import Path

SCHEMA_VERSION = 1

REQUIRED_META = {
    "schema_version",
    "split",
    "start",
    "stop",
    "manifest_sha256",
    "model_sha256",
    "sample_rate",
    "audio_length",
    "crop",
    "pad",
    "excluded_ids",
}


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def pair_key_sha256(
    offline_id: str,
    online_id: str,
) -> str:
    """Stable digest without storing recording IDs."""

    raw = (
        offline_id.encode("utf-8")
        + b"\x00"
        + online_id.encode("utf-8")
    )

    return _sha256_bytes(raw)


def shard_path(
    root: str | Path,
    split: str,
    start: int,
    stop: int,
) -> Path:

    if split not in {"train", "dev"}:
        raise ValueError("Invalid split")

    if not 0 <= start < stop:
        raise ValueError("Invalid shard bounds")

    return (
        Path(root)
        / split
        / f"{split}_{start:06d}_{stop:06d}.json.gz"
    )


def validate_payload(
    metadata: dict,
    records: list[dict],
) -> None:

    if not REQUIRED_META.issubset(metadata):
        raise ValueError("Missing metadata fields")

    if metadata["schema_version"] != SCHEMA_VERSION:
        raise ValueError("Unsupported schema version")

    if metadata["split"] not in {"train", "dev"}:
        raise ValueError("Unsupported split")

    start = int(metadata["start"])
    stop = int(metadata["stop"])

    if start < 0 or stop <= start:
        raise ValueError("Invalid coverage bounds")

    if len(records) != stop - start:
        raise ValueError("Shard coverage mismatch")

    if (
        int(metadata["sample_rate"]) != 16000
        or int(metadata["audio_length"]) != 64600
    ):
        raise ValueError("Unexpected audio configuration")

    if (
        metadata["crop"] != "left"
        or metadata["pad"] != "repeat"
    ):
        raise ValueError("Unexpected crop/pad settings")

    for name in ("manifest_sha256", "model_sha256"):
        if not re.fullmatch(
            r"[0-9a-f]{64}",
            metadata[name],
        ):
            raise ValueError(f"Invalid {name}")

    if not isinstance(metadata["excluded_ids"], list):
        raise ValueError("Invalid excluded_ids")

    for index, record in enumerate(records, start):

        expected_fields = {
            "pair_index",
            "pair_hash",
            "offline_frames",
            "online_frames",
            "matches",
        }

        if set(record) != expected_fields:
            raise ValueError("Unexpected record fields")

        if record["pair_index"] != index:
            raise ValueError("Non-contiguous pair indices")

        if not re.fullmatch(
            r"[0-9a-f]{64}",
            record["pair_hash"],
        ):
            raise ValueError("Invalid pair hash")

        off_n = record["offline_frames"]
        on_n = record["online_frames"]

        if (
            not isinstance(off_n, int)
            or not isinstance(on_n, int)
            or min(off_n, on_n) <= 0
        ):
            raise ValueError("Invalid frame counts")

        previous_off = -1
        previous_on = -1

        for match in record["matches"]:

            if (
                not isinstance(match, list)
                or len(match) != 5
                or not all(
                    isinstance(value, int)
                    for value in match
                )
            ):
                raise ValueError("Malformed match")

            token, os_, oe, ns, ne = match

            if not (
                0 <= token < 392
                and 0 <= os_ < oe <= off_n
                and 0 <= ns < ne <= on_n
            ):
                raise ValueError(
                    "Invalid token or phoneme frame span"
                )

            if (
                os_ <= previous_off
                or ns <= previous_on
            ):
                raise ValueError(
                    "Non-monotonic matching"
                )

            previous_off = os_
            previous_on = ns


def write_shard(
    root: str | Path,
    metadata: dict,
    records: list[dict],
) -> tuple[Path, str]:
    """
    Atomically write one complete shard.

    Existing outputs are never overwritten.
    """

    validate_payload(metadata, records)

    path = shard_path(
        root,
        metadata["split"],
        metadata["start"],
        metadata["stop"],
    )

    checksum_path = Path(
        str(path) + ".sha256"
    )

    if path.exists() or checksum_path.exists():
        raise FileExistsError(
            "Shard already exists; inspect before resuming"
        )

    payload = {
        "metadata": metadata,
        "records": records,
    }

    raw = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    ).encode("utf-8")

    compressed = gzip.compress(
        raw,
        compresslevel=6,
        mtime=0,
    )

    digest = _sha256_bytes(compressed)

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = path.with_name(
        path.name + f".{uuid.uuid4().hex}.partial"
    )

    temporary_checksum = checksum_path.with_name(
        checksum_path.name
        + f".{uuid.uuid4().hex}.partial"
    )

    try:
        with temporary_path.open("xb") as f:
            f.write(compressed)
            f.flush()
            os.fsync(f.fileno())

        with temporary_checksum.open(
            "x",
            encoding="ascii",
        ) as f:
            f.write(digest + "\n")
            f.flush()
            os.fsync(f.fileno())

        os.replace(temporary_path, path)
        os.replace(temporary_checksum, checksum_path)

    finally:
        # Only remove temporary files created by this call.
        temporary_path.unlink(missing_ok=True)
        temporary_checksum.unlink(missing_ok=True)

    return path, digest


def read_shard(
    path: str | Path,
    *,
    expected_metadata: dict | None = None,
) -> dict:

    path = Path(path)

    checksum_path = Path(
        str(path) + ".sha256"
    )

    if (
        not path.is_file()
        or not checksum_path.is_file()
    ):
        raise FileNotFoundError(
            "Incomplete shard or missing checksum"
        )

    expected_sha = checksum_path.read_text(
        encoding="ascii"
    ).strip()

    compressed = path.read_bytes()

    actual_sha = _sha256_bytes(compressed)

    if (
        not re.fullmatch(
            r"[0-9a-f]{64}",
            expected_sha,
        )
        or actual_sha != expected_sha
    ):
        raise ValueError(
            "Shard SHA256 verification failed"
        )

    payload = json.loads(
        gzip.decompress(compressed).decode("utf-8")
    )

    validate_payload(
        payload["metadata"],
        payload["records"],
    )

    metadata = payload["metadata"]

    expected_name = shard_path(
        ".",
        metadata["split"],
        metadata["start"],
        metadata["stop"],
    ).name

    if path.name != expected_name:
        raise ValueError(
            "Shard filename and metadata disagree"
        )

    if (
        expected_metadata is not None
        and metadata != expected_metadata
    ):
        raise ValueError(
            "Cache metadata mismatch"
        )

    return payload
'''


# ------------------------------------------------------------
# 3. Validate syntax BEFORE creating the file
# ------------------------------------------------------------

compile(
    MODULE_SOURCE,
    str(MODULE_PATH),
    "exec",
)

print("\n[1] MODULE SYNTAX")
print("[PASS] Python syntax validated")


# ------------------------------------------------------------
# 4. Create module
# ------------------------------------------------------------

MODULE_PATH.write_text(
    MODULE_SOURCE,
    encoding="utf-8",
)

print("\n[2] MODULE CREATION")
print("[PASS] Created:", MODULE_PATH.relative_to(REPO_DIR))


# ------------------------------------------------------------
# 5. Import module for tests
# ------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "e02_phoneme_cache_test",
    MODULE_PATH,
)

cache_module = importlib.util.module_from_spec(spec)

spec.loader.exec_module(cache_module)

print("\n[3] MODULE IMPORT")
print("[PASS] Cache module imported")


# ------------------------------------------------------------
# 6. Synthetic round-trip tests
# ------------------------------------------------------------

print("\n[4] SYNTHETIC CACHE TESTS")

metadata = {
    "schema_version": 1,
    "split": "train",
    "start": 0,
    "stop": 2,
    "manifest_sha256": "a" * 64,
    "model_sha256": "b" * 64,
    "sample_rate": 16000,
    "audio_length": 64600,
    "crop": "left",
    "pad": "repeat",
    "excluded_ids": [1, 2, 3],
}

records = [
    {
        "pair_index": 0,
        "pair_hash": "c" * 64,
        "offline_frames": 201,
        "online_frames": 201,
        "matches": [
            [3, 10, 12, 11, 13],
            [5, 20, 24, 22, 26],
        ],
    },
    {
        "pair_index": 1,
        "pair_hash": "d" * 64,
        "offline_frames": 201,
        "online_frames": 201,
        "matches": [],
    },
]

with tempfile.TemporaryDirectory(
    prefix="e02_pcl_cache_test_"
) as temporary_dir:

    shard_file, digest = cache_module.write_shard(
        temporary_dir,
        metadata,
        records,
    )

    recovered = cache_module.read_shard(
        shard_file,
        expected_metadata=metadata,
    )

    assert recovered["records"] == records

    print("[PASS] Atomic shard writing")
    print("[PASS] SHA256 verification")
    print("[PASS] Exact record round-trip")
    print("[PASS] Empty-match record supported")

    try:
        cache_module.write_shard(
            temporary_dir,
            metadata,
            records,
        )
    except FileExistsError:
        print("[PASS] Existing shard overwrite blocked")
    else:
        raise AssertionError(
            "Overwrite protection failed"
        )


# ------------------------------------------------------------
# 7. Git verification
# ------------------------------------------------------------

print("\n[5] GIT STATUS")

status = git("status", "--short")

print(status)

assert (
    "src/rtcsdd/data/phoneme_cache.py"
    in status
)

print("\n" + "=" * 65)
print("E02 — PHONEME CACHE MODULE READY")
print("=" * 65)

print("✅ Cache writer implemented.")
print("✅ Cache reader implemented.")
print("✅ SHA256 verification implemented.")
print("✅ Atomic shard writing implemented.")
print("✅ Existing-file protection implemented.")
print("✅ Synthetic tests passed.")
print("✅ E01 files untouched.")
print("✅ No changes committed or pushed.")
print("✅ No dataset preprocessing started.")
print("✅ No model training started.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — FIRST REAL PHONEME CACHE SHARD
# KEEP
# GPU 0 — PREPROCESSING ONLY
# ============================================================

from pathlib import Path
import subprocess
import hashlib
import json
import time
import statistics
import sys

import numpy as np
import torch

REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

DATASET_ROOT = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

CACHE_ROOT = Path(
    "/kaggle/working/e02_pcl_cache/v1"
)

MODEL_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.dataset import RTCFakePackedDataset
from rtcsdd.data.pairs import load_pair_groups
from rtcsdd.data import phoneme_cache

from rtcsdd.data.phoneme_alignment import (
    ctc_token_segments,
    align_phoneme_segments,
    alignment_similarity,
)


DEVICE = torch.device("cuda:0")

SHARD_START = 0
SHARD_STOP = 256

EXPECTED_COMMIT = (
    "25f5a6ded1db9da26f8fdfb915bd3f71716f7eb9"
)

EXPECTED_MANIFEST_SHA256 = (
    "e109aefdc01e7b44e8de6615d9732bab"
    "8f6222e1ebf57a2d6e5bd548a5446f00"
)

EXPECTED_MODEL_SHA256 = (
    "04366b6c8d24099ef313cf02f0e58d26"
    "f5dddfda16edbfc8eb2c713d94a9f551"
)


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()


def file_sha256(path):
    sha = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            sha.update(block)

    return sha.hexdigest()


print("=" * 70)
print("E02 — FIRST REAL PHONEME CACHE SHARD")
print("=" * 70)


# ------------------------------------------------------------
# 1. Source safety
# ------------------------------------------------------------

print("\n[1] SOURCE SAFETY")

assert git("branch", "--show-current") == "experiment/e02-pcl"
assert git("rev-parse", "HEAD") == EXPECTED_COMMIT

status = git(
    "status",
    "--porcelain",
    "--untracked-files=all",
)

assert status.strip() == (
    "?? src/rtcsdd/data/phoneme_cache.py"
), "Unexpected source changes"

print("[PASS] Correct E02 branch")
print("[PASS] Expected source commit")
print("[PASS] Only new cache module is untracked")


# ------------------------------------------------------------
# 2. Frozen GPU model
# ------------------------------------------------------------

print("\n[2] GPU MODEL")

assert torch.cuda.is_available()
assert "phoneme_gpu_model" in globals()
assert "phoneme_gpu_processor" in globals()

model = phoneme_gpu_model
processor = phoneme_gpu_processor

assert next(model.parameters()).device == DEVICE
assert not model.training
assert all(not p.requires_grad for p in model.parameters())

blank_id = int(model.config.pad_token_id)

excluded_ids = sorted(
    set(processor.tokenizer.all_special_ids)
    - {blank_id}
)

print("[PASS] Frozen phoneme model")
print("[PASS] GPU:", torch.cuda.get_device_name(DEVICE))
print("CTC blank ID:", blank_id)
print("Excluded IDs:", len(excluded_ids))


# ------------------------------------------------------------
# 3. Configuration and checkpoint identity
# ------------------------------------------------------------

print("\n[3] PREPROCESSING IDENTITY")

config_path = (
    REPO_DIR
    / "src/rtcsdd/configs/"
    "p01_xlsr_aasist_official_like_fp32.json"
)

with config_path.open("r", encoding="utf-8") as f:
    config = json.load(f)

data_cfg = config["data"]

sample_rate = int(data_cfg["sample_rate"])
audio_length = int(data_cfg["audio_length"])

assert sample_rate == 16000
assert audio_length == 64600
assert data_cfg["crop"] == "left"
assert data_cfg["pad"] == "repeat"

manifest_path = (
    DATASET_ROOT / "train_offline_online_pairs.csv"
)

assert (
    file_sha256(manifest_path)
    == EXPECTED_MANIFEST_SHA256
)

checkpoint_path = MODEL_DIR / "pytorch_model.bin"

assert (
    file_sha256(checkpoint_path)
    == EXPECTED_MODEL_SHA256
)

alignment_module_hash = file_sha256(
    REPO_DIR / "src/rtcsdd/data/phoneme_alignment.py"
)

print("[PASS] Audio configuration")
print("[PASS] Train pair manifest SHA256")
print("[PASS] Phoneme checkpoint SHA256")
print("[PASS] Alignment module fingerprint")


# ------------------------------------------------------------
# 4. Load first 256 complete pairs
# ------------------------------------------------------------

print("\n[4] PAIRED DATA")

train_records = read_protocol(
    DATASET_ROOT / "train_label.txt",
    "train",
)

all_groups = load_pair_groups(
    manifest_path,
    train_records,
    split="train",
)

complete_pairs = [
    group for group in all_groups
    if group.online is not None
]

assert len(complete_pairs) == 37125

shard_pairs = complete_pairs[
    SHARD_START:SHARD_STOP
]

assert len(shard_pairs) == 256

print("Total complete pairs:", len(complete_pairs))
print("Selected pair range :", SHARD_START, SHARD_STOP)
print("Selected pairs      :", len(shard_pairs))


# ------------------------------------------------------------
# 5. Cache metadata and destination
# ------------------------------------------------------------

print("\n[5] CACHE METADATA")

metadata = {
    "schema_version": 1,
    "split": "train",
    "start": SHARD_START,
    "stop": SHARD_STOP,
    "manifest_sha256": EXPECTED_MANIFEST_SHA256,
    "model_sha256": EXPECTED_MODEL_SHA256,
    "sample_rate": sample_rate,
    "audio_length": audio_length,
    "crop": "left",
    "pad": "repeat",
    "excluded_ids": excluded_ids,
    "phoneme_model_id": (
        "facebook/wav2vec2-xlsr-53-espeak-cv-ft"
    ),
    "phoneme_revision": (
        "2c733782da5604684829819a5eb744c193fe9398"
    ),
    "framework_commit": EXPECTED_COMMIT,
    "alignment_module_sha256": alignment_module_hash,
    "audio_policy": "fixed-length-no-rawboost",
}

shard_path = phoneme_cache.shard_path(
    CACHE_ROOT,
    "train",
    SHARD_START,
    SHARD_STOP,
)

checksum_path = Path(
    str(shard_path) + ".sha256"
)

print("Split:", metadata["split"])
print("Range:", SHARD_START, SHARD_STOP)
print("Shard filename:", shard_path.name)


# ------------------------------------------------------------
# 6. Process or validate existing shard
# ------------------------------------------------------------

if shard_path.exists() or checksum_path.exists():

    print("\n[6] EXISTING SHARD")

    print("[INFO] Existing shard detected")
    print("[INFO] Validating without overwriting")

    recovered = phoneme_cache.read_shard(
        shard_path,
        expected_metadata=metadata,
    )

    assert len(recovered["records"]) == 256

    print("[PASS] Existing shard integrity verified")

else:

    print("\n[6] REAL GPU PREPROCESSING")

    examples = []

    for group in shard_pairs:
        assert group.offline.label == group.online.label

        examples.extend([
            group.offline,
            group.online,
        ])

    audio_dataset = RTCFakePackedDataset(
        examples,
        DATASET_ROOT,
        length=audio_length,
        crop="left",
        pad="repeat",
        rawboost={"enabled": False},
        seed=1234,
    )

    assert len(audio_dataset) == 512

    torch.cuda.reset_peak_memory_stats(DEVICE)

    def extract_segments(index):

        audio, label, _ = audio_dataset[index]

        waveform = (
            audio.detach()
            .cpu()
            .numpy()
            .astype(np.float32, copy=False)
            .reshape(-1)
        )

        assert len(waveform) == audio_length
        assert np.isfinite(waveform).all()

        inputs = processor(
            waveform,
            sampling_rate=sample_rate,
            return_tensors="pt",
        )

        values = inputs.input_values.to(DEVICE)

        with torch.inference_mode():
            logits = model(values).logits

        assert torch.isfinite(logits).all().item()

        frame_ids = (
            logits.argmax(dim=-1)[0]
            .detach()
            .cpu()
            .tolist()
        )

        frame_count = len(frame_ids)

        segments = ctc_token_segments(
            frame_ids,
            blank_id=blank_id,
            excluded_ids=excluded_ids,
        )

        # Cache schema requires half-open, non-empty spans.
        for segment in segments:
            assert (
                0
                <= segment.start_frame
                < segment.end_frame
                <= frame_count
            ), "Invalid CTC frame span"

        return segments, frame_count


    records = []
    similarities = []

    start_time = time.perf_counter()

    for pair_index, group in enumerate(shard_pairs):

        offline_segments, offline_frames = extract_segments(
            2 * pair_index
        )

        online_segments, online_frames = extract_segments(
            2 * pair_index + 1
        )

        matched = align_phoneme_segments(
            offline_segments,
            online_segments,
        )

        matches = []

        for match in matched:
            matches.append([
                int(match.token_id),
                int(match.offline.start_frame),
                int(match.offline.end_frame),
                int(match.online.start_frame),
                int(match.online.end_frame),
            ])

        similarity = alignment_similarity(
            len(matched),
            len(offline_segments),
            len(online_segments),
        )

        similarities.append(similarity)

        record = {
            "pair_index": SHARD_START + pair_index,
            "pair_hash": phoneme_cache.pair_key_sha256(
                str(group.offline.utterance_id),
                str(group.online.utterance_id),
            ),
            "offline_frames": int(offline_frames),
            "online_frames": int(online_frames),
            "matches": matches,
        }

        records.append(record)

        if (pair_index + 1) % 32 == 0:
            print(
                f"[PROGRESS] {pair_index + 1}/256 pairs"
            )

    torch.cuda.synchronize(DEVICE)

    elapsed = time.perf_counter() - start_time

    print("\n[7] PRE-WRITE VALIDATION")

    phoneme_cache.validate_payload(
        metadata,
        records,
    )

    assert len(records) == 256

    print("[PASS] All records validated")

    print("\n[8] ATOMIC SHARD WRITE")

    saved_path, saved_hash = phoneme_cache.write_shard(
        CACHE_ROOT,
        metadata,
        records,
    )

    print("[PASS] Shard written atomically")
    print("[PASS] SHA256:", saved_hash)

    recovered = phoneme_cache.read_shard(
        saved_path,
        expected_metadata=metadata,
    )

    assert recovered["records"] == records

    print("[PASS] Exact disk round-trip")

    print("\n[9] PERFORMANCE")

    print("Pairs processed:", len(records))
    print("Recordings:", len(records) * 2)
    print("Processing time:", f"{elapsed:.2f} seconds")
    print(
        "Mean similarity:",
        f"{statistics.mean(similarities):.3f}"
    )
    print(
        "Peak GPU VRAM:",
        f"{torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 3):.3f} GiB"
    )


# ------------------------------------------------------------
# 7. Verify recovered pair identities
# ------------------------------------------------------------

print("\n[10] FINAL CACHE INTEGRITY")

cached_records = recovered["records"]

assert len(cached_records) == 256

for index, (group, record) in enumerate(
    zip(shard_pairs, cached_records)
):

    assert record["pair_index"] == SHARD_START + index

    expected_pair_hash = phoneme_cache.pair_key_sha256(
        str(group.offline.utterance_id),
        str(group.online.utterance_id),
    )

    assert record["pair_hash"] == expected_pair_hash

assert shard_path.is_file()
assert checksum_path.is_file()

print("[PASS] SHA256 verification")
print("[PASS] Exact pair ordering")
print("[PASS] All pair hashes verified")
print("[PASS] 256 records recovered")

print(
    "Compressed shard size:",
    f"{shard_path.stat().st_size / (1024 ** 2):.3f} MiB"
)


# ------------------------------------------------------------
# 8. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E02 — FIRST REAL CACHE SHARD COMPLETE")
print("=" * 70)

print("✅ First 256 complete pairs cached.")
print("✅ 512 real recordings represented.")
print("✅ Real phoneme alignments stored.")
print("✅ Exact disk round-trip verified.")
print("✅ Cache SHA256 verified.")
print("✅ Pair identities verified.")
print("✅ Existing output overwrite prevented.")
print("✅ No source files modified.")
print("✅ No GitHub push performed.")
print("✅ No Hugging Face upload performed.")
print("✅ No model training started.")

In [ ]:
# ============================================================
# E02 — FIRST PHONEME CACHE SHARD HF BACKUP
# KEEP
# CPU ONLY
# ============================================================

from pathlib import Path
from kaggle_secrets import UserSecretsClient
from huggingface_hub import (
    HfApi,
    CommitOperationAdd,
    hf_hub_download,
)

import subprocess
import hashlib
import tempfile


REPO_DIR = Path("/kaggle/working/RTC-SDD-Research")

CACHE_ROOT = Path(
    "/kaggle/working/e02_pcl_cache/v1"
)

HF_REPO_ID = (
    "AhmedLabib1/rtc-sdd-research-checkpoints"
)

HF_REPO_TYPE = "model"

HF_PREFIX = "preprocessing/e02_pcl_cache/v1/train"

SHARD_NAME = "train_000000_000256.json.gz"

SHARD_PATH = CACHE_ROOT / "train" / SHARD_NAME

CHECKSUM_PATH = Path(
    str(SHARD_PATH) + ".sha256"
)

EXPECTED_SHARD_SHA256 = (
    "3f6b76a0c202692687707576266c9e8"
    "ede151e8f2cc9a171ca3bfb4fc532b5b1"
)


def sha256_file(path):
    sha = hashlib.sha256()

    with Path(path).open("rb") as f:
        for chunk in iter(
            lambda: f.read(8 * 1024 * 1024),
            b"",
        ):
            sha.update(chunk)

    return sha.hexdigest()


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_DIR), *args],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()


print("=" * 70)
print("E02 — FIRST PHONEME CACHE SHARD HF BACKUP")
print("=" * 70)


# ------------------------------------------------------------
# 1. Source verification
# ------------------------------------------------------------

print("\n[1] RESEARCH FRAMEWORK")

assert git("branch", "--show-current") == "experiment/e02-pcl"

commit = git("rev-parse", "HEAD")

assert commit.startswith("41a6b3e1d104")

assert not git("status", "--porcelain")

print("[PASS] E02 branch verified")
print("[PASS] Commit:", commit[:12])
print("[PASS] Working tree clean")


# ------------------------------------------------------------
# 2. Local cache integrity
# ------------------------------------------------------------

print("\n[2] LOCAL CACHE INTEGRITY")

assert SHARD_PATH.is_file()
assert CHECKSUM_PATH.is_file()

actual_sha = sha256_file(SHARD_PATH)

stored_sha = CHECKSUM_PATH.read_text(
    encoding="ascii"
).strip()

assert actual_sha == EXPECTED_SHARD_SHA256
assert stored_sha == EXPECTED_SHARD_SHA256

from rtcsdd.data.phoneme_cache import read_shard

payload = read_shard(SHARD_PATH)

assert payload["metadata"]["schema_version"] == 1
assert payload["metadata"]["split"] == "train"
assert payload["metadata"]["start"] == 0
assert payload["metadata"]["stop"] == 256

assert len(payload["records"]) == 256

print("[PASS] Shard SHA256 verified")
print("[PASS] Checksum sidecar verified")
print("[PASS] Cache schema validated")
print("[PASS] 256 records recovered")


# ------------------------------------------------------------
# 3. Hugging Face authentication
# ------------------------------------------------------------

print("\n[3] HUGGING FACE AUTHENTICATION")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

assert hf_token and hf_token.strip()

hf_token = hf_token.strip()

api = HfApi(token=hf_token)

repo_info = api.repo_info(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
)

assert repo_info.private is True, (
    "HF repository is not private. Upload blocked."
)

print("[PASS] Repository accessible")
print("[PASS] Repository is private")


# ------------------------------------------------------------
# 4. Remote destination safety
# ------------------------------------------------------------

print("\n[4] REMOTE DESTINATION")

remote_shard = f"{HF_PREFIX}/{SHARD_NAME}"

remote_checksum = (
    f"{HF_PREFIX}/{SHARD_NAME}.sha256"
)

remote_files = set(
    api.list_repo_files(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
    )
)

shard_exists = remote_shard in remote_files
checksum_exists = remote_checksum in remote_files

if shard_exists != checksum_exists:
    raise RuntimeError(
        "Partial remote backup detected. "
        "Manual review required before continuing."
    )

print("Remote shard:", remote_shard)
print("Remote checksum:", remote_checksum)


# ------------------------------------------------------------
# 5. Upload both files in one commit
# ------------------------------------------------------------

print("\n[5] HUGGING FACE BACKUP")

if shard_exists and checksum_exists:

    print("[INFO] Both files already exist remotely")
    print("[INFO] Skipping upload; verifying content")

else:

    operations = [
        CommitOperationAdd(
            path_in_repo=remote_shard,
            path_or_fileobj=str(SHARD_PATH),
        ),
        CommitOperationAdd(
            path_in_repo=remote_checksum,
            path_or_fileobj=str(CHECKSUM_PATH),
        ),
    ]

    commit_info = api.create_commit(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        operations=operations,
        commit_message=(
            "e02: backup first verified PCL alignment shard"
        ),
    )

    print("[PASS] Both files uploaded")
    print("[PASS] Remote commit created")


# ------------------------------------------------------------
# 6. Download verification
# ------------------------------------------------------------

print("\n[6] REMOTE DOWNLOAD VERIFICATION")

with tempfile.TemporaryDirectory(
    prefix="e02_hf_verify_"
) as verify_dir:

    downloaded_shard = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=remote_shard,
        token=hf_token,
        cache_dir=verify_dir,
    )

    downloaded_checksum = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=remote_checksum,
        token=hf_token,
        cache_dir=verify_dir,
    )

    remote_shard_hash = sha256_file(downloaded_shard)

    remote_checksum_hash = sha256_file(downloaded_checksum)

    assert remote_shard_hash == sha256_file(SHARD_PATH)

    assert (
        remote_checksum_hash
        == sha256_file(CHECKSUM_PATH)
    )

    assert (
        Path(downloaded_checksum)
        .read_text(encoding="ascii")
        .strip()
        == EXPECTED_SHARD_SHA256
    )

    print("[PASS] Downloaded shard matches local SHA256")
    print("[PASS] Downloaded checksum matches local file")
    print("[PASS] Remote checksum content verified")


# ------------------------------------------------------------
# 7. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("E02 — FIRST PHONEME CACHE HF BACKUP COMPLETE")
print("=" * 70)

print("✅ Private Hugging Face repository verified.")
print("✅ First cache shard backed up.")
print("✅ SHA256 sidecar backed up.")
print("✅ Remote files downloaded successfully.")
print("✅ Exact local/remote integrity verified.")
print("✅ E01 checkpoint paths untouched.")
print("✅ No source files modified.")
print("✅ No model training started.")
print("✅ No GPU used.")

In [ ]:
# ============================================================
# E02 — FULL RESUMABLE GPU PHONEME CACHE + HF BACKUP
# KEEP — RUN THIS ONE CELL FOR TRAIN + DEV
# GPU REQUIRED | NO MODEL TRAINING
# ============================================================

from pathlib import Path
from collections import Counter
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, CommitOperationAdd, hf_hub_download
from transformers import AutoProcessor, AutoModelForCTC

import ast
import importlib
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import tempfile
import time
import uuid

import numpy as np
import torch


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REPO = Path("/kaggle/working/RTC-SDD-Research")

DATA = Path(
    "/kaggle/input/datasets/a7medlabib/rtcfake-packed-private"
)

CACHE = Path("/kaggle/working/e02_pcl_cache/v1")

PHONEME_DIR = Path(
    "/kaggle/working/rtcsdd-phoneme/metadata"
)

HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"
HF_PREFIX = "preprocessing/e02_pcl_cache/v1"

SHARD_SIZE = 256
HF_BACKUP_EVERY = 8

# No application-imposed processing time limit.
# Kaggle's own session limits still apply.

EXPECTED_HEAD_PREFIX = "41a6b3e1d104"

EXPECTED_MODEL_SHA = (
    "04366b6c8d24099ef313cf02f0e58d26"
    "f5dddfda16edbfc8eb2c713d94a9f551"
)

EXPECTED_FIRST_SHA = (
    "3f6b76a0c202692687707576266c9e8"
    "ede151e8f2cc9a171ca3bfb4fc532b5b1"
)

EXPECTED_DEV_MANIFEST = (
    "857c877dd30c9896d38b0fe49c4c79b2d3"
    "ab6a49ac46c7cb517bce84011ab4b9"
)

EXPECTED_BASE_COMMIT = (
    "25f5a6ded1db9da26f8fdfb915bd3f71716f7eb9"
)

MODEL_ID = "facebook/wav2vec2-xlsr-53-espeak-cv-ft"

MODEL_REVISION = (
    "2c733782da5604684829819a5eb744c193fe9398"
)

SPLIT_COUNTS = {
    "train": 37125,
    "dev": 7465,
}


# ------------------------------------------------------------
# 2. Framework imports
# ------------------------------------------------------------

if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from rtcsdd.data.pairs import load_pair_groups

from rtcsdd.data.phoneme_alignment import (
    ctc_token_segments,
    align_phoneme_segments,
)

from rtcsdd.data.phoneme_cache import (
    pair_key_sha256,
    read_shard,
    write_shard,
    shard_path,
)


def discover_symbol(symbol_name, node_type):
    """
    Find an existing framework symbol without assuming
    a module filename.
    """
    src_root = REPO / "src" / "rtcsdd"
    candidates = []

    for source in src_root.rglob("*.py"):
        tree = ast.parse(
            source.read_text(encoding="utf-8")
        )

        found = any(
            isinstance(node, node_type)
            and node.name == symbol_name
            for node in tree.body
        )

        if found:
            parts = (
                source.relative_to(REPO / "src")
                .with_suffix("")
                .parts
            )

            if parts[-1] == "__init__":
                parts = parts[:-1]

            candidates.append(".".join(parts))

    assert len(candidates) == 1, (
        f"Missing or ambiguous framework symbol: "
        f"{symbol_name}"
    )

    module = importlib.import_module(candidates[0])

    return getattr(module, symbol_name)


read_protocol = discover_symbol(
    "read_protocol",
    ast.FunctionDef,
)

RTCFakePackedDataset = discover_symbol(
    "RTCFakePackedDataset",
    ast.ClassDef,
)


# ------------------------------------------------------------
# 3. Integrity utilities
# ------------------------------------------------------------

def sha256_file(path):
    sha = hashlib.sha256()

    with Path(path).open("rb") as stream:
        for chunk in iter(
            lambda: stream.read(8 * 1024 * 1024),
            b"",
        ):
            sha.update(chunk)

    return sha.hexdigest()


def git(*args):
    return subprocess.check_output(
        ["git", "-C", str(REPO), *args],
        text=True,
    ).strip()


def checksum_path(path):
    return Path(str(path) + ".sha256")


def assert_local_complete(
    path,
    expected_metadata,
    groups,
):
    payload = read_shard(
        path,
        expected_metadata=expected_metadata,
    )

    for record in payload["records"]:
        pair = groups[record["pair_index"]]

        expected_hash = pair_key_sha256(
            pair.offline.utterance_id,
            pair.online.utterance_id,
        )

        assert record["pair_hash"] == expected_hash, (
            "Cache pair order or identity changed"
        )

    return payload


def remote_names(split, path):
    remote_blob = (
        f"{HF_PREFIX}/{split}/{path.name}"
    )

    return (
        remote_blob,
        remote_blob + ".sha256",
    )


# ------------------------------------------------------------
# 4. Initial safety checks
# ------------------------------------------------------------

print("=" * 70)
print("E02 — FULL RESUMABLE GPU PREPROCESSING")
print("=" * 70)

print("\n[1] FRAMEWORK AND GPU")

assert git("branch", "--show-current") == (
    "experiment/e02-pcl"
)

head = git("rev-parse", "HEAD")

assert head.startswith(EXPECTED_HEAD_PREFIX), (
    "Unexpected E02 source commit"
)

assert not git("status", "--porcelain"), (
    "Expected clean E02 working tree"
)

assert torch.cuda.is_available(), (
    "Enable GPU before executing this cell"
)

checkpoint = PHONEME_DIR / "pytorch_model.bin"

assert checkpoint.is_file(), (
    "Phoneme checkpoint not found"
)

assert sha256_file(checkpoint) == EXPECTED_MODEL_SHA, (
    "Phoneme checkpoint SHA256 mismatch"
)

print("[PASS] E02 development branch")
print("[PASS] Expected source commit")
print("[PASS] Working tree clean")
print("[PASS] Original pretrained phoneme checkpoint")
print("[PASS] GPU:", torch.cuda.get_device_name(0))


# ------------------------------------------------------------
# 5. Hugging Face authentication
# ------------------------------------------------------------

print("\n[2] HUGGING FACE")

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

assert hf_token and hf_token.strip()

hf_token = hf_token.strip()

api = HfApi(token=hf_token)

info = api.repo_info(
    repo_id=HF_REPO,
    repo_type="model",
)

assert info.private is True

remote_files = set(
    api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="model",
    )
)

print("[PASS] Private Hugging Face repository")
print("[PASS] Remote file inventory available")


# ------------------------------------------------------------
# 6. Restore or verify original first shard
# ------------------------------------------------------------

print("\n[3] ORIGINAL CACHE IDENTITY")

first = (
    CACHE
    / "train/train_000000_000256.json.gz"
)

first_sidecar = checksum_path(first)

assert first.is_file() == first_sidecar.is_file(), (
    "Incomplete first local shard"
)

if not first.is_file():

    remote_first = (
        f"{HF_PREFIX}/train/{first.name}"
    )

    remote_first_check = remote_first + ".sha256"

    assert remote_first in remote_files
    assert remote_first_check in remote_files

    with tempfile.TemporaryDirectory(
        prefix="e02_restore_first_"
    ) as folder:

        remote_a = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_first,
                token=hf_token,
                cache_dir=folder,
            )
        )

        remote_b = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_first_check,
                token=hf_token,
                cache_dir=folder,
            )
        )

        assert sha256_file(remote_a) == EXPECTED_FIRST_SHA

        assert (
            remote_b.read_text(
                encoding="ascii"
            ).strip()
            == EXPECTED_FIRST_SHA
        )

        first.parent.mkdir(
            parents=True,
            exist_ok=True,
        )

        staged_a = first.with_name(
            first.name
            + "."
            + uuid.uuid4().hex
            + ".partial"
        )

        staged_b = first_sidecar.with_name(
            first_sidecar.name
            + "."
            + uuid.uuid4().hex
            + ".partial"
        )

        try:
            shutil.copyfile(remote_a, staged_a)
            shutil.copyfile(remote_b, staged_b)

            os.replace(staged_a, first)
            os.replace(staged_b, first_sidecar)

        finally:
            staged_a.unlink(missing_ok=True)
            staged_b.unlink(missing_ok=True)

    print("[RESTORED] First shard from HF")


assert sha256_file(first) == EXPECTED_FIRST_SHA

first_payload = read_shard(first)

base_meta = dict(
    first_payload["metadata"]
)

assert (
    base_meta["framework_commit"]
    == EXPECTED_BASE_COMMIT
)

assert base_meta["phoneme_model_id"] == MODEL_ID

assert (
    base_meta["phoneme_revision"]
    == MODEL_REVISION
)

assert (
    base_meta["model_sha256"]
    == EXPECTED_MODEL_SHA
)

assert base_meta["sample_rate"] == 16000
assert base_meta["audio_length"] == 64600
assert base_meta["crop"] == "left"
assert base_meta["pad"] == "repeat"

alignment_path = (
    REPO
    / "src/rtcsdd/data/phoneme_alignment.py"
)

assert (
    sha256_file(alignment_path)
    == base_meta["alignment_module_sha256"]
)

manifest_hash = {}

for split in ("train", "dev"):

    manifest = (
        DATA
        / f"{split}_offline_online_pairs.csv"
    )

    manifest_hash[split] = sha256_file(manifest)

assert (
    manifest_hash["train"]
    == base_meta["manifest_sha256"]
)

assert (
    manifest_hash["dev"]
    == EXPECTED_DEV_MANIFEST
)

print("[PASS] Original shard verified")
print("[PASS] Phoneme model identity")
print("[PASS] Alignment implementation")
print("[PASS] Train and Dev manifests")


# ------------------------------------------------------------
# 7. Versioned execution policy
# ------------------------------------------------------------

print("\n[4] EXECUTION POLICY")

# Preserve the old run_plan.json without overwriting it.
# This version explicitly runs until completion or
# an external Kaggle interruption.

policy = {
    "version": 2,
    "experiment": "e02_ce_pcl_fp32",

    "mode": (
        "run_until_all_shards_complete_or_kaggle_interrupts"
    ),

    "time_limit_minutes": None,

    "shard_size": SHARD_SIZE,

    "hf_backup_every_completed_shards": (
        HF_BACKUP_EVERY
    ),

    "verify_hf_download": True,

    "restore_verified_remote_shards": True,

    "source_commit": head,

    "first_shard_sha256": EXPECTED_FIRST_SHA,
}

policy_path = (
    CACHE
    / "execution_policy_full_v2.json"
)

policy_bytes = (
    json.dumps(
        policy,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

if policy_path.exists():

    assert (
        policy_path.read_bytes()
        == policy_bytes
    ), "Existing execution policy differs"

else:

    with policy_path.open("xb") as stream:
        stream.write(policy_bytes)


remote_policy = (
    f"{HF_PREFIX}/execution_policy_full_v2.json"
)

if remote_policy not in remote_files:

    api.create_commit(
        repo_id=HF_REPO,
        repo_type="model",
        operations=[
            CommitOperationAdd(
                path_in_repo=remote_policy,
                path_or_fileobj=str(policy_path),
            )
        ],
        commit_message=(
            "e02: pin full run policy with 8-shard backup"
        ),
    )

    remote_files.add(remote_policy)


with tempfile.TemporaryDirectory(
    prefix="e02_policy_verify_"
) as folder:

    downloaded = hf_hub_download(
        repo_id=HF_REPO,
        repo_type="model",
        filename=remote_policy,
        token=hf_token,
        cache_dir=folder,
    )

    assert (
        Path(downloaded).read_bytes()
        == policy_bytes
    )


print("[PASS] No 90-minute application stop")
print("[PASS] Execution policy backed up")
print("[INFO] HF backup every 8 shards")


# ------------------------------------------------------------
# 8. Load frozen phoneme model
# ------------------------------------------------------------

print("\n[5] PHONEME MODEL")

processor = AutoProcessor.from_pretrained(
    str(PHONEME_DIR),
    local_files_only=True,
    do_phonemize=False,
)

assert (
    processor.feature_extractor.sampling_rate
    == 16000
)

assert processor.tokenizer.pad_token_id == 0

model = AutoModelForCTC.from_pretrained(
    str(PHONEME_DIR),
    local_files_only=True,
    use_safetensors=False,
)

model = model.to("cuda:0").eval()

for parameter in model.parameters():
    parameter.requires_grad_(False)

assert model.config.pad_token_id == 0

assert not any(
    p.requires_grad
    for p in model.parameters()
)

excluded = tuple(
    sorted(
        int(i)
        for i in base_meta["excluded_ids"]
    )
)

assert 0 not in excluded

print("[PASS] Frozen model loaded on GPU")
print("[PASS] Model in EVAL mode")
print("[PASS] No trainable phoneme parameters")


# ------------------------------------------------------------
# 9. Real phoneme extraction
# ------------------------------------------------------------

def aligned_record(
    group,
    pair_index,
    dataset,
    indices,
):
    waves = []

    for example in (
        group.offline,
        group.online,
    ):

        waveform, label, utterance_id = (
            dataset[indices[example.utterance_id]]
        )

        assert utterance_id == example.utterance_id

        assert int(label) == int(example.label)

        arr = (
            waveform.numpy()
            .astype(np.float32, copy=False)
            .reshape(-1)
        )

        assert arr.shape == (64600,)
        assert np.isfinite(arr).all()

        waves.append(arr)

    inputs = processor(
        waves,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
    )

    input_values = (
        inputs["input_values"]
        .to("cuda:0")
    )

    with torch.inference_mode():

        logits = model(input_values).logits

        assert torch.isfinite(logits).all().item()

        ids = (
            logits.argmax(-1)
            .cpu()
            .tolist()
        )

    del logits, input_values, inputs

    offline_segments = ctc_token_segments(
        ids[0],
        blank_id=0,
        excluded_ids=excluded,
    )

    online_segments = ctc_token_segments(
        ids[1],
        blank_id=0,
        excluded_ids=excluded,
    )

    matches = align_phoneme_segments(
        offline_segments,
        online_segments,
    )

    return {
        "pair_index": pair_index,

        "pair_hash": pair_key_sha256(
            group.offline.utterance_id,
            group.online.utterance_id,
        ),

        "offline_frames": len(ids[0]),

        "online_frames": len(ids[1]),

        "matches": [
            [
                int(match.token_id),

                int(match.offline.start_frame),
                int(match.offline.end_frame),

                int(match.online.start_frame),
                int(match.online.end_frame),
            ]
            for match in matches
        ],
    }


# ------------------------------------------------------------
# 10. Verify remote shard and restore when necessary
# ------------------------------------------------------------

def verify_hf(
    path,
    split,
    expected_metadata,
    groups,
):
    remote_blob, remote_check = remote_names(
        split,
        path,
    )

    with tempfile.TemporaryDirectory(
        prefix="e02_verify_"
    ) as folder:

        remote_a = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_blob,
                token=hf_token,
                cache_dir=folder,
            )
        )

        remote_b = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type="model",
                filename=remote_check,
                token=hf_token,
                cache_dir=folder,
            )
        )

        expected = (
            checksum_path(path)
            .read_text(encoding="ascii")
            if checksum_path(path).is_file()
            else remote_b.read_text(encoding="ascii")
        )

        assert (
            expected
            == remote_b.read_text(encoding="ascii")
        ), "Remote checksum mismatch"

        assert (
            sha256_file(remote_a)
            == expected.strip()
        ), "Remote shard SHA256 mismatch"

        if path.is_file():

            assert (
                sha256_file(path)
                == sha256_file(remote_a)
            ), "Local and HF shard differ"

        else:

            assert not checksum_path(path).exists(), (
                "Incomplete local cache"
            )

            path.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            work_a = path.with_name(
                path.name
                + "."
                + uuid.uuid4().hex
                + ".partial"
            )

            work_b = checksum_path(path).with_name(
                path.name
                + ".sha256."
                + uuid.uuid4().hex
                + ".partial"
            )

            try:

                shutil.copyfile(remote_a, work_a)
                shutil.copyfile(remote_b, work_b)

                assert (
                    sha256_file(work_a)
                    == expected.strip()
                )

                os.replace(work_a, path)

                os.replace(
                    work_b,
                    checksum_path(path),
                )

            finally:

                work_a.unlink(missing_ok=True)
                work_b.unlink(missing_ok=True)

    assert_local_complete(
        path,
        expected_metadata,
        groups,
    )


# ------------------------------------------------------------
# 11. HF backup queue
# ------------------------------------------------------------

pending = []
counts = Counter()


def flush_pending(reason):

    if not pending:
        return

    print(
        f"\n[HF] Backing up {len(pending)} shards "
        f"({reason})",
        flush=True,
    )

    names_now = set(
        api.list_repo_files(
            repo_id=HF_REPO,
            repo_type="model",
        )
    )

    new_pairs = []

    for split, path, metadata, groups in pending:

        blob, checksum = remote_names(
            split,
            path,
        )

        present = (
            blob in names_now,
            checksum in names_now,
        )

        assert present not in (
            (True, False),
            (False, True),
        ), "Incomplete remote shard backup"

        if not any(present):

            assert_local_complete(
                path,
                metadata,
                groups,
            )

            new_pairs.append(
                (split, path, metadata, groups)
            )

    operations = []

    for split, path, _, _ in new_pairs:

        blob, checksum = remote_names(
            split,
            path,
        )

        operations.extend([
            CommitOperationAdd(
                path_in_repo=blob,
                path_or_fileobj=str(path),
            ),

            CommitOperationAdd(
                path_in_repo=checksum,
                path_or_fileobj=str(
                    checksum_path(path)
                ),
            ),
        ])

    if operations:

        api.create_commit(
            repo_id=HF_REPO,
            repo_type="model",
            operations=operations,
            commit_message=(
                f"e02: back up {len(new_pairs)} "
                f"verified phoneme shards"
            ),
        )

    for split, path, metadata, groups in pending:

        verify_hf(
            path,
            split,
            metadata,
            groups,
        )

        counts["hf_verified"] += 1

    pending.clear()

    print(
        "[HF PASS] Batch uploaded and verified",
        flush=True,
    )


# ------------------------------------------------------------
# 12. Full Train + Dev preprocessing
# ------------------------------------------------------------

print("\n[6] STARTING FULL PREPROCESSING")

start_wall = time.monotonic()

try:

    for split in ("train", "dev"):

        print(
            f"\n[DATA] Loading {split}...",
            flush=True,
        )

        records = read_protocol(
            DATA / f"{split}_label.txt",
            split,
        )

        pair_groups = load_pair_groups(
            DATA / f"{split}_offline_online_pairs.csv",
            records,
            split=split,
        )

        complete = [
            pair
            for pair in pair_groups
            if pair.online is not None
        ]

        assert (
            len(complete)
            == SPLIT_COUNTS[split]
        ), f"Unexpected {split} pair count"

        assert len({
            record.utterance_id
            for record in records
        }) == len(records)

        index_of = {
            record.utterance_id: index
            for index, record in enumerate(records)
        }

        assert all(
            pair.offline.utterance_id in index_of
            and pair.online.utterance_id in index_of
            for pair in complete
        )

        dataset = None

        total_shards = math.ceil(
            len(complete) / SHARD_SIZE
        )

        for shard_i, start in enumerate(
            range(0, len(complete), SHARD_SIZE),
            start=1,
        ):

            stop = min(
                start + SHARD_SIZE,
                len(complete),
            )

            path = shard_path(
                CACHE,
                split,
                start,
                stop,
            )

            # Preserve the original preprocessing identity.
            metadata = dict(base_meta)

            metadata.update(
                split=split,
                start=start,
                stop=stop,
                manifest_sha256=manifest_hash[split],
            )

            blob, checksum = remote_names(
                split,
                path,
            )

            remote_present = (
                blob in remote_files,
                checksum in remote_files,
            )

            assert remote_present not in (
                (True, False),
                (False, True),
            ), "Incomplete remote cache entry"

            local_present = (
                path.is_file(),
                checksum_path(path).is_file(),
            )

            assert local_present not in (
                (True, False),
                (False, True),
            ), "Incomplete local cache entry"

            # ----------------------------------------
            # Case A — Already cached locally
            # ----------------------------------------

            if all(local_present):

                assert_local_complete(
                    path,
                    metadata,
                    complete,
                )

                counts["local_verified"] += 1

            # ----------------------------------------
            # Case B — Restore from Hugging Face
            # ----------------------------------------

            elif all(remote_present):

                verify_hf(
                    path,
                    split,
                    metadata,
                    complete,
                )

                counts["restored"] += 1

                print(
                    f"[RESTORED] {split} "
                    f"{start}:{stop}",
                    flush=True,
                )

            # ----------------------------------------
            # Case C — Generate new shard on GPU
            # ----------------------------------------

            else:

                if dataset is None:

                    dataset = RTCFakePackedDataset(
                        records,
                        DATA,
                        length=64600,
                        crop="left",
                        pad="repeat",
                        rawboost={"enabled": False},
                        seed=1234,
                    )

                shard_start_time = time.monotonic()

                result = [
                    aligned_record(
                        complete[idx],
                        idx,
                        dataset,
                        index_of,
                    )
                    for idx in range(start, stop)
                ]

                written, digest = write_shard(
                    CACHE,
                    metadata,
                    result,
                )

                assert written == path

                assert (
                    sha256_file(path)
                    == digest
                )

                assert_local_complete(
                    path,
                    metadata,
                    complete,
                )

                counts["generated"] += 1

                elapsed = (
                    time.monotonic()
                    - shard_start_time
                )

                print(
                    f"[GPU] {split} "
                    f"{shard_i}/{total_shards} "
                    f"({start}:{stop}) "
                    f"cached in {elapsed:.1f}s",
                    flush=True,
                )

            # ----------------------------------------
            # Backup queue
            # ----------------------------------------

            if not all(remote_present):

                pending.append(
                    (
                        split,
                        path,
                        metadata,
                        complete,
                    )
                )

            else:

                if all(local_present):

                    verify_hf(
                        path,
                        split,
                        metadata,
                        complete,
                    )

                counts["hf_existing_verified"] += 1

            # Upload every 8 completed shards.
            if len(pending) >= HF_BACKUP_EVERY:

                flush_pending("periodic")

        # Upload remaining shards at split boundary.
        flush_pending(f"{split} complete")

        print(
            f"[SPLIT PASS] {split}: "
            f"{total_shards} shards",
            flush=True,
        )

    # --------------------------------------------------------
    # 13. Final coverage validation
    # --------------------------------------------------------

    print(
        "\n[7] FINAL LOCAL + HF COVERAGE CHECK",
        flush=True,
    )

    remote_final = set(
        api.list_repo_files(
            repo_id=HF_REPO,
            repo_type="model",
        )
    )

    for split, total in SPLIT_COUNTS.items():

        for start in range(
            0,
            total,
            SHARD_SIZE,
        ):

            stop = min(
                start + SHARD_SIZE,
                total,
            )

            path = shard_path(
                CACHE,
                split,
                start,
                stop,
            )

            assert path.is_file()
            assert checksum_path(path).is_file()

            assert (
                sha256_file(path)
                == checksum_path(path)
                .read_text()
                .strip()
            )

            blob, checksum = remote_names(
                split,
                path,
            )

            assert blob in remote_final
            assert checksum in remote_final

except BaseException:

    if pending:

        print(
            "\n[WARN] Run interrupted. "
            "Attempting to back up completed shards...",
            flush=True,
        )

        try:
            flush_pending("interruption")

        except Exception:

            print(
                "[WARN] HF backup could not finish. "
                "Completed local shards remain available "
                "for resume.",
                flush=True,
            )

    raise


# ------------------------------------------------------------
# 14. Final summary
# ------------------------------------------------------------

elapsed_hours = (
    time.monotonic() - start_wall
) / 3600

print("\n" + "=" * 70)
print("E02 — ALL PHONEME CACHE SHARDS COMPLETE")
print("=" * 70)

print("Train pairs          :", SPLIT_COUNTS["train"])
print("Dev pairs            :", SPLIT_COUNTS["dev"])

print("Train shards         : 146")
print("Dev shards           : 30")
print("Total shards         : 176")

print("New shards generated :", counts["generated"])
print("Shards restored      :", counts["restored"])
print("HF shards verified   :", counts["hf_verified"])

print("Elapsed hours        :", round(elapsed_hours, 2))

print("\n✅ All 176 cache shards exist locally.")
print("✅ All 176 cache shards have HF backup paths.")
print("✅ SHA256 checks completed.")
print("✅ HF backups executed every 8 pending shards.")
print("✅ Final remaining shards uploaded.")
print("✅ Existing shard overwrite prevented.")
print("✅ E01 files untouched.")
print("✅ No model training started.")

In [ ]:

# ============================================================
# E02 — PCL CACHE READINESS & SCHEMA VERIFICATION
# KEEP | CPU ONLY | NO TRAINING | NO MODEL DOWNLOAD
# ============================================================

from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import os
import re
import json
import gzip
import hashlib

print("=" * 72)
print("E02 — PCL CACHE READINESS VERIFICATION")
print("=" * 72)

HF_REPO = "AhmedLabib1/rtc-sdd-research-checkpoints"
PREFIX = "preprocessing/e02_pcl_cache/v1"

OUTPUT = Path(
    "/kaggle/working/e02_cache_readiness_report.json"
)

SPLITS = {
    "train": 37125,
    "dev": 7465,
}

SHARD_SIZE = 256

# ------------------------------------------------------------
# 1. HF AUTHENTICATION
# ------------------------------------------------------------

print("\n[1/5] HUGGING FACE AUTHENTICATION")

token = os.environ.get("HF_TOKEN")

if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")

assert token, "HF_TOKEN missing"

api = HfApi(token=token)

user = api.whoami()
print("[PASS] Authenticated:", user.get("name"))

# ------------------------------------------------------------
# 2. BUILD EXACT EXPECTED SHARD INVENTORY
# ------------------------------------------------------------

print("\n[2/5] EXPECTED CACHE INVENTORY")

expected = {}
split_summary = {}

for split, total in SPLITS.items():
    paths = []

    for start in range(0, total, SHARD_SIZE):
        end = min(start + SHARD_SIZE, total)

        filename = (
            f"{split}_{start:06d}_{end:06d}.json.gz"
        )

        remote = f"{PREFIX}/{split}/{filename}"
        paths.append(remote)

    expected[split] = paths

    split_summary[split] = {
        "pairs": total,
        "shards": len(paths),
        "first": paths[0],
        "last": paths[-1],
    }

    print(
        f"[EXPECTED] {split}: "
        f"{total:,} pairs / {len(paths)} shards"
    )

assert len(expected["train"]) == 146
assert len(expected["dev"]) == 30

# ------------------------------------------------------------
# 3. VERIFY REMOTE COVERAGE
# ------------------------------------------------------------

print("\n[3/5] VERIFYING HUGGING FACE FILE INVENTORY")

remote_files = set(
    api.list_repo_files(
        repo_id=HF_REPO,
        repo_type="model"
    )
)

missing = []

for split, paths in expected.items():

    found = 0

    for path in paths:
        if path in remote_files:
            found += 1
        else:
            missing.append(path)

        if f"{path}.sha256" not in remote_files:
            missing.append(f"{path}.sha256")

    print(
        f"[REMOTE] {split}: "
        f"{found}/{len(paths)} cache files"
    )

    print(
        f"[REMOTE] {split}: "
        "SHA256 sidecars checked"
    )

if missing:
    print("\nMissing remote artifacts:")
    for path in missing[:20]:
        print(" -", path)

    raise RuntimeError(
        f"Remote cache incomplete: "
        f"{len(missing)} files or checksums missing"
    )

print("[PASS] All 176 shards and SHA256 sidecars found")

# ------------------------------------------------------------
# 4. SAMPLE CONTENT / HASH / SCHEMA INSPECTION
# ------------------------------------------------------------

print("\n[4/5] SAMPLE CONTENT & SHA256 VALIDATION")

# Verify first and last shard for both splits.
samples = {
    "train_first": expected["train"][0],
    "train_last": expected["train"][-1],
    "dev_first": expected["dev"][0],
    "dev_last": expected["dev"][-1],
}

inspection = {}


def download_small(remote_path):
    return Path(
        hf_hub_download(
            repo_id=HF_REPO,
            repo_type="model",
            filename=remote_path,
            token=token,
        )
    )


def describe_schema(data):
    """Describe structure without exposing recording IDs."""

    result = {
        "root_type": type(data).__name__,
    }

    if isinstance(data, dict):
        result["top_level_keys"] = list(data.keys())[:40]

        result["field_types"] = {
            str(k): type(v).__name__
            for k, v in list(data.items())[:40]
        }

        result["list_lengths"] = {
            str(k): len(v)
            for k, v in data.items()
            if isinstance(v, list)
        }

        for key, value in data.items():
            if isinstance(value, list) and value:
                result["first_list_field"] = str(key)
                result["first_list_item_type"] = (
                    type(value[0]).__name__
                )

                if isinstance(value[0], dict):
                    result["first_item_keys"] = list(
                        value[0].keys()
                    )[:40]

                break

    elif isinstance(data, list):
        result["record_count"] = len(data)

        if data and isinstance(data[0], dict):
            result["first_item_keys"] = list(
                data[0].keys()
            )[:40]

    return result


for label, remote in samples.items():

    cache_path = download_small(remote)
    checksum_path = download_small(
        f"{remote}.sha256"
    )

    expected_sha = checksum_path.read_text(
        encoding="utf-8"
    ).strip().split()[0].lower()

    assert re.fullmatch(
        r"[a-f0-9]{64}",
        expected_sha
    ), f"Invalid SHA256 sidecar: {label}"

    actual_sha = hashlib.sha256(
        cache_path.read_bytes()
    ).hexdigest()

    assert actual_sha == expected_sha, (
        f"SHA256 mismatch: {label}"
    )

    with gzip.open(
        cache_path,
        mode="rt",
        encoding="utf-8"
    ) as f:
        payload = json.load(f)

    schema = describe_schema(payload)

    inspection[label] = {
        "remote_path": remote,
        "sha256_verified": True,
        "schema": schema,
    }

    print(f"\n[PASS] {label}")
    print(
        json.dumps(
            schema,
            indent=2,
            ensure_ascii=False
        )
    )

# ------------------------------------------------------------
# 5. SAVE REPORT
# ------------------------------------------------------------

print("\n[5/5] WRITING READINESS REPORT")

report = {
    "experiment": "E02_PCL",
    "phase": "post_preprocessing",
    "hf_repo": HF_REPO,
    "cache_prefix": PREFIX,
    "split_summary": split_summary,
    "remote_shards_present": 176,
    "remote_sha256_sidecars_present": 176,
    "remote_inventory_complete": True,
    "sample_shards_sha256_verified": len(inspection),
    "sample_inspection": inspection,
    "all_remote_shards_content_rehashed": False,
    "training_started": False,
    "next_step": "PCL loss integration",
}

OUTPUT.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("E02 — CACHE READINESS SUMMARY")
print("=" * 72)

print("Train pairs:", f"{SPLITS['train']:,}")
print("Dev pairs:", f"{SPLITS['dev']:,}")
print("Remote shards:", "176 / 176")
print("SHA256 sidecars:", "176 / 176")
print("Sample payload hashes verified:", len(inspection))
print("E02 fine-tuning started: NO")
print("E01 checkpoints modified: NO")
print("Report:", OUTPUT)
print("=" * 72)


In [ ]:

# ============================================================
# E02 — FULL CACHE INTEGRITY & PCL STRUCTURE AUDIT
# KEEP | CPU ONLY | NO MODEL | NO TRAINING
# ============================================================

from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import hashlib
import gzip
import json
import re

ROOT = Path("/kaggle/working/e02_pcl_cache/v1")
OUTPUT = Path(
    "/kaggle/working/e02_pcl_cache_structure_report.json"
)

SPLITS = {
    "train": 37125,
    "dev": 7465,
}

SHARD_SIZE = 256

print("=" * 72)
print("E02 — FULL CACHE INTEGRITY & PCL STRUCTURE AUDIT")
print("=" * 72)

assert ROOT.is_dir(), (
    f"Local cache missing: {ROOT}\n"
    "Do not rerun GPU preprocessing. "
    "Restore existing cached shards instead."
)

# ------------------------------------------------------------
# 1. GENERATE EXPECTED FILE INVENTORY
# ------------------------------------------------------------

print("\n[1/4] LOCAL CACHE INVENTORY")

expected = {}

for split, total in SPLITS.items():

    expected[split] = []

    for start in range(0, total, SHARD_SIZE):
        end = min(start + SHARD_SIZE, total)

        filename = (
            f"{split}_{start:06d}_{end:06d}.json.gz"
        )

        expected[split].append(
            (ROOT / split / filename, start, end)
        )

    print(
        f"{split}: {len(expected[split])} "
        f"shards, {total:,} expected records"
    )

assert len(expected["train"]) == 146
assert len(expected["dev"]) == 30

# ------------------------------------------------------------
# 2. SAFE STRUCTURE PREVIEW
# ------------------------------------------------------------

def preview(value, depth=0):
    """
    Show numeric alignment data and structure.
    Mask arbitrary text identifiers and hashes.
    """

    if depth >= 4:
        return f"<{type(value).__name__}>"

    if isinstance(value, dict):
        return {
            str(k): preview(v, depth + 1)
            for k, v in list(value.items())[:15]
        }

    if isinstance(value, list):
        return {
            "type": "list",
            "length": len(value),
            "first_items": [
                preview(x, depth + 1)
                for x in value[:2]
            ],
        }

    if isinstance(value, bool):
        return value

    if isinstance(value, (int, float)):
        return value

    if value is None:
        return None

    if isinstance(value, str):
        return "<string>"

    return f"<{type(value).__name__}>"


def read_verified(path):
    sidecar = Path(str(path) + ".sha256")

    assert path.is_file(), f"Missing shard: {path}"
    assert sidecar.is_file(), (
        f"Missing SHA256 sidecar: {sidecar}"
    )

    expected_sha = (
        sidecar.read_text(encoding="utf-8")
        .strip().split()[0].lower()
    )

    assert re.fullmatch(
        r"[0-9a-f]{64}", expected_sha
    ), f"Invalid checksum: {sidecar}"

    raw = path.read_bytes()

    actual_sha = hashlib.sha256(raw).hexdigest()

    assert actual_sha == expected_sha, (
        f"SHA256 mismatch: {path.name}"
    )

    with gzip.open(path, "rt", encoding="utf-8") as f:
        payload = json.load(f)

    assert isinstance(payload, dict)
    assert isinstance(payload.get("metadata"), dict)
    assert isinstance(payload.get("records"), list)

    return payload


# ------------------------------------------------------------
# 3. VERIFY ALL SHARDS & INSPECT RECORDS
# ------------------------------------------------------------

print("\n[2/4] VERIFYING ALL 176 SHARDS")

results = {}
examples = {}

for split, shards in expected.items():

    total_records = 0
    matched_records = 0
    empty_match_records = 0

    match_item_total = 0
    frame_type_counts = Counter()
    match_type_counts = Counter()

    seen_indices = set()
    duplicate_indices = 0

    metadata_keys = set()
    sample_record = None
    sample_nonempty = None

    for path, start, end in tqdm(
        shards,
        desc=f"Auditing {split}",
        unit="shard"
    ):
        payload = read_verified(path)
        records = payload["records"]

        assert len(records) == end - start, (
            f"Wrong record count: {path.name}"
        )

        metadata_keys.update(
            payload["metadata"].keys()
        )

        total_records += len(records)

        for record in records:

            assert isinstance(record, dict)

            required = {
                "matches",
                "offline_frames",
                "online_frames",
                "pair_hash",
                "pair_index",
            }

            assert required.issubset(record), (
                f"Missing fields: {path.name}"
            )

            index = record["pair_index"]

            assert isinstance(index, int)
            assert start <= index < end, (
                f"Unexpected pair_index in {path.name}: "
                f"{index}"
            )

            if index in seen_indices:
                duplicate_indices += 1

            seen_indices.add(index)

            matches = record["matches"]

            match_type_counts[
                type(matches).__name__
            ] += 1

            if isinstance(matches, (list, dict)):
                match_count = len(matches)
            else:
                match_count = int(bool(matches))

            match_item_total += match_count

            if match_count:
                matched_records += 1

                if sample_nonempty is None:
                    sample_nonempty = record
            else:
                empty_match_records += 1

            for field in [
                "offline_frames",
                "online_frames",
            ]:
                frame_type_counts[
                    f"{field}:"
                    f"{type(record[field]).__name__}"
                ] += 1

            if sample_record is None:
                sample_record = record

    expected_total = SPLITS[split]

    assert total_records == expected_total
    assert duplicate_indices == 0
    assert len(seen_indices) == expected_total
    assert seen_indices == set(range(expected_total))

    results[split] = {
        "expected_records": expected_total,
        "verified_records": total_records,
        "verified_shards": len(shards),
        "unique_pair_indices": len(seen_indices),
        "duplicate_pair_indices": duplicate_indices,
        "records_with_matches": matched_records,
        "records_without_matches": empty_match_records,
        "total_match_entries": match_item_total,
        "match_field_types": dict(match_type_counts),
        "frame_field_types": dict(frame_type_counts),
        "metadata_keys": sorted(metadata_keys),
    }

    examples[split] = {
        "metadata_example": preview(
            read_verified(shards[0][0])["metadata"]
        ),
        "first_record": preview({
            k: v for k, v in sample_record.items()
            if k != "pair_hash"
        }),
        "first_nonempty_match_record": (
            preview({
                k: v
                for k, v in sample_nonempty.items()
                if k != "pair_hash"
            })
            if sample_nonempty else None
        ),
    }

    print(f"\n[PASS] {split.upper()}")
    print("  Shards:", len(shards))
    print("  Records:", total_records)
    print("  Unique indices:", len(seen_indices))
    print("  Records with matches:", matched_records)
    print("  Records without matches:", empty_match_records)
    print("  Total match entries:", match_item_total)

# ------------------------------------------------------------
# 4. PRINT DATA STRUCTURE
# ------------------------------------------------------------

print("\n[3/4] PCL DATA STRUCTURE")

for split in ["train", "dev"]:

    print("\n" + "-" * 72)
    print(split.upper(), "SAMPLE STRUCTURE")
    print("-" * 72)

    print(
        json.dumps(
            examples[split],
            indent=2,
            ensure_ascii=False
        )
    )

# ------------------------------------------------------------
# 5. SAVE REPORT
# ------------------------------------------------------------

print("\n[4/4] SAVING AUDIT REPORT")

report = {
    "experiment": "E02_PCL",
    "stage": "cache_structure_validation",
    "local_cache": str(ROOT),
    "all_expected_shards_verified": True,
    "all_local_sha256_verified": True,
    "training_started": False,
    "splits": results,
    "structure_examples": examples,
}

OUTPUT.write_text(
    json.dumps(
        report,
        indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("E02 — FULL CACHE STRUCTURE AUDIT SUMMARY")
print("=" * 72)

print(
    "Verified shards:",
    sum(x["verified_shards"] for x in results.values())
)

print(
    "Verified pair records:",
    sum(x["verified_records"] for x in results.values())
)

print("Local SHA256 checks: PASSED")
print("Pair index integrity: PASSED")
print("GPU required: NO")
print("Training started: NO")
print("Report:", OUTPUT)
print("=" * 72)


In [ ]:
# ============================================================
# E02 — RESTORE OFFICIAL PRETRAINED XLS-R 300M
# KEEP | DOWNLOAD ONLY | NO TRAINING | NO CACHE CHANGES
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import requests

URL = "https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt"
TARGET = Path("/kaggle/working/checkpoints/pretrained/xlsr2_300m.pt")
PARTIAL = TARGET.with_name(TARGET.name + ".part")
RECORD = TARGET.with_name(TARGET.name + ".download.json")
MIN_BYTES = 700 * 1024**2  # sanity bound, not an authenticity proof
CHUNK = 8 * 1024**2

print("=" * 74)
print("E02 — RESTORE OFFICIAL XLS-R 300M CHECKPOINT")
print("=" * 74)
print("Official source:", URL)
print("Target:", TARGET)

TARGET.parent.mkdir(parents=True, exist_ok=True)

# Do not delete or overwrite an existing complete checkpoint.
if TARGET.is_file() and TARGET.stat().st_size >= MIN_BYTES:
    print("[FOUND] Existing pretrained checkpoint")
    print("[INFO] Reusing file without modifying it")
else:
    if TARGET.exists():
        raise RuntimeError(
            "An undersized file already exists at the target. "
            "Inspect it manually; refusing to overwrite automatically."
        )

    available = shutil.disk_usage(TARGET.parent).free
    if available < 2 * 1024**3:
        raise RuntimeError(
            f"Insufficient disk free space: {available / 1024**3:.2f} GiB"
        )

    # Resume a partial download if the server supports HTTP Range.
    offset = PARTIAL.stat().st_size if PARTIAL.is_file() else 0
    headers = {"Range": f"bytes={offset}-"} if offset else {}
    print(f"[INFO] Existing partial: {offset / 1024**2:.1f} MiB")

    with requests.get(
        URL,
        stream=True,
        timeout=(25, 180),
        allow_redirects=True,
        headers=headers,
    ) as response:
        response.raise_for_status()
        if response.status_code not in (200, 206):
            raise RuntimeError(f"Unexpected HTTP status: {response.status_code}")
        content_type = response.headers.get("Content-Type", "").lower()
        if "text/html" in content_type:
            raise RuntimeError("Received HTML, not the checkpoint file")

        if response.status_code == 206:
            match = re.fullmatch(
                r"bytes\s+(\d+)-(\d+)/(\d+|\*)",
                response.headers.get("Content-Range", "")
            )
            if not match or int(match.group(1)) != offset:
                raise RuntimeError("Server returned an invalid HTTP Range response")
            mode = "ab"
            expected_total = (
                int(match.group(3)) if match.group(3) != "*" else None
            )
            initial = offset
            print("[PASS] Resuming with verified HTTP Range")
        else:
            if offset:
                print("[INFO] Server did not accept resume; restarting download")
            mode = "wb"
            initial = 0
            header_length = response.headers.get("Content-Length", "")
            expected_total = int(header_length) if header_length.isdigit() else None

        with tqdm(
            total=expected_total,
            initial=initial,
            desc="Official XLS-R 300M",
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            with PARTIAL.open(mode) as dest:
                for block in response.iter_content(chunk_size=CHUNK):
                    if block:
                        dest.write(block)
                        bar.update(len(block))
                dest.flush()
                os.fsync(dest.fileno())

    received = PARTIAL.stat().st_size
    if received < MIN_BYTES:
        raise RuntimeError(
            f"Checkpoint unexpectedly small: {received / 1024**2:.1f} MiB. "
            "Partial download retained for retry."
        )
    if expected_total is not None and received != expected_total:
        raise RuntimeError(
            f"Incomplete download: got {received} of {expected_total} bytes. "
            "Partial download retained for retry."
        )

    os.replace(PARTIAL, TARGET)  # atomic publish after transfer checks
    print("[PASS] Download completed and checkpoint published")

# Hash the exact downloaded/reused checkpoint for reproducibility.
sha = hashlib.sha256()
with TARGET.open("rb") as f:
    for block in iter(lambda: f.read(CHUNK), b""):
        sha.update(block)

meta = {
    "filename": TARGET.name,
    "official_url": URL,
    "local_path": str(TARGET),
    "size_bytes": TARGET.stat().st_size,
    "sha256_observed": sha.hexdigest(),
    "sha256_independently_authenticated": False,
    "recorded_utc": datetime.now(timezone.utc).isoformat(),
}
RECORD.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")

print("\n[CHECKPOINT]")
print("[PASS] Path:", TARGET)
print(f"[PASS] Size: {TARGET.stat().st_size / 1024**3:.3f} GiB")
print("[PASS] Recorded SHA256:", meta["sha256_observed"])
print("[NOTE] SHA256 is locally computed, not checked against a published digest")

# Early warning for the NEXT smoke-cell requirement.
print("\n[TRAINING RUNTIME PRECHECK]")
venv_python = Path("/kaggle/working/rtcsdd-baseline-py310/bin/python")
for candidate in [venv_python, Path(sys.executable)]:
    if not candidate.is_file():
        continue
    result = subprocess.run(
        [str(candidate), "-c", (
            "import torch, fairseq; "
            "assert torch.cuda.is_available(); "
            "print(torch.__version__, torch.cuda.get_device_name(0))"
        )], capture_output=True, text=True,
    )
    if result.returncode == 0:
        print("[PASS] CUDA + Fairseq:", candidate)
        print("      ", result.stdout.strip())
        break
    print("[NOT READY] CUDA + Fairseq:", candidate)
    print("           ", (result.stderr or result.stdout).strip().splitlines()[-1:])
else:
    print("[ACTION NEEDED] Restore/create the Python 3.10 Fairseq training environment")

print("\nPretrained checkpoint restored; no fine-tuning performed.")
print("Next: rerun E02 Real Model + PCL GPU Gradient Smoke Cell.")
print("=" * 74)


In [ ]:
# ============================================================
# E02 — RESTORE PINNED PYTHON 3.10 CUDA / FAIRSEQ RUNTIME
# KEEP | ENVIRONMENT SETUP ONLY | NO MODEL TRAINING
# Does not touch phoneme environment, E02 caches, or checkpoints.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys

REPO = Path('/kaggle/working/RTC-SDD-Research')
VENV = Path('/kaggle/working/rtcsdd-baseline-py310')
PY310 = VENV / 'bin/python'
INSTALLER = REPO / 'scripts/setup/install_training_env.py'
BOOTSTRAP = REPO / 'scripts/setup/bootstrap_official.py'
PIN = REPO / 'third_party/official_rtc_sdd.pin.json'
REQS = REPO / 'third_party/official_rtc_sdd/requirements.txt'
PCL = REPO / 'src/rtcsdd/training/phoneme_contrastive.py'
REPORT = Path('/kaggle/working/e02_pinned_training_environment_report.json')

print('=' * 72, flush=True)
print('E02 — RESTORE PINNED GPU TRAINING ENVIRONMENT', flush=True)
print('=' * 72, flush=True)

# 1. Fail closed: use only this repository's verified pin/installer.
print('\n[1/5] PROJECT AND PINNED INSTALLER', flush=True)
for p in (REPO, INSTALLER, BOOTSTRAP, PIN, REQS, PCL):
    if not p.exists():
        raise FileNotFoundError(
            f'Required E02 source missing: {p}\n'
            'Restore the repository setup; do not install arbitrary dependencies.'
        )
print('[PASS] E02 repository, PCL implementation, pin and installer found', flush=True)

# 2. Install uv only when absent (not in the phoneme venv).
print('\n[2/5] UV / PYTHON 3.10', flush=True)
uv = shutil.which('uv')
if uv is None:
    print('[SETUP] Installing uv launcher in notebook environment', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True
    )
    uv = shutil.which('uv')
if uv is None:
    raise RuntimeError('uv was installed but is not on PATH')
print('[PASS] uv:', uv, flush=True)

# Keep uv wheel/archive downloads OUT of the Kaggle output directory.
env = dict(os.environ)
env['UV_CACHE_DIR'] = '/tmp/rtcsdd-e02-uv-cache'
env['PYTHONUNBUFFERED'] = '1'

if not PY310.is_file():
    print('[SETUP] Installing Python 3.10 if needed', flush=True)
    subprocess.run([uv, 'python', 'install', '3.10'], env=env, check=True)
    if VENV.exists() and any(VENV.iterdir()):
        raise RuntimeError(
            f'Nonempty incomplete environment exists: {VENV}. '
            'Refusing to overwrite; inspect it first.'
        )
    subprocess.run(
        [uv, 'venv', '--python', '3.10', str(VENV)],
        env=env, check=True
    )

version = subprocess.check_output(
    [str(PY310), '-c', 'import sys; print(f"{sys.version_info.major}.{sys.version_info.minor}")'],
    text=True
).strip()
if version != '3.10':
    raise RuntimeError(f'Expected isolated Python 3.10, got {version}')
print('[PASS] Python:', version, flush=True)
print('[PASS] Isolated environment:', VENV, flush=True)

# 3. Check pinned upstream before any package installation.
print('\n[3/5] OFFICIAL SOURCE PIN VERIFICATION', flush=True)
subprocess.run(
    [sys.executable, str(BOOTSTRAP), '--verify-only'],
    cwd=REPO, env=env, check=True
)
print('[PASS] Original official source verified', flush=True)

# 4. If existing venv already works, skip the expensive reinstall.
print('\n[4/5] INSTALL / REUSE PINNED DEPENDENCIES', flush=True)
probe = r'''
import sys
import numpy as np
for name, value in {'float': float, 'int': int, 'complex': complex,
                    'bool': bool, 'object': object, 'str': str}.items():
    if name not in np.__dict__:
        setattr(np, name, value)
import torch
import fairseq
import rtcsdd
assert sys.version_info[:2] == (3, 10)
assert torch.__version__.startswith('2.5.1+cu121'), torch.__version__
assert torch.cuda.is_available(), 'CUDA not available in training venv'
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
print('Fairseq:', getattr(fairseq, '__version__', 'installed'))
print('Project: rtcsdd import OK')
'''

def test_runtime():
    return subprocess.run(
        [str(PY310), '-c', probe],
        cwd=REPO,
        env={**env, 'PYTHONPATH': str(REPO / 'src')},
        capture_output=True, text=True,
    )

initial = test_runtime()
if initial.returncode == 0:
    print('[PASS] Existing pinned training environment reused', flush=True)
else:
    print('[INFO] Runtime needs installation or repair.', flush=True)
    print('[INFO] Probe reason:', '\n'.join(
        (initial.stderr or initial.stdout).strip().splitlines()[-3:]
    ), flush=True)
    print('[SETUP] Running repository-pinned installer', flush=True)
    subprocess.run(
        [sys.executable, str(INSTALLER), '--python', str(PY310), '--install'],
        cwd=REPO, env=env, check=True
    )
    print('[PASS] Pinned dependency installer completed', flush=True)

# 5. Final runtime check: never report success from installation alone.
print('\n[5/5] FINAL CUDA + FAIRSEQ + PROJECT VERIFICATION', flush=True)
final = test_runtime()
print(final.stdout, end='', flush=True)
if final.returncode != 0:
    print(final.stderr[-5000:], flush=True)
    raise RuntimeError(
        'Pinned packages were installed, but the real CUDA/Fairseq runtime '
        'probe failed. Do not run the model smoke cell until diagnosed.'
    )

report = {
    'stage': 'E02_pinned_training_runtime_ready',
    'checked_utc': datetime.now(timezone.utc).isoformat(),
    'python_executable': str(PY310),
    'python_version': version,
    'pin_file': str(PIN),
    'requirements': str(REQS),
    'pcl_module': str(PCL),
    'cuda_fairseq_verified': True,
    'fine_tuning_started': False,
    'pretrained_checkpoint_modified': False,
    'phoneme_cache_modified': False,
    'phoneme_environment_modified': False,
}
REPORT.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
print('\n' + '=' * 72, flush=True)
print('E02 — TRAINING ENVIRONMENT: READY', flush=True)
print('Interpreter:', PY310, flush=True)
print('Report:', REPORT, flush=True)
print('Training started: NO', flush=True)
print('Next: rerun E02 Real Model + PCL Gradient Smoke Test', flush=True)
print('=' * 72, flush=True)


In [ ]:
# ================================================================
# E02 — REAL XLS-R/AASIST + PCL GRADIENT SMOKE
# KEEP | GPU | NO OPTIMIZER STEP | NO CHECKPOINT SAVE/UPLOAD
# ================================================================

from pathlib import Path
import csv
import gzip
import hashlib
import json
import os
import subprocess
import sys
import textwrap

ROOT = Path("/kaggle/working")
REPO = ROOT / "RTC-SDD-Research"
DATA = Path("/kaggle/input/datasets/a7medlabib/rtcfake-packed-private")
CACHE = ROOT / "e02_pcl_cache/v1/train"
PCL_MODULE = REPO / "src/rtcsdd/training/phoneme_contrastive.py"
PAIR_CSV = DATA / "train_offline_online_pairs.csv"
CHECKPOINT = ROOT / "checkpoints/pretrained/xlsr2_300m.pt"
REPORT = ROOT / "e02_real_model_pcl_smoke_report.json"

print("=" * 72, flush=True)
print("E02 — REAL MODEL + PCL GPU GRADIENT SMOKE (STRICT CSV ID FIX)", flush=True)
print("=" * 72, flush=True)

# First fail fast on missing inputs. This cell NEVER starts fine-tuning.
required = {
    "research repository": REPO,
    "PCL loss module": PCL_MODULE,
    "paired train manifest": PAIR_CSV,
    "PCL train cache": CACHE,
    "pretrained XLS-R checkpoint": CHECKPOINT,
    "official model": (
        REPO / "third_party/official_rtc_sdd/xlsr_aasist/model/model.py"
    ),
    "train labels": DATA / "train_label.txt",
    "offline train PACK": DATA / "train_offline.pack",
    "online train PACK": DATA / "train_online.pack",
}
for name, p in required.items():
    if not p.exists():
        raise FileNotFoundError(
            f"[MISSING] {name}: {p}\n"
            "Restore the relevant E02 setup artifacts before rerunning."
        )
    print("[PASS]", name, flush=True)

# Run with the exact Python environment that has CUDA and Fairseq.
candidates = [
    ROOT / "rtcsdd-baseline-py310/bin/python",
    Path(sys.executable),
]
seen = set()
chosen = None
probe = """
import numpy as np
for name, value in {
    'float': float, 'int': int, 'complex': complex,
    'bool': bool, 'object': object, 'str': str,
}.items():
    if name not in np.__dict__:
        setattr(np, name, value)
import torch
import fairseq
assert torch.cuda.is_available(), "CUDA unavailable"
print(torch.__version__, torch.cuda.get_device_name(0))
"""
for candidate in candidates:
    if str(candidate) in seen or not candidate.is_file():
        continue
    seen.add(str(candidate))
    p = subprocess.run(
        [str(candidate), "-c", probe],
        text=True, capture_output=True
    )
    if p.returncode == 0:
        chosen = candidate
        print("[PASS] Compatible training runtime:", candidate, flush=True)
        print("       ", p.stdout.strip(), flush=True)
        break
    print("[INFO] Runtime not ready:", candidate, flush=True)
    print("       ", (p.stderr or p.stdout).strip().splitlines()[-1:], flush=True)

if chosen is None:
    raise RuntimeError(
        "No CUDA+Fairseq Python environment found in E02. "
        "Prepare the E01-style Python 3.10 training environment first; "
        "do NOT rerun phoneme preprocessing."
    )

# The subprocess is isolated from the notebook's Python installation.
driver = r"""
from pathlib import Path
import csv
import gzip
import hashlib
import json
import os
import sys
import torch
import torch.nn.functional as F

root = Path("/kaggle/working")
repo = root / "RTC-SDD-Research"
data = Path("/kaggle/input/datasets/a7medlabib/rtcfake-packed-private")
cache_path = root / "e02_pcl_cache/v1/train/train_000000_000256.json.gz"
pairs_csv = data / "train_offline_online_pairs.csv"
checkpoint = root / "checkpoints/pretrained/xlsr2_300m.pt"
report_path = root / "e02_real_model_pcl_smoke_report.json"

sys.path.insert(0, str(repo / "src"))
from rtcsdd.data.protocol import read_protocol
from rtcsdd.data.packed import PackedAudioStore
from rtcsdd.data.audio import crop_or_pad
from rtcsdd.models.xlsr_aasist import load_xlsr_aasist
from rtcsdd.training.phoneme_contrastive import phoneme_contrastive_loss

print("\n[1/4] CACHE ↔ PAIR MANIFEST IDENTITY", flush=True)
with gzip.open(cache_path, "rt", encoding="utf-8") as f:
    payload = json.load(f)
metadata = payload["metadata"]

# Hash checking prevents accidental row-index alignment to a different manifest.
digest = hashlib.sha256()
with pairs_csv.open("rb") as f:
    for block in iter(lambda: f.read(1024 * 1024), b""):
        digest.update(block)
manifest_sha = digest.hexdigest()
expected_sha = metadata.get("manifest_sha256")
assert expected_sha == manifest_sha, (
    "Paired CSV SHA256 does not equal cached manifest_sha256. "
    "STOP: the cache must not be paired with the wrong audio IDs.\n"
    f"Cache: {expected_sha}\n"
    f"CSV:   {manifest_sha}"
)
print("[PASS] Pair CSV matches cache manifest SHA256", flush=True)

with pairs_csv.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    headers = reader.fieldnames
    assert headers, "Paired CSV has no header"
    first = next(reader, None)

assert first is not None, "Paired CSV is empty"

def find_column(side):
    opts = [
        c for c in headers
        if side in c.lower()
        and not any(k in c.lower() for k in (
            "hash", "label", "score", "index", "frame", "duration"
        ))
    ]
    wav_opts = [c for c in opts if ".wav" in str(first[c]).lower()]
    if len(wav_opts) == 1:
        return wav_opts[0]
    if len(opts) == 1:
        return opts[0]
    raise RuntimeError(
        f"Cannot unambiguously identify {side} ID column. "
        f"Headers: {headers}; candidate columns: {opts}"
    )

offline_col, online_col = find_column("offline"), find_column("online")
assert offline_col != online_col

rec = payload["records"][0]
assert rec["pair_index"] == 0
assert rec["matches"], "First pair has no phoneme matches"
assert metadata["audio_length"] == 64600
assert rec["offline_frames"] == rec["online_frames"] == 201

labels = {
    e.utterance_id: e
    for e in read_protocol(data / "train_label.txt", "train")
}

# The paired CSV sometimes stores a relative ID rather than the fully
# qualified protocol ID (e.g. without offline/en/ prefix).
# Resolve ONLY when the match in the authoritative training protocol
# is unique. Never silently match an ambiguous filename.
from pathlib import PurePosixPath

def resolve_id(value, side, row):
    raw = str(value).strip().strip("\"'").replace(chr(92), "/")
    raw = raw.split("?", 1)[0]  # ignore optional URI query only
    raw = raw.removeprefix("./")
    raw = raw.removeprefix("/")
    raw = raw.strip()
    if not raw or raw.lower() in {"none", "nan"}:
        raise ValueError(f"Empty paired {side} value")

    side_ids = sorted(uid for uid, ex in labels.items() if ex.view == side)
    # A candidate is valid only if it's a whole POSIX path suffix.
    candidates = [uid for uid in side_ids if (
        uid == raw or uid.endswith("/" + raw)
    )]
    # If a CSV field includes an extra absolute storage prefix, match the
    # protocol ID as a whole suffix of that field.
    if not candidates:
        candidates = [uid for uid in side_ids if raw.endswith("/" + uid)]

    # Explicit language, when present in CSV, can disambiguate a basename.
    if len(candidates) > 1:
        language_hints = {
            str(v).strip().lower()
            for k, v in row.items()
            if k and ("lang" in k.lower() or k.lower() in {"locale"})
            and str(v).strip().lower() in {"en", "zh"}
        }
        if len(language_hints) == 1:
            language = next(iter(language_hints))
            candidates = [uid for uid in candidates
                          if uid.startswith(f"{side}/{language}/")]

    if len(candidates) != 1:
        print("[DIAGNOSTIC] Paired CSV columns:", headers, flush=True)
        print("[DIAGNOSTIC] CSV first pair values:", {
            str(k): str(v)[:160] for k, v in row.items()
        }, flush=True)
        print("[DIAGNOSTIC] Protocol examples for side:",
              side_ids[:5], flush=True)
        print("[DIAGNOSTIC] Candidate count:", len(candidates), flush=True)
        raise RuntimeError(
            f"Cannot uniquely map CSV {side} value to train_label.txt. "
            "No audio was loaded and no training started."
        )
    return candidates[0]

off_id = resolve_id(first[offline_col], "offline", first)
on_id = resolve_id(first[online_col], "online", first)
off_example, on_example = labels[off_id], labels[on_id]
assert off_example.view == "offline" and on_example.view == "online"
print("[PASS] Pair index 0 resolved in training labels", flush=True)
print("[INFO] Offline label:", off_example.label, flush=True)
print("[INFO] Online label:", on_example.label, flush=True)
print("[INFO] Matched phonemes:", len(rec["matches"]), flush=True)

print("\n[2/4] LOAD REAL PAIRED AUDIO", flush=True)
store = PackedAudioStore(data)
try:
    off_wav, off_sr = store.read(off_example)
    on_wav, on_sr = store.read(on_example)
finally:
    store.close()
assert off_sr == on_sr == 16000

off_fixed = crop_or_pad(
    off_wav, length=64600,
    crop=metadata["crop"],
    pad=metadata["pad"]
)
on_fixed = crop_or_pad(
    on_wav, length=64600,
    crop=metadata["crop"],
    pad=metadata["pad"]
)
audio = torch.tensor(
    __import__("numpy").stack([off_fixed, on_fixed]),
    dtype=torch.float32,
    device="cuda:0"
)
targets = torch.tensor(
    [off_example.label, on_example.label],
    dtype=torch.long, device="cuda:0"
)
assert tuple(audio.shape) == (2, 64600)
print("[PASS] Paired real audio:", tuple(audio.shape), flush=True)

print("\n[3/4] FORWARD: ONE XLS-R PASS PER RECORDING", flush=True)
torch.manual_seed(1234)
torch.backends.cuda.matmul.allow_tf32 = False
model = load_xlsr_aasist(
    repo / "third_party/official_rtc_sdd",
    checkpoint,
    "cuda:0"
)
model.train()

assert hasattr(model, "ssl_model")
assert hasattr(model.ssl_model, "model")
feature_tensors = []

def capture_fairseq_features(module, args, output):
    assert isinstance(output, dict) and "x" in output, (
        "Unexpected Fairseq features_only output"
    )
    feature_tensors.append(output["x"])

handle = model.ssl_model.model.register_forward_hook(
    capture_fairseq_features
)
try:
    logits = model(audio)
finally:
    handle.remove()

assert len(feature_tensors) == 1, (
    f"Expected one captured XLS-R forward call; "
    f"got {len(feature_tensors)}"
)
features = feature_tensors[0]
assert tuple(logits.shape) == (2, 2), (
    f"Unexpected classification logits: {tuple(logits.shape)}"
)
assert features.ndim == 3 and features.shape[0] == 2, (
    f"Unexpected XLS-R features: {tuple(features.shape)}"
)
assert features.shape[1] == 201, (
    f"CTC/XLS-R time dimension mismatch: "
    f"cache=201; XLS-R={features.shape[1]}"
)
assert features.shape[2] == 1024

class_weights = torch.tensor(
    [0.1, 0.9],
    dtype=torch.float32, device="cuda:0"
)
ce = F.cross_entropy(logits.float(), targets, weight=class_weights)
pcl = phoneme_contrastive_loss(
    features[0],
    features[1],
    rec["matches"],
    offline_frames=rec["offline_frames"],
    online_frames=rec["online_frames"],
    temperature=0.1,
    max_matches=96,
)
assert torch.isfinite(ce) and torch.isfinite(pcl)

print("[PASS] XLS-R feature shape:", tuple(features.shape), flush=True)
print("[PASS] AASIST logits shape:", tuple(logits.shape), flush=True)
print("[INFO] CE test loss:", float(ce.detach()), flush=True)
print("[INFO] PCL test loss:", float(pcl.detach()), flush=True)

print("\n[4/4] DIFFERENTIABLE BACKWARD — NO OPTIMIZER STEP", flush=True)

# Verify explicitly that PCL contributes gradient to XLS-R output.
pcl_feature_grad = torch.autograd.grad(
    pcl, features, retain_graph=True
)[0]
assert torch.isfinite(pcl_feature_grad).all()
assert float(pcl_feature_grad.norm().detach()) > 0

# lambda is for the smoke test only; not a selected training hyperparameter.
lambda_smoke = 0.1
total = ce + lambda_smoke * pcl
assert torch.isfinite(total)
total.backward()

ssl_count = ssl_nonzero = head_count = head_nonzero = 0
for name, param in model.named_parameters():
    if not param.requires_grad or param.grad is None:
        continue
    grad = param.grad
    assert torch.isfinite(grad).all(), f"Nonfinite gradient: {name}"
    if name.startswith("ssl_model."):
        ssl_count += 1
        ssl_nonzero += int(bool(torch.any(grad != 0)))
    else:
        head_count += 1
        head_nonzero += int(bool(torch.any(grad != 0)))

assert ssl_count > 0 and ssl_nonzero > 0, (
    "No nonzero gradients in XLS-R"
)
assert head_count > 0 and head_nonzero > 0, (
    "No nonzero gradients in AASIST"
)

report = {
    "experiment": "E02_PCL",
    "stage": "real_model_gpu_smoke",
    "pair_csv_matches_cache_manifest_sha256": True,
    "pair_index": 0,
    "real_audio": True,
    "feature_shape": list(features.shape),
    "logits_shape": list(logits.shape),
    "ce_smoke_loss": float(ce.detach()),
    "pcl_smoke_loss": float(pcl.detach()),
    "pcl_feature_grad_norm": float(
        pcl_feature_grad.norm().detach()
    ),
    "xlsr_grad_tensors": ssl_count,
    "xlsr_nonzero_grad_tensors": ssl_nonzero,
    "aasist_grad_tensors": head_count,
    "aasist_nonzero_grad_tensors": head_nonzero,
    "optimizer_created": False,
    "optimizer_step_performed": False,
    "checkpoint_modified": False,
    "full_finetuning_started": False,
    "lambda_smoke_only": lambda_smoke,
}
report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8"
)
print("[PASS] PCL gradients reached XLS-R features", flush=True)
print(f"[PASS] XLS-R nonzero grad tensors: "
      f"{ssl_nonzero}/{ssl_count}", flush=True)
print(f"[PASS] AASIST nonzero grad tensors: "
      f"{head_nonzero}/{head_count}", flush=True)
print("[PASS] No optimizer step; pretrained weights unchanged", flush=True)
print("[REPORT]", report_path, flush=True)
print("=" * 72, flush=True)
print("E02 — REAL MODEL PCL GPU SMOKE: PASS", flush=True)
print("=" * 72, flush=True)
"""

env = dict(os.environ)
env["PYTHONPATH"] = (
    str(REPO / "src")
    + os.pathsep
    + env.get("PYTHONPATH", "")
)
print("\n[START] Real GPU forward/backward smoke (no training)", flush=True)
subprocess.run(
    [str(chosen), "-u", "-c", driver],
    env=env, cwd=REPO, check=True
)


In [ ]:
# ============================================================
# E02 — VERIFIED CE + PCL TRAINER INTEGRATION / PRE-FLIGHT
# KEEP | CPU ONLY | NO FINETUNING | NO MODEL DOWNLOAD
# Creates NEW independent module; refuses to overwrite edits
# ============================================================
from pathlib import Path
import hashlib
import json
import os
import subprocess

ROOT = Path('/kaggle/working')
REPO = ROOT / 'RTC-SDD-Research'
DATA = Path('/kaggle/input/datasets/a7medlabib/rtcfake-packed-private')
CACHE = ROOT / 'e02_pcl_cache/v1'
PRETRAINED = ROOT / 'checkpoints/pretrained/xlsr2_300m.pt'
TRAINING_PY = ROOT / 'rtcsdd-baseline-py310/bin/python'
MODULE = REPO / 'src/rtcsdd/training/e02_pcl_trainer.py'
CONFIG = REPO / 'src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json'
OUT = REPO / 'outputs/experiments/e02_ce_pcl_fp32'
REPORT = ROOT / 'e02_trainer_integration_preflight.json'

print('=' * 75)
print('E02 — CE + PCL TRAINER INTEGRATION AND FULL CONTRACT PREFLIGHT')
print('=' * 75)

for label, path in {
    'repository': REPO,
    'paired dataset': DATA,
    'PCL cache': CACHE,
    'original pretrained XLS-R': PRETRAINED,
    'baseline config': CONFIG,
    'Python 3.10 training runtime': TRAINING_PY,
    'PCL loss': REPO / 'src/rtcsdd/training/phoneme_contrastive.py',
    'pair sampler': REPO / 'src/rtcsdd/data/paired_batches.py',
}.items():
    if not path.exists():
        raise FileNotFoundError(f'[MISSING] {label}: {path}')
    print('[PASS]', label, flush=True)

# ------------------------------------------------------------
# 1. CREATE CANONICAL INDEPENDENT E02 TRAINER, NO OVERWRITES
# ------------------------------------------------------------
SOURCE = '"""E02: Independent, fail-closed CE + phoneme-aligned PCL trainer.\n\nUses original XLS-R/AASIST weights, paired batches, all labeled recordings,\ncache-bound hashes, explicit per-window gradient normalization, atomic resume,\nclean full Dev and the existing HF backup manager.\n\nThis file is CREATED by the Kaggle integration cell, not run until the launch cell.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gzip\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport time\nfrom pathlib import Path, PurePosixPath\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom rtcsdd.data.protocol import read_protocol\nfrom rtcsdd.data.dataset import RTCFakePackedDataset\nfrom rtcsdd.data.paired_batches import PairPreservingBatchSampler\nfrom rtcsdd.data.phoneme_cache import pair_key_sha256\nfrom rtcsdd.models.xlsr_aasist import load_xlsr_aasist\nfrom rtcsdd.training.phoneme_contrastive import phoneme_contrastive_loss\nfrom rtcsdd.training.config import fingerprint\nfrom rtcsdd.training.checkpointing import (\n    save_resume, read_resume, save_best_model, write_json_atomic,\n)\nfrom rtcsdd.training.trainer import rng_snapshot, restore_rng\nfrom rtcsdd.eval.dev_evaluator import evaluate_full_dev\nfrom rtcsdd.eval.competition import dev_online_macro_f1\nfrom rtcsdd.storage.backup import CheckpointBackupManager\n\nEXPECTED_COUNTS = {\'train\': (75785, 37125, 1535), \'dev\': (15204, 7465, 274)}\nREQUIRED_CACHE_FIELDS = {\'pair_hash\', \'pair_index\', \'offline_frames\', \'online_frames\', \'matches\'}\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef canonical_id(value: str, side: str, split: str) -> str:\n    """Trust only an exact trailing side/language/filename path, not a basename."""\n    raw = str(value).strip().replace(\'\\\\\', \'/\')\n    if not raw or raw.lower() in {\'nan\', \'none\', \'null\'}:\n        raise ValueError(f\'Missing {side} identity\')\n    parts = PurePosixPath(raw).parts\n    found = []\n    for i in range(len(parts) - 2):\n        if parts[i] == side and parts[i+1] in {\'en\', \'zh\'}:\n            if i+3 == len(parts) and parts[i+2].endswith(\'.wav\'):\n                found.append(\'/\'.join(parts[i:i+3]))\n    if len(found) != 1:\n        raise ValueError(f\'Invalid or ambiguous {split}/{side} path, suffix={raw[-90:]}\')\n    return found[0]\n\n\ndef read_pair_groups(dataset_root: Path, split: str, examples):\n    csv_path = dataset_root / f\'{split}_offline_online_pairs.csv\'\n    uid_map = {ex.utterance_id: ex for ex in examples}\n    if len(uid_map) != len(examples):\n        raise ValueError(\'Duplicate train/dev protocol IDs\')\n    groups = []\n    complete = {}\n    singleton_count = 0\n    with csv_path.open(\'r\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        reader = csv.DictReader(stream)\n        if reader.fieldnames != [\'offline_id\', \'online_id\']:\n            raise ValueError(f\'Unexpected {split} paired CSV schema: {reader.fieldnames}\')\n        for rowno, row in enumerate(reader, 2):\n            off_id = canonical_id(row[\'offline_id\'], \'offline\', split)\n            off = uid_map.get(off_id)\n            if off is None or off.view != \'offline\':\n                raise ValueError(f\'{split} CSV row {rowno}: unknown offline recording\')\n            value = str(row[\'online_id\'] or \'\').strip()\n            if not value:\n                groups.append(SimpleNamespace(offline=off, online=None))\n                singleton_count += 1\n                continue\n            on_id = canonical_id(value, \'online\', split)\n            on = uid_map.get(on_id)\n            if on is None or on.view != \'online\':\n                raise ValueError(f\'{split} CSV row {rowno}: unknown online recording\')\n            key = pair_key_sha256(off_id, on_id)\n            if key in complete:\n                raise ValueError(f\'{split}: duplicate complete-pair hash\')\n            complete[key] = (off_id, on_id)\n            groups.append(SimpleNamespace(offline=off, online=on))\n    expected_n, expected_pairs, expected_singletons = EXPECTED_COUNTS[split]\n    if (len(examples), len(complete), singleton_count) != (\n        expected_n, expected_pairs, expected_singletons\n    ):\n        raise ValueError(\n            f\'{split} unexpected counts: samples={len(examples)}, \'\n            f\'complete_pairs={len(complete)}, offline_only={singleton_count}\'\n        )\n    return groups, complete, singleton_count, digest(csv_path)\n\n\ndef validate_cache(cache_root: Path, split: str, manifest_sha: str, full_pairs: dict):\n    """SHA verify and join ALL records by identity hash; forbid guessing indices."""\n    paths = sorted((cache_root / split).glob(\'*.json.gz\'))\n    n_expected = EXPECTED_COUNTS[split][1]\n    if len(paths) != math.ceil(n_expected / 256):\n        raise ValueError(f\'{split}: unexpected cache shard count {len(paths)}\')\n    seen = {}\n    cursor = 0\n    ref_metadata = None\n    for path in paths:\n        sidecar = Path(str(path) + \'.sha256\')\n        if not sidecar.is_file():\n            raise FileNotFoundError(sidecar)\n        claimed = sidecar.read_text(encoding=\'utf-8\').strip().split()[0].lower()\n        if len(claimed) != 64 or digest(path) != claimed:\n            raise ValueError(f\'Cache SHA256 mismatch: {path.name}\')\n        with gzip.open(path, \'rt\', encoding=\'utf-8\') as stream:\n            payload = json.load(stream)\n        metadata, records = payload[\'metadata\'], payload[\'records\']\n        if metadata[\'manifest_sha256\'] != manifest_sha:\n            raise ValueError(f\'{split}: cache/CSV manifest checksum mismatch\')\n        if metadata[\'start\'] != cursor or metadata[\'stop\'] != cursor + len(records):\n            raise ValueError(f\'{split}: shard indices not contiguous at {path.name}\')\n        stable = {k: v for k, v in metadata.items() if k not in {\'start\', \'stop\'}}\n        if ref_metadata is None:\n            ref_metadata = stable\n        elif ref_metadata != stable:\n            raise ValueError(f\'{split}: cache generation metadata inconsistency\')\n        for record in records:\n            if set(record) != REQUIRED_CACHE_FIELDS:\n                raise ValueError(f\'{split}: unexpected cache record schema\')\n            if record[\'pair_index\'] != cursor:\n                raise ValueError(f\'{split}: noncontiguous record pair_index={record["pair_index"]}\')\n            cursor += 1\n            key = record[\'pair_hash\']\n            if key in seen or key not in full_pairs:\n                raise ValueError(f\'{split}: unknown or duplicate pair hash {key[:14]}\')\n            if record[\'offline_frames\'] != 201 or record[\'online_frames\'] != 201:\n                raise ValueError(\'Unexpected feature frame count: cache expects 201\')\n            seen[key] = record\n    if set(seen) != set(full_pairs) or cursor != n_expected:\n        raise ValueError(f\'{split}: pair/cache hash join incomplete or inconsistent\')\n    return seen, ref_metadata\n\n\ndef build_contract(repo: Path, data: Path, cache: Path):\n    contexts = {}\n    for split in (\'train\', \'dev\'):\n        examples = read_protocol(data / f\'{split}_label.txt\', split)\n        groups, pairmap, alone, manifest_sha = read_pair_groups(data, split, examples)\n        cached, meta = validate_cache(cache, split, manifest_sha, pairmap)\n        contexts[split] = {\n            \'examples\': examples, \'groups\': groups, \'pairmap\': pairmap,\n            \'cache\': cached, \'offline_only\': alone,\n            \'manifest_sha256\': manifest_sha, \'metadata\': meta,\n        }\n        sampler = PairPreservingBatchSampler(examples, groups, seed=1234, epoch=0)\n        assert sampler.paired_batch_count == len(pairmap)\n        assert sampler.ce_only_batch_count == math.ceil(alone / 2)\n        actual = list(sampler)\n        assert sum(map(len, actual)) == len(examples)\n        assert len({i for batch in actual for i in batch}) == len(examples)\n        print(f\'[PASS] {split}: samples={len(examples):,} \'\n              f\'paired={len(pairmap):,} offline_only={alone:,} \'\n              f\'microbatches={len(sampler):,}; hash joins=100%\', flush=True)\n    assert contexts[\'train\'][\'metadata\'][\'audio_length\'] == 64600\n    return contexts\n\n\ndef e02_hash(base_config, contexts, args, module: Path) -> str:\n    # Config includes all training controls; source hash prevents changed loss on resume.\n    payload = {\n        \'experiment\': \'e02_ce_pcl_fp32\',\n        \'base_training_fingerprint\': fingerprint(base_config),\n        \'train_config\': base_config[\'training\'],\n        \'data_config\': base_config[\'data\'],\n        \'lambda_pcl\': args.lambda_pcl,\n        \'temperature\': args.temperature,\n        \'max_matches\': args.max_matches,\n        \'loss_sha256\': digest(module),\n        \'trainer_sha256\': digest(Path(__file__)),\n        \'pair_manifests\': {\n            split: contexts[split][\'manifest_sha256\'] for split in (\'train\', \'dev\')\n        },\n        \'phoneme_cache_metadata\': {\n            split: contexts[split][\'metadata\'] for split in (\'train\', \'dev\')\n        },\n    }\n    # Epoch count and backup settings are not included, to allow a genuine continuation.\n    payload[\'train_config\'] = {\n        k: v for k, v in payload[\'train_config\'].items()\n        if k not in {\'epochs\', \'max_minutes\'}\n    }\n    return hashlib.sha256(json.dumps(\n        payload, sort_keys=True, separators=(\',\', \':\'), ensure_ascii=True\n    ).encode()).hexdigest()\n\n\ndef configure_seed(seed: int):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    torch.backends.cudnn.deterministic = True\n    torch.backends.cudnn.benchmark = False\n\n\ndef get_loader(dataset, batch_sampler=None, *, batch_size=None, workers=2, seed=1234):\n    g = torch.Generator().manual_seed(seed)\n    kw = {\'num_workers\': workers, \'pin_memory\': True, \'generator\': g}\n    if batch_sampler is not None:\n        return DataLoader(dataset, batch_sampler=batch_sampler, **kw)\n    return DataLoader(dataset, batch_size=batch_size, shuffle=False, **kw)\n\n\ndef run(args):\n    repo = args.repo.resolve()\n    data = args.data.resolve()\n    cache = args.cache.resolve()\n    config_path = args.config.resolve()\n    pretrain = args.pretrained.resolve()\n    out = args.out.resolve()\n    module = repo / \'src/rtcsdd/training/phoneme_contrastive.py\'\n    if not (repo / \'third_party/official_rtc_sdd/xlsr_aasist/model/model.py\').is_file():\n        raise FileNotFoundError(\'Pinned AASIST model missing\')\n    if not module.is_file():\n        raise FileNotFoundError(\'PCL module missing\')\n    config = json.loads(config_path.read_text(encoding=\'utf-8\'))\n    config[\'experiment\'] = \'e02_ce_pcl_fp32\'\n    config[\'training\'][\'epochs\'] = args.epochs\n    config[\'training\'][\'max_minutes\'] = args.max_minutes\n    config[\'training\'][\'precision\'] = \'fp32\'\n    config[\'checkpoint_backup\'] = {\n        \'enabled\': True,\n        \'repo_id\': \'AhmedLabib1/rtc-sdd-research-checkpoints\',\n        \'remote_prefix\': \'experiments/e02_ce_pcl_fp32\',\n        \'upload_every_optimizer_steps\': 1000,\n        \'upload_on_pause\': True,\n        \'upload_after_epoch\': True,\n        \'upload_best_after_epoch\': True,\n        \'max_attempts\': 2,\n        \'retry_delay_seconds\': 2,\n        \'create_private\': False,\n        \'delete_local_after_upload\': False,\n    }\n    if not (0 <= args.lambda_pcl <= 1):\n        raise ValueError(\'lambda_pcl must be in [0,1]\')\n    if not (0 < args.temperature <= 1):\n        raise ValueError(\'temperature must be in (0,1]\')\n    if args.max_matches < 2:\n        raise ValueError(\'max_matches must be >=2\')\n    train_cfg = config[\'training\']\n    data_cfg = config[\'data\']\n    if train_cfg[\'batch_size\'] != 2 or train_cfg[\'gradient_accumulation\'] != 8:\n        raise ValueError(\'Expected baseline microbatch 2 x accumulation 8\')\n    if train_cfg[\'class_weights\'] != [0.1, 0.9] or float(train_cfg[\'learning_rate\']) != 1e-6:\n        raise ValueError(\'Unexpected E01-comparable loss weights or LR\')\n    if data_cfg[\'audio_length\'] != 64600:\n        raise ValueError(\'Unexpected input audio length\')\n    contexts = build_contract(repo, data, cache)\n    for split in (\'train\',\'dev\'):\n        meta = contexts[split][\'metadata\']\n        if data_cfg[\'crop\'] != meta[\'crop\'] or data_cfg[\'pad\'] != meta[\'pad\']:\n            raise ValueError(\'E02 cache crop/pad differs from audio training config\')\n    config_hash = e02_hash(config, contexts, args, module)\n    print(\'[PASS] E02 exact cache/manifest contract; config hash\', config_hash, flush=True)\n    if args.preflight_only:\n        return\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'CUDA required for E02 fine-tuning\')\n    if not pretrain.is_file():\n        raise FileNotFoundError(pretrain)\n    if out.name != \'e02_ce_pcl_fp32\':\n        raise ValueError(\'E02 output directory must be named e02_ce_pcl_fp32\')\n    if args.resume:\n        if not (out / \'last.pt\').is_file() or not (out / \'config.json\').is_file():\n            raise FileNotFoundError(\'Cannot resume without original config.json and last.pt\')\n        old = json.loads((out/\'config.json\').read_text(encoding=\'utf-8\'))\n        if old[\'config_fingerprint\'] != config_hash:\n            raise RuntimeError(\'E02 fingerprint mismatch. Refusing unsafe resume.\')\n    else:\n        if (out / \'last.pt\').exists() or (out / \'config.json\').exists():\n            raise FileExistsError(\'Fresh E02 output exists; use --resume, not overwrite\')\n    out.mkdir(parents=True, exist_ok=True)\n    configure_seed(train_cfg[\'seed\'])\n    print(\'[MODEL] Loading original pretrained XLS-R+AASIST\', flush=True)\n    model = load_xlsr_aasist(repo / \'third_party/official_rtc_sdd\', pretrain, \'cuda:0\')\n    model.train()\n    for p in model.parameters():\n        p.requires_grad_(True)\n    ssl = [p for n,p in model.named_parameters() if n.startswith(\'ssl_model.\')]\n    head = [p for n,p in model.named_parameters() if not n.startswith(\'ssl_model.\')]\n    assert ssl and head\n    optimizer = torch.optim.Adam(model.parameters(), lr=train_cfg[\'learning_rate\'],\n                                 weight_decay=train_cfg[\'weight_decay\'])\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=False)\n    class_weights = torch.tensor([0.1,0.9], device=\'cuda:0\', dtype=torch.float32)\n    backup = CheckpointBackupManager(config, out)\n    if not args.resume:\n        write_json_atomic({\n            \'config\': config, \'config_fingerprint\': config_hash,\n            \'dataset_root\': str(data), \'pretrained\': str(pretrain),\n            \'pair_cache_root\': str(cache),\n            \'objective\': \'weighted_ce_plus_lambda_symmetric_phoneme_infonce\',\n            \'lambda_pcl\': args.lambda_pcl, \'temperature\': args.temperature,\n            \'max_matches\': args.max_matches,\n        }, out/\'config.json\')\n    epoch0 = batch_cursor = global_step = 0\n    best_dev_loss = math.inf\n    best_online_f1 = -1.0\n    no_improve_epochs = 0\n    saved_rng = None\n    if args.resume:\n        ckpt = read_resume(out/\'last.pt\', expected_fingerprint=config_hash)\n        model.load_state_dict(ckpt[\'model\'])\n        optimizer.load_state_dict(ckpt[\'optimizer\'])\n        scaler.load_state_dict(ckpt[\'scaler\'])\n        epoch0, batch_cursor, global_step = ckpt[\'epoch\'], ckpt[\'next_batch\'], ckpt[\'global_step\']\n        best_dev_loss, best_online_f1 = ckpt[\'best_dev_loss\'], ckpt[\'best_online_macro_f1\']\n        no_improve_epochs, saved_rng = ckpt[\'no_improve_epochs\'], ckpt[\'rng\']\n        del ckpt\n        print(f\'[RESUME] epoch={epoch0+1} next_batch={batch_cursor} step={global_step}\', flush=True)\n    dev_ctx = contexts[\'dev\']\n    dev_data = RTCFakePackedDataset(dev_ctx[\'examples\'], data,\n                                     length=64600, crop=data_cfg[\'crop\'], pad=data_cfg[\'pad\'],\n                                     rawboost={\'enabled\': False}, seed=train_cfg[\'seed\'])\n    dev_loader = get_loader(dev_data, batch_size=train_cfg[\'eval_batch_size\'],\n                            workers=train_cfg[\'workers\'], seed=train_cfg[\'seed\']+1000000)\n    train_ctx = contexts[\'train\']\n    train_data = RTCFakePackedDataset(train_ctx[\'examples\'], data,\n                                      length=64600, crop=data_cfg[\'crop\'], pad=data_cfg[\'pad\'],\n                                      rawboost=data_cfg[\'rawboost\'],\n                                      upstream_root=repo / \'third_party/official_rtc_sdd\',\n                                      seed=train_cfg[\'seed\'])\n    max_save_interval = train_cfg.get(\'save_every_optimizer_steps\', 1000)\n    log_interval = train_cfg.get(\'log_every_optimizer_steps\', 100)\n    t0 = time.monotonic()\n    print(\'[TRAIN] fp32 pair-preserving, CE + PCL; full Dev; checkpoint+HF backup enabled\', flush=True)\n    print(\'[TRAIN] original pretrained, not E01 fine-tuned weights\', flush=True)\n\n    def snapshot(ep, next_batch, reason):\n        save_resume(out/\'last.pt\', model=model, optimizer=optimizer, scaler=scaler,\n                    config_hash=config_hash, epoch=ep, next_batch=next_batch,\n                    global_step=global_step, best_dev_loss=best_dev_loss,\n                    best_online_macro_f1=best_online_f1, no_improve_epochs=no_improve_epochs,\n                    rng_state=rng_snapshot(), accumulated_samples=0)\n        print(f\'[SAVED] last.pt epoch={ep+1} next_batch={next_batch} step={global_step}\', flush=True)\n        backup.saved_last(epoch=ep+1, global_step=global_step, reason=reason)\n\n    for ep in range(epoch0, args.epochs):\n        sampler = PairPreservingBatchSampler(train_ctx[\'examples\'], train_ctx[\'groups\'],\n                                              seed=train_cfg[\'seed\'], epoch=ep)\n        microbatches = list(sampler)\n        if ep != epoch0:\n            batch_cursor = 0\n        if not 0 <= batch_cursor <= len(microbatches):\n            raise ValueError(\'Invalid resume microbatch cursor\')\n        train_data.set_epoch(ep)\n        model.train()\n        if batch_cursor < len(microbatches):\n            # Pre-compute window denominators: correct CE and PCL normalization.\n            sliced = microbatches[batch_cursor:]\n            window_size = train_cfg[\'gradient_accumulation\']\n            train_loader = get_loader(train_data,\n                batch_sampler=sliced, workers=train_cfg[\'workers\'],\n                seed=train_cfg[\'seed\']+ep)\n            iterator = iter(train_loader)\n            if saved_rng is not None:\n                restore_rng(saved_rng)\n                saved_rng = None\n            cursor = batch_cursor\n            optimizer.zero_grad(set_to_none=True)\n            all_examples = train_ctx[\'examples\']\n            cache_by_hash = train_ctx[\'cache\']\n            for win_start in range(0, len(sliced), window_size):\n                window = sliced[win_start:win_start + window_size]\n                denom = 0.0\n                pcl_slots = 0\n                for batch_indices in window:\n                    denom += sum((0.1 if all_examples[i].label == 0 else 0.9) for i in batch_indices)\n                    if len(batch_indices) == 2:\n                        aa, bb = [all_examples[i].utterance_id for i in batch_indices]\n                        if sampler.is_pair_batch((aa,bb)):\n                            matched = cache_by_hash[pair_key_sha256(aa,bb)][\'matches\']\n                            if len({m[0] for m in matched}) >= 2:\n                                pcl_slots += 1\n                if denom <= 0:\n                    raise ValueError(\'Zero window loss denominator\')\n                ce_logged = pcl_logged = 0.0\n                for batch_indices in window:\n                    audio, labels, ids = next(iterator)\n                    if tuple(ids) != tuple(all_examples[i].utterance_id for i in batch_indices):\n                        raise RuntimeError(\'DataLoader batch order differs from pair sampler\')\n                    audio, labels = audio.to(\'cuda:0\'), labels.to(\'cuda:0\')\n                    feats = []\n                    def capture(module, inputs, output):\n                        if not isinstance(output,dict) or \'x\' not in output:\n                            raise RuntimeError(\'Unexpected XLS-R feature interface\')\n                        feats.append(output[\'x\'])\n                    hook = model.ssl_model.model.register_forward_hook(capture)\n                    try:\n                        logits = model(audio)\n                    finally:\n                        hook.remove()\n                    if len(feats) != 1 or tuple(feats[0].shape[1:]) != (201,1024):\n                        raise RuntimeError(f\'Unexpected XLS-R features: {[x.shape for x in feats]}\')\n                    ce_num = F.cross_entropy(logits.float(), labels,\n                                             weight=class_weights, reduction=\'sum\')\n                    if not torch.isfinite(ce_num):\n                        raise FloatingPointError(\'Nonfinite weighted CE\')\n                    loss = ce_num / denom\n                    ce_logged += float(ce_num.detach())\n                    if len(ids) == 2 and sampler.is_pair_batch(ids):\n                        cache_rec = cache_by_hash[pair_key_sha256(ids[0],ids[1])]\n                        if len({m[0] for m in cache_rec[\'matches\']}) >= 2:\n                            pcl = phoneme_contrastive_loss(\n                                feats[0][0], feats[0][1], cache_rec[\'matches\'],\n                                offline_frames=cache_rec[\'offline_frames\'],\n                                online_frames=cache_rec[\'online_frames\'],\n                                temperature=args.temperature, max_matches=args.max_matches,\n                            )\n                            if not torch.isfinite(pcl):\n                                raise FloatingPointError(\'Nonfinite PCL\')\n                            assert pcl_slots > 0\n                            loss = loss + (args.lambda_pcl / pcl_slots) * pcl\n                            pcl_logged += float(pcl.detach())\n                    loss.backward()\n                    cursor += 1\n                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(),\n                    train_cfg[\'gradient_clip_norm\'], error_if_nonfinite=True)\n                if not math.isfinite(float(grad_norm)):\n                    raise FloatingPointError(\'Nonfinite gradient norm\')\n                if global_step == 0:\n                    ssl_grad = sum(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in ssl)\n                    head_grad = sum(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in head)\n                    assert ssl_grad and head_grad, \'Missing model gradients\'\n                    print(f\'[GRAD] XLS-R={ssl_grad} AASIST={head_grad}\', flush=True)\n                optimizer.step()\n                optimizer.zero_grad(set_to_none=True)\n                global_step += 1\n                if global_step == 1 or global_step % log_interval == 0:\n                    print(f\'[TRAIN] epoch={ep+1}/{args.epochs} \'\n                          f\'batch={cursor}/{len(microbatches)} step={global_step} \'\n                          f\'CE={ce_logged/denom:.5f} \'\n                          f\'PCL={pcl_logged/pcl_slots if pcl_slots else 0.0:.5f} \'\n                          f\'grad_norm={float(grad_norm):.3f} \'\n                          f\'elapsed_min={(time.monotonic()-t0)/60:.1f}\', flush=True)\n                time_exceeded = args.max_minutes > 0 and (time.monotonic()-t0)/60 >= args.max_minutes\n                if global_step % max_save_interval == 0 or time_exceeded:\n                    snapshot(ep, cursor, \'pause\' if time_exceeded else \'periodic\')\n                if time_exceeded:\n                    print(\'[PAUSED] Use --resume in a restored session.\', flush=True)\n                    return\n            snapshot(ep, len(microbatches), \'pre_eval\')\n        else:\n            if saved_rng is not None:\n                restore_rng(saved_rng)\n                saved_rng = None\n            print(f\'[RESUME] epoch={ep+1}: training already completed, evaluating Dev\', flush=True)\n        print(\'[DEV] Evaluating clean full Dev, CE and threshold=0.5\', flush=True)\n        metrics, loss = evaluate_full_dev(model, dev_loader, dev_ctx[\'examples\'],\n                                          device=\'cuda:0\', amp=False,\n                                          class_weights=train_cfg[\'class_weights\'])\n        online_f1 = dev_online_macro_f1(metrics)\n        epoch_metrics = {\n            \'epoch\': ep + 1, \'step\': global_step, \'dev_weighted_loss\': loss,\n            \'dev_online_macro_f1\': online_f1,\n            \'official_hidden_weighted_f1\': None,\n            \'metric_kind\': \'clean_online_dev_only_not_competition_score\',\n            \'subgroups\': metrics,\n            \'training_objective\': \'weighted_CE_plus_lambda_PCL\',\n            \'lambda_pcl\': args.lambda_pcl,\n        }\n        write_json_atomic(epoch_metrics, out/f\'dev_epoch_{ep+1:03d}.json\')\n        write_json_atomic(epoch_metrics, out/\'dev_metrics.json\')\n        print(f\'[DEV] epoch={ep+1} loss={loss:.5f} online_macro_f1={online_f1:.5f} \'\n              f\'all_macro_f1={metrics["all_dev"]["macro_f1"]:.5f}\', flush=True)\n        if loss < best_dev_loss:\n            best_dev_loss = loss\n            no_improve_epochs = 0\n            save_best_model(out/\'best_dev_loss.pt\', model=model, epoch=ep+1,\n                global_step=global_step, dev_loss=loss, online_macro_f1=online_f1,\n                config_hash=config_hash, selection=\'min_dev_weighted_loss\')\n            backup.saved_best(\'best_dev_loss.pt\', epoch=ep+1, global_step=global_step)\n        else:\n            no_improve_epochs += 1\n        if online_f1 > best_online_f1:\n            best_online_f1 = online_f1\n            save_best_model(out/\'best_online_f1.pt\', model=model, epoch=ep+1,\n                global_step=global_step, dev_loss=loss, online_macro_f1=online_f1,\n                config_hash=config_hash, selection=\'max_clean_online_dev_macro_f1\')\n            backup.saved_best(\'best_online_f1.pt\', epoch=ep+1, global_step=global_step)\n        snapshot(ep+1, 0, \'epoch_end\')\n        if no_improve_epochs >= train_cfg[\'early_stopping_patience\']:\n            print(\'[EARLY STOP] No Dev loss improvement\', flush=True)\n            break\n    print(\'[DONE] E02 CE + PCL epochs complete\', flush=True)\n    print(\'[OUTPUT]\', out, flush=True)\n\n\ndef parser():\n    p = argparse.ArgumentParser(description=\'E02 CE + PCL independent trainer\')\n    p.add_argument(\'--repo\', type=Path, required=True)\n    p.add_argument(\'--data\', type=Path, required=True)\n    p.add_argument(\'--cache\', type=Path, required=True)\n    p.add_argument(\'--pretrained\', type=Path, required=True)\n    p.add_argument(\'--config\', type=Path, required=True)\n    p.add_argument(\'--out\', type=Path, required=True)\n    p.add_argument(\'--epochs\', type=int, default=1)\n    p.add_argument(\'--max-minutes\', type=float, default=540)\n    p.add_argument(\'--lambda-pcl\', type=float, default=0.1)\n    p.add_argument(\'--temperature\', type=float, default=0.1)\n    p.add_argument(\'--max-matches\', type=int, default=96)\n    p.add_argument(\'--preflight-only\', action=\'store_true\')\n    p.add_argument(\'--resume\', action=\'store_true\')\n    return p\n\n\nif __name__ == \'__main__\':\n    run(parser().parse_args())\n'
EXPECTED_SHA = hashlib.sha256(SOURCE.encode('utf-8')).hexdigest()

if MODULE.exists():
    current_sha = hashlib.sha256(MODULE.read_bytes()).hexdigest()
    if current_sha != EXPECTED_SHA:
        raise RuntimeError(
            f'Existing E02 trainer differs. No automatic overwrite: {MODULE}\n'
            f'Existing SHA256: {current_sha}\nExpected SHA256: {EXPECTED_SHA}'
        )
    print('[PASS] Identical E02 training module already exists')
else:
    MODULE.parent.mkdir(parents=True, exist_ok=True)
    MODULE.write_text(SOURCE, encoding='utf-8')
    print('[PASS] Wrote NEW independent E02 trainer:', MODULE)

assert hashlib.sha256(MODULE.read_bytes()).hexdigest() == EXPECTED_SHA

# ------------------------------------------------------------
# 2. VALIDATE SOURCE SYNTAX (WITHOUT EXECUTING TRAINING)
# ------------------------------------------------------------
subprocess.run([str(TRAINING_PY), '-m', 'py_compile', str(MODULE)], check=True)
print('[PASS] E02 trainer source syntax')

# ------------------------------------------------------------
# 3. FULL CSV / CACHE HASH-JOIN + SAMPLER COVERAGE TEST
# ------------------------------------------------------------
print('\n[CHECK] All 44,590 cached pairs against original manifests')
command = [
    str(TRAINING_PY), '-u', '-m', 'rtcsdd.training.e02_pcl_trainer',
    '--repo', str(REPO), '--data', str(DATA), '--cache', str(CACHE),
    '--pretrained', str(PRETRAINED), '--config', str(CONFIG),
    '--out', str(OUT), '--epochs', '1', '--max-minutes', '540',
    '--lambda-pcl', '0.1', '--temperature', '0.1', '--max-matches', '96',
    '--preflight-only',
]
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO/'src') + os.pathsep + env.get('PYTHONPATH', '')
result = subprocess.run(
    command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    cwd=REPO, env=env,
)
print(result.stdout, flush=True)
if result.returncode:
    raise RuntimeError(
        f'E02 full cache↔pair preflight FAILED (exit={result.returncode}).\n'
        'No training was started. Keep the error output for diagnosis.'
    )

# ------------------------------------------------------------
# 4. SAVE REPORT
# ------------------------------------------------------------
report = {
    'experiment': 'e02_ce_pcl_fp32',
    'training_started': False,
    'gpu_forward_started': False,
    'module': str(MODULE),
    'module_sha256': EXPECTED_SHA,
    'csv_cache_hash_join_verified': True,
    'sampling_coverage_verified': True,
    'train_complete_pairs': 37125,
    'train_offline_only': 1535,
    'dev_complete_pairs': 7465,
    'dev_offline_only': 274,
    'loss': 'WeightedCE + lambda_PCL',
    'lambda_pcl_proposed': 0.1,
    'initialization': 'original_xlsr_300m_plus_fresh_aasist',
    'resume_format': 'project_checkpoint_format_v2',
    'hf_backup_prefix': 'experiments/e02_ce_pcl_fp32',
    'next_step': 'GPU small optimizer-step smoke and then training launch',
}
REPORT.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print('=' * 75)
print('E02 — TRAINER INTEGRATION PREFLIGHT: PASS')
print('Training started: NO')
print('Trainer file:', MODULE)
print('Report:', REPORT)
print('NEXT: E02 one-optimizer-step GPU validation before full epoch')
print('=' * 75)


# ============================================================================
# E02 — FIRST REAL OPTIMIZER STEP, VERIFIED PAUSE + RESUMABLE HF CHECKPOINT
# KEEP | GPU | EXACTLY ONE REAL STEP | NO RESTART OF E01 OR PREPROCESSING
# ============================================================================

from pathlib import Path
import hashlib
import json
import os
import re
import shutil
import subprocess

ROOT = Path('/kaggle/working')
REPO = ROOT / 'RTC-SDD-Research'
DATA = Path('/kaggle/input/datasets/a7medlabib/rtcfake-packed-private')
CACHE = ROOT / 'e02_pcl_cache/v1'
PRETRAINED = ROOT / 'checkpoints/pretrained/xlsr2_300m.pt'
PY310 = ROOT / 'rtcsdd-baseline-py310/bin/python'
TRAINER = REPO / 'src/rtcsdd/training/e02_pcl_trainer.py'
PCL = REPO / 'src/rtcsdd/training/phoneme_contrastive.py'
BASE_CONFIG = REPO / 'src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json'
OUT = REPO / 'outputs/experiments/e02_ce_pcl_fp32'
PREFLIGHT = ROOT / 'e02_trainer_integration_preflight.json'
RESULT = ROOT / 'e02_first_optimizer_step_report.json'
LOG = ROOT / 'e02_first_optimizer_step.log'

HF_REPO = 'AhmedLabib1/rtc-sdd-research-checkpoints'
HF_PREFIX = 'experiments/e02_ce_pcl_fp32'
EXPECTED_COMMIT = '41a6b3e1d104592bb47732c5bbc82dc9b28dcc63'
EXPECTED_TRAINER_SHA = 'fb2a6df1ff36d8d86a4b5a36508d9568bba0ab96c597a4ea2d7b2c0b378c39eb'
EXPECTED_PCL_SHA = '4cb287428466e0725e195fa7bc30dadd44576eb1e3860dc197b100a34443704b'

GiB = 1024**3


def sha256_file(path):
    d = hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda: f.read(8 * 1024**2), b''):
            d.update(b)
    return d.hexdigest()


def run(command, *, env, capture=False):
    if capture:
        return subprocess.run(command, cwd=REPO, env=env, check=True,
                              text=True, capture_output=True)
    return subprocess.run(command, cwd=REPO, env=env, check=True)


print('=' * 76, flush=True)
print('E02 — ONE REAL OPTIMIZER STEP + VERIFIED RESUME CHECKPOINT', flush=True)
print('=' * 76, flush=True)

print('\n[1/6] IMMUTABLE SOURCE / DATA PREFLIGHT', flush=True)
for name, path in {
    'repository': REPO, 'training Python 3.10': PY310,
    'E02 independent trainer': TRAINER, 'PCL loss': PCL,
    'E01-comparable baseline config': BASE_CONFIG,
    'paired data': DATA, 'verified PCL cache': CACHE,
    'original XLS-R pretrained': PRETRAINED,
    'integration preflight report': PREFLIGHT,
}.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing {name}: {path}')
    print('[PASS]', name, flush=True)

commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert commit == EXPECTED_COMMIT, (
    f'Unexpected repository commit: {commit}. Refusing to train'
)
assert sha256_file(TRAINER) == EXPECTED_TRAINER_SHA, (
    'E02 trainer changed since verified integration cell. Stop and inspect.'
)
assert sha256_file(PCL) == EXPECTED_PCL_SHA, (
    'PCL loss changed since unit tests. Stop and inspect.'
)
preflight = json.loads(PREFLIGHT.read_text(encoding='utf-8'))
assert preflight.get('module_sha256') == EXPECTED_TRAINER_SHA
assert preflight.get('csv_cache_hash_join_verified') is True
assert preflight.get('sampling_coverage_verified') is True
assert preflight.get('training_started') is False
print('[PASS] Pinned Git commit, trainer SHA256, PCL SHA256', flush=True)

base = json.loads(BASE_CONFIG.read_text(encoding='utf-8'))
train = base['training']
assert train['batch_size'] == 2 and train['gradient_accumulation'] == 8
assert train['precision'] == 'fp32'
assert train['class_weights'] == [0.1, 0.9]
assert train['learning_rate'] == 1e-6
assert base['data']['audio_length'] == 64600
assert base['data']['crop'] == 'left' and base['data']['pad'] == 'repeat'
assert base['data']['rawboost']['enabled'] is True
assert base['model']['xlsr_trainable'] and base['model']['aasist_trainable']

# Full precision checkpoint is ~3.54 GiB. Future atomic overwrite needs
# another temporary full-sized checkpoint + enough headroom.
free = shutil.disk_usage(ROOT).free
print(f'[DISK] Free: {free / GiB:.2f} GiB', flush=True)

already_saved = (OUT / 'last.pt').is_file() and (OUT / 'config.json').is_file()
if not already_saved:
    if OUT.exists() and any(OUT.iterdir()):
        raise RuntimeError(
            'E02 output contains files but no complete config.json+last.pt. '
            'Inspect partial output; never overwrite a possibly interrupted run.'
        )
    if free < 8.2 * GiB:
        raise RuntimeError(
            'Insufficient disk for full FP32 checkpoint plus future atomic '
            f'replacement (free={free/GiB:.2f} GiB; need >=8.2 GiB). '
            'Do not delete the PCL cache or original pretrained checkpoint.'
        )
else:
    print('[SAFE RE-RUN] Existing E02 resume checkpoint detected: '
          'will verify it WITHOUT performing another optimizer step.', flush=True)

print('\n[2/6] GPU AND HUGGING FACE AUTHORIZATION', flush=True)
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + env.get('PYTHONPATH', '')
probe = (
    'import sys,torch; '
    'assert sys.version_info[:2] == (3,10); '
    'assert torch.cuda.is_available(); '
    'assert torch.__version__ == "2.5.1+cu121"; '
    'print("[PASS]",torch.__version__,torch.cuda.get_device_name(0))'
)
run([str(PY310), '-c', probe], env=env)

from huggingface_hub import HfApi, hf_hub_download

token = os.environ.get('HF_TOKEN')
if not token:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
assert token, 'HF_TOKEN unavailable: enable Kaggle Secret before training'
env['HF_TOKEN'] = token
api = HfApi(token=token)
repo_info = api.repo_info(repo_id=HF_REPO, repo_type='model', token=token)
assert repo_info.private is True, 'HF checkpoint repository must be private'
print('[PASS] HF private checkpoint destination accessible (token hidden)', flush=True)

# The trainer itself rechecks EVERY cached pair and CSV SHA256 before loading GPU.
print('\n[3/6] START FIRST REAL OPTIMIZER STEP', flush=True)
print('[PLAN] 8 pair-preserving microbatches x 2 audio = effective batch 16', flush=True)
print('[PLAN] Weighted CE + 0.1 * phoneme PCL (experimental hyperparameter)', flush=True)
print('[PLAN] Trainer pauses ONLY after its first completed optimizer update', flush=True)
print('[PLAN] last.pt + HF backup; next run uses --resume', flush=True)

if not already_saved:
    command = [
        str(PY310), '-u', '-m', 'rtcsdd.training.e02_pcl_trainer',
        '--repo', str(REPO), '--data', str(DATA),
        '--cache', str(CACHE), '--pretrained', str(PRETRAINED),
        '--config', str(BASE_CONFIG), '--out', str(OUT),
        '--epochs', '1',
        # Trainer checks the time limit ONLY AFTER optimizer.step().
        # 0.001 minutes (=0.06s) therefore stops after exactly the first step.
        '--max-minutes', '0.001',
        '--lambda-pcl', '0.1',
        '--temperature', '0.1',
        '--max-matches', '96',
    ]
    with LOG.open('w', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            command, cwd=REPO, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1,
        )
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
                log_file.write(line)
                log_file.flush()
        finally:
            if process.stdout is not None:
                process.stdout.close()
        return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f'E02 trainer exited with code {return_code}. '
            f'Inspect {LOG}. Do NOT restart fresh automatically.'
        )
else:
    print('[SKIP] Existing E02 checkpoint: no repeated training', flush=True)

print('\n[4/6] VERIFIED CHECKPOINT CONTENTS', flush=True)
last = OUT / 'last.pt'
config_path = OUT / 'config.json'
assert last.is_file() and config_path.is_file(), (
    'Missing checkpoint or experiment manifest after first step'
)
assert last.stat().st_size > 3 * GiB, 'Checkpoint smaller than expected'

# mmap inspects checkpoint metadata without bringing full model+Adam state
# into the notebook kernel. Load only this trusted locally created checkpoint.
verify_python = r'''
import json,sys,torch
from pathlib import Path
p=Path(sys.argv[1]); config_path=Path(sys.argv[2]); out=Path(sys.argv[3])
ckpt=torch.load(p,map_location='cpu',weights_only=False,mmap=True)
manifest=json.loads(config_path.read_text(encoding='utf-8'))
assert ckpt['format_version']==2
assert ckpt['global_step']==1, f"Expected ONE optimizer step, got {ckpt['global_step']}"
assert ckpt['epoch']==0, f"Expected epoch index zero, got {ckpt['epoch']}"
assert ckpt['next_batch']==8, f"Expected eight microbatches, got {ckpt['next_batch']}"
assert ckpt['accumulated_samples']==0
assert ckpt['config_fingerprint']==manifest['config_fingerprint']
assert bool(ckpt['model']) and bool(ckpt['optimizer'])
assert manifest['objective']=='weighted_ce_plus_lambda_symmetric_phoneme_infonce'
result={'global_step':ckpt['global_step'],'epoch_index':ckpt['epoch'],
        'next_batch':ckpt['next_batch'],'config_fingerprint':ckpt['config_fingerprint'],
        'optimizer_state_present':bool(ckpt['optimizer']),
        'rng_state_present':bool(ckpt['rng'])}
out.write_text(json.dumps(result,indent=2)+'\n',encoding='utf-8')
print('[PASS] checkpoint: step=1 epoch=1 next_batch=8 optimizer+RNG restored')
'''
checkpoint_meta = ROOT / 'e02_first_step_checkpoint_metadata.json'
run([str(PY310), '-c', verify_python, str(last), str(config_path),
     str(checkpoint_meta)], env=env)

if LOG.is_file():
    full_log = LOG.read_text(encoding='utf-8')
    if '[TRAIN]' not in full_log or 'step=1' not in full_log or '[PAUSED]' not in full_log:
        raise RuntimeError('First step/pause missing from trainer log. Inspect before resuming.')
    matched = re.search(r'\[TRAIN\].*?step=1\b.*?PCL=([0-9.eE+-]+)', full_log)
    if matched is None or float(matched.group(1)) <= 0:
        raise RuntimeError('First training step did not log positive PCL. Inspect loss integration.')
    print('[PASS] First optimizer step included positive PCL:',
          matched.group(1), flush=True)

print('\n[5/6] VERIFY REMOTE BACKUP RECEIPT', flush=True)
remote_checkpoint = f'{HF_PREFIX}/last.pt'
remote_receipt = f'{remote_checkpoint}.sha256.json'
remote_confirmed = False
note = ''
try:
    remote_files = set(api.list_repo_files(HF_REPO, repo_type='model', token=token))
    assert remote_checkpoint in remote_files and remote_receipt in remote_files
    receipt_path = Path(hf_hub_download(
        repo_id=HF_REPO, repo_type='model', filename=remote_receipt,
        token=token, force_download=True
    ))
    receipt = json.loads(receipt_path.read_text(encoding='utf-8'))
    assert receipt.get('size_bytes') == last.stat().st_size
    local_sha = sha256_file(last)
    assert receipt.get('sha256') == local_sha
    remote_confirmed = True
    print('[PASS] HF last.pt and SHA256 receipt match local step-1 checkpoint', flush=True)
except Exception as exc:
    note = f'{type(exc).__name__}: {exc}'
    print('[WARNING] Cannot confirm HF backup:', note, flush=True)
    print('[WARNING] Local last.pt remains intact. Do not close Kaggle yet.', flush=True)

print('\n[6/6] SAVE FIRST-STEP VERIFICATION REPORT', flush=True)
meta = json.loads(checkpoint_meta.read_text(encoding='utf-8'))
result = {
    'experiment': 'e02_ce_pcl_fp32',
    'stage': 'first_real_optimizer_step',
    'status': 'PASS' if remote_confirmed else 'LOCAL_PASS_REMOTE_UNCONFIRMED',
    'optimizer_step': meta['global_step'],
    'next_batch': meta['next_batch'],
    'trainer_sha256': EXPECTED_TRAINER_SHA,
    'pcl_module_sha256': EXPECTED_PCL_SHA,
    'config_fingerprint': meta['config_fingerprint'],
    'checkpoint': str(last),
    'checkpoint_size_bytes': last.stat().st_size,
    'checkpoint_remote_path': remote_checkpoint,
    'hf_backup_verified': remote_confirmed,
    'hf_backup_warning': note or None,
    'log': str(LOG),
    'full_epoch_completed': False,
    'next_action': 'E02 resume from step 1 (batch 8) to complete epoch 1',
}
RESULT.write_text(json.dumps(result, indent=2) + '\n', encoding='utf-8')

print('=' * 76, flush=True)
print('E02 — FIRST REAL OPTIMIZER STEP SUMMARY', flush=True)
print('Optimizer steps completed: 1', flush=True)
print('Next microbatch cursor: 8', flush=True)
print('Original XLS-R used: YES', flush=True)
print('Model + Adam + RNG saved locally: YES', flush=True)
print('HF backup verified:', 'YES' if remote_confirmed else 'NOT YET', flush=True)
print('Full epoch completed: NO', flush=True)
print('Next: continue E02 Epoch 1 with --resume', flush=True)
print('Report:', RESULT, flush=True)
print('=' * 76, flush=True)


In [ ]:
# ============================================================================
# E02 — EXPORT PORTABLE HANDOFF FROM CURRENT KAGGLE SESSION
# KEEP | GPU MAY REMAIN ON | NO TRAINING OR CHECKPOINT CHANGES
# Saves small untracked source, verified PCL cache and config, Git snapshot.
# The 3.8-GiB resumable checkpoint is already backed up to private HF.
# ============================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import shutil
import subprocess
import tempfile
import zipfile

ROOT = Path('/kaggle/working')
REPO = ROOT / 'RTC-SDD-Research'
CACHE = ROOT / 'e02_pcl_cache/v1'
OUT = REPO / 'outputs/experiments/e02_ce_pcl_fp32'
TARGET = ROOT / 'e02_step1_portable_handoff.zip'
EXPECTED_COMMIT = '41a6b3e1d104592bb47732c5bbc82dc9b28dcc63'
EXPECTED_TRAINER = 'fb2a6df1ff36d8d86a4b5a36508d9568bba0ab96c597a4ea2d7b2c0b378c39eb'
EXPECTED_PCL = '4cb287428466e0725e195fa7bc30dadd44576eb1e3860dc197b100a34443704b'
EXPECTED_FINGERPRINT = '7cd54f717b464d855f5d7dc849afcf1c9acd1bef9a2a72eb38e1c43434a4b854'
TRAINER = REPO / 'src/rtcsdd/training/e02_pcl_trainer.py'
PCL = REPO / 'src/rtcsdd/training/phoneme_contrastive.py'
LAST = OUT / 'last.pt'
PRETRAINED = ROOT / 'checkpoints/pretrained/xlsr2_300m.pt'
CONFIG = OUT / 'config.json'
GIB = 1024**3

def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            h.update(block)
    return h.hexdigest()

print('='*74)
print('E02 — PORTABLE ONE-SHOT HANDOFF EXPORT')
print('='*74)
for name,p in {'repository':REPO,'PCL cache':CACHE,'trainer':TRAINER,
               'PCL module':PCL,'checkpoint':LAST,'original XLS-R':PRETRAINED,'config.json':CONFIG}.items():
    if not p.exists():
        raise FileNotFoundError(f'Missing {name}: {p}')
    print('[PASS]',name)

commit = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
assert commit == EXPECTED_COMMIT, ('Repository HEAD changed; refuse to export a mismatched snapshot',commit)
assert digest(TRAINER) == EXPECTED_TRAINER
assert digest(PCL) == EXPECTED_PCL
config = json.loads(CONFIG.read_text(encoding='utf-8'))
assert config['config_fingerprint'] == EXPECTED_FINGERPRINT
assert config['lambda_pcl'] == 0.1 and config['temperature'] == 0.1
assert config['max_matches'] == 96
assert LAST.stat().st_size > 3 * GIB, 'Resumable checkpoint unexpectedly small'

cache_files = []
for split, required in [('train',146),('dev',30)]:
    shard_files = sorted((CACHE/split).glob('*.json.gz'))
    assert len(shard_files) == required, (split,len(shard_files))
    for shard in shard_files:
        sidecar = Path(str(shard)+'.sha256')
        if not sidecar.is_file():
            raise FileNotFoundError(sidecar)
        claimed = sidecar.read_text(encoding='utf-8').strip().split()[0]
        assert digest(shard) == claimed, f'Cache corruption: {shard.name}'
        cache_files.extend([shard,sidecar])
print('[PASS] All 176 shards + SHA256 sidecars verified')
print('[CHECK] Hashing step-1 checkpoint (checkpoint itself will NOT be zipped)')
checkpoint_sha = digest(LAST)
pretrained_sha = digest(PRETRAINED)
print('[PASS] Original last.pt SHA256:',checkpoint_sha)
print('[PASS] Original XLS-R SHA256:',pretrained_sha)

additional = [ROOT/'e02_first_optimizer_step_report.json',
              ROOT/'e02_trainer_integration_preflight.json',
              ROOT/'e02_real_model_pcl_smoke_report.json',
              ROOT/'checkpoints/pretrained/xlsr2_300m.pt.download.json']
additional = [p for p in additional if p.is_file()]

with tempfile.TemporaryDirectory(prefix='e02_handoff_') as td:
    bundle = Path(td)/'e02_repository.bundle'
    subprocess.run(['git','-C',str(REPO),'bundle','create',str(bundle),'--all'],check=True)
    subprocess.run(['git','-C',str(REPO),'bundle','verify',str(bundle)],check=True)

    files = {
        'repo/e02_repository.bundle':bundle,
        'source/e02_pcl_trainer.py':TRAINER,
        'source/phoneme_contrastive.py':PCL,
        'experiment/config.json':CONFIG,
    }
    for p in cache_files:
        files['cache/'+str(p.relative_to(CACHE)).replace('\\','/')] = p
    for p in additional:
        files['reports/'+p.name] = p

    metadata = {
        'format':'e02_step1_portable_handoff_v1',
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'git_commit':EXPECTED_COMMIT,
        'git_branch':'experiment/e02-pcl',
        'source_sha256':{'trainer':EXPECTED_TRAINER,'pcl':EXPECTED_PCL},
        'config_fingerprint':EXPECTED_FINGERPRINT,
        'pretrained':{'sha256':pretrained_sha,'size_bytes':PRETRAINED.stat().st_size},
        'checkpoint':{
            'sha256':checkpoint_sha,'size_bytes':LAST.stat().st_size,
            'remote_repo':'AhmedLabib1/rtc-sdd-research-checkpoints',
            'remote_path':'experiments/e02_ce_pcl_fp32/last.pt',
            'included_in_zip':False,
        },
        'cache_shards':{'train':146,'dev':30},
        'file_hashes':{arc:digest(path) for arc,path in files.items()},
    }
    provisional = Path(str(TARGET)+'.partial')
    if provisional.exists():
        provisional.unlink()
    try:
        with zipfile.ZipFile(provisional,'w',compression=zipfile.ZIP_STORED,
                             allowZip64=True) as z:
            for arc,p in sorted(files.items()):
                z.write(p,arcname=arc)
            z.writestr('handoff_manifest.json',json.dumps(metadata,indent=2,sort_keys=True)+'\n')
        with zipfile.ZipFile(provisional) as z:
            assert z.testzip() is None, 'ZIP CRC validation failed'
            written = set(z.namelist())
            assert written == set(files)|{'handoff_manifest.json'}
        os.replace(provisional,TARGET)
    finally:
        if provisional.exists(): provisional.unlink()

print('\n[PASS] Created:',TARGET)
print(f'[SIZE] {TARGET.stat().st_size/1024**2:.1f} MiB')
print('[SHA256]',digest(TARGET))
print('[INCLUDES] Git snapshot, exact Trainer/PCL sources, all cache shards + sidecars,')
print('           config.json, small setup reports, expected step-1 checkpoint SHA.')
print('[EXCLUDES] 3.8-GiB last.pt, 1.2-GiB original XLS-R, environment.')
print('[IMPORTANT] Save this ZIP locally; upload it as a PRIVATE Kaggle Dataset')
print('            and attach that Dataset as an input to the new notebook.')
print('            Also attach the original RTCFake Packed Private Dataset.')
print('            Keep HF_TOKEN enabled for restoring last.pt.')
print('='*74)


---

## Phase B — Fresh Kaggle GPU session: restore handoff and finish Epoch 1

**Separate historical runtime.** This phase began from the private handoff
containing the pinned source, cache, config and step-1 checkpoint. All
executed code below is preserved from the original one-shot notebook without
edits, but previous logs/outputs are removed to keep the GitHub artifact small.

**Do not run this phase on the already-completed E02 experiment.**


# RTC-SDD Research — E02 CE + PCL | Epoch 1 One-Shot Resume

**Historical run condition:** this phase was executed in a FRESH Kaggle GPU session (T4 x2 with Internet). **Do not Run All in this combined archive.** The GPU remains enabled for the entire session; setup and disk I/O are CPU operations that do not require disabling the GPU. The *original single-GPU trainer intentionally uses `cuda:0` only*, not both T4 cards.

**Required Kaggle inputs (private):**
1. Original **RTCFake Packed Private** dataset (Train/Dev `.pack`, label TXT and paired CSV files).
2. A private Kaggle Dataset containing **`e02_step1_portable_handoff.zip` or its auto-extracted `cache/`, `repo/`, `source/`, `experiment/` folders with `handoff_manifest.json`**, exported by `e02_export_current_session_bundle.py` in the old session.

**Required secret:** Kaggle Secret `HF_TOKEN` with read/write access to `AhmedLabib1/rtc-sdd-research-checkpoints`. Enable Notebook access to the secret. The original `last.pt` (step 1, Adam+RNG) is pulled from this HF model repository, **not from E01**. You do **not** need to upload the 3.8-GiB checkpoint or rebuild phoneme alignments.

**Workflow:** GPU preflight → restore pinned Git commit + 2 untracked modules + complete PCL cache + config → bootstrap official source and isolated Python 3.10 CUDA/Fairseq → download *original* XLS-R and verify old SHA → retrieve exact E02 step-1 checkpoint from private HF and verify SHA → strict 44,590-pair contract → **single invocation of E02 Epoch-1 CE + 0.1×PCL training** → full clean Dev evaluation and HF checkpoint verification.

**Important Kaggle limit:** This notebook **attempts to finish Epoch 1 in one Run All**, but a 12-hour Kaggle hard session cutoff is outside the notebook's control. To avoid losing hours of work, the unchanged trainer periodically saves and backs up `last.pt`; a late safety stop at an optimizer boundary is enabled if needed. Do not interpret a safe pause as epoch completion.

The cells below are retained unchanged as a historical research record; **do not rerun them on completed E02**.


### 01 — GPU always on · auto-detect ZIP or extracted handoff
**KEEP · GPU enabled throughout · No training yet.** Kaggle may unpack the uploaded ZIP automatically. Both formats are supported.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, json, hashlib, time
import torch
RUN_STARTED_MONOTONIC=time.monotonic()

ROOT = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
REPO = ROOT / 'RTC-SDD-Research'
CACHE = ROOT / 'e02_pcl_cache/v1'
OUT = REPO / 'outputs/experiments/e02_ce_pcl_fp32'
PRETRAINED = ROOT / 'checkpoints/pretrained/xlsr2_300m.pt'
PY310 = ROOT / 'rtcsdd-baseline-py310/bin/python'
BUNDLE_NAME = 'e02_step1_portable_handoff.zip'
HF_REPO = 'AhmedLabib1/rtc-sdd-research-checkpoints'
HF_PREFIX = 'experiments/e02_ce_pcl_fp32'
EXPECTED_COMMIT = '41a6b3e1d104592bb47732c5bbc82dc9b28dcc63'
EXPECTED_TRAINER = 'fb2a6df1ff36d8d86a4b5a36508d9568bba0ab96c597a4ea2d7b2c0b378c39eb'
EXPECTED_PCL = '4cb287428466e0725e195fa7bc30dadd44576eb1e3860dc197b100a34443704b'
EXPECTED_FINGERPRINT = '7cd54f717b464d855f5d7dc849afcf1c9acd1bef9a2a72eb38e1c43434a4b854'

assert torch.cuda.is_available(), 'STOP: enable Kaggle GPU accelerator before Run All'
assert torch.cuda.device_count() >= 1
print('GPU ENABLED FOR ENTIRE SESSION:', torch.cuda.get_device_name(0),flush=True)
subprocess.run(['nvidia-smi','-L'],check=True)

def sha256_file(path):
    sha=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(8*1024*1024),b''):
            sha.update(chunk)
    return sha.hexdigest()


# Kaggle often auto-extracts uploaded ZIPs into folders when making a dataset.
# Accept exactly one intact handoff in EITHER form (zip file OR extracted tree).
zip_candidates = sorted(p for p in INPUT.rglob(BUNDLE_NAME) if p.is_file())
if (ROOT / BUNDLE_NAME).is_file():
    zip_candidates.append(ROOT / BUNDLE_NAME)
extracted_candidates = sorted({
    p.parent for p in INPUT.rglob('handoff_manifest.json')
    if p.is_file()
    and (p.parent/'repo/e02_repository.bundle').is_file()
    and (p.parent/'source/e02_pcl_trainer.py').is_file()
    and (p.parent/'source/phoneme_contrastive.py').is_file()
    and (p.parent/'experiment/config.json').is_file()
})

# If both paths exist, keep the ZIP path (still SHA-verified in next cell).
if zip_candidates:
    assert len(zip_candidates)==1, ('Ambiguous handoff ZIPs', list(map(str,zip_candidates)))
    HANDOFF_ZIP=zip_candidates[0]
    HANDOFF_DIR=None
    print('[INPUT] Handoff mode: ZIP',HANDOFF_ZIP,flush=True)
else:
    assert len(extracted_candidates)==1, (
        'Need ONE handoff input with handoff_manifest.json, repo/, source/, cache/, experiment/',
        [str(p) for p in extracted_candidates]
    )
    HANDOFF_ZIP=None
    HANDOFF_DIR=extracted_candidates[0]
    print('[INPUT] Handoff mode: Kaggle auto-extracted folder',HANDOFF_DIR,flush=True)

required_data_files = (
 'train_label.txt','dev_label.txt','train_offline.pack','train_online.pack',
 'dev_offline.pack','dev_online.pack',
 'train_offline_online_pairs.csv','dev_offline_online_pairs.csv'
)
def valid_data(p):return all((p/name).is_file() for name in required_data_files)
known=Path('/kaggle/input/datasets/a7medlabib/rtcfake-packed-private')
if valid_data(known):
    DATA=known
else:
    matches=sorted({p.parent for p in INPUT.rglob('train_label.txt') if valid_data(p.parent)})
    assert len(matches)==1, ('Expected exactly one RTCFake dataset',list(map(str,matches)))
    DATA=matches[0]

print('[PASS] Portable handoff:',HANDOFF_ZIP or HANDOFF_DIR,flush=True)
print('[PASS] Original RTCFake input:',DATA,flush=True)
print('[PASS] GPU always enabled; later training runs on cuda:0',flush=True)


### 02 — Restore pinned Git commit, E02 source, cache and manifest
**KEEP · GPU stays enabled · I/O only.** Detects ZIP or already extracted Kaggle dataset. SHA-256 checks every handoff file and restores source without overwriting changed files.


In [ ]:
# ============================================================
# E02 — RESTORE VERIFIED HANDOFF + REPAIR MISSING CACHE FROM HF
# KEEP | GPU REMAINS ENABLED | NO TRAINING
# ============================================================
from pathlib import Path, PurePosixPath
from contextlib import ExitStack
from concurrent.futures import ThreadPoolExecutor, as_completed
import zipfile, tempfile, json, os, subprocess, shutil, hashlib, sys

if REPO.exists():
    assert (REPO/'.git').is_dir(), 'Existing repo path is not a Git repository'
    got=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    assert got==EXPECTED_COMMIT, ('Unexpected existing repository commit',got)

HANDOFF_META_PATH=ROOT/'e02_handoff_manifest.json'

with ExitStack() as stack:
    if HANDOFF_ZIP is not None:
        z=stack.enter_context(zipfile.ZipFile(HANDOFF_ZIP,'r'))
        names=set(z.namelist())
        assert 'handoff_manifest.json' in names, 'ZIP missing handoff manifest'
        assert z.testzip() is None, 'ZIP CRC failed'
        meta=json.loads(z.read('handoff_manifest.json'))
        stage=Path(stack.enter_context(tempfile.TemporaryDirectory(prefix='e02_portable_stage_')))
    else:
        assert HANDOFF_DIR is not None and HANDOFF_DIR.is_dir()
        stage=HANDOFF_DIR   # Kaggle auto-extracted input: STRICTLY READ ONLY
        meta=json.loads((stage/'handoff_manifest.json').read_text(encoding='utf-8'))
        names=None

    assert meta['format']=='e02_step1_portable_handoff_v1'
    assert meta['git_commit']==EXPECTED_COMMIT
    assert meta['source_sha256']=={'trainer':EXPECTED_TRAINER,'pcl':EXPECTED_PCL}
    assert meta['config_fingerprint']==EXPECTED_FINGERPRINT
    assert meta['cache_shards']=={'train':146,'dev':30}
    assert meta['checkpoint']['remote_repo']==HF_REPO
    assert meta['checkpoint']['remote_path']==HF_PREFIX+'/last.pt'
    if names is not None:
        assert set(meta['file_hashes'])|{'handoff_manifest.json'}==names, 'Unexpected ZIP members'

    all_entries=sorted(meta['file_hashes'].items())
    sources={}
    missing_cache=[]
    for arc, expected in all_entries:
        p=PurePosixPath(arc)
        assert (not p.is_absolute() and '..' not in p.parts
                and len(p.parts)>1 and '\\' not in arc), ('Unsafe handoff path',arc)
        target=stage.joinpath(*p.parts)
        assert target.resolve().is_relative_to(stage.resolve()), ('Handoff path escapes root',arc)
        if HANDOFF_ZIP is not None:
            target.parent.mkdir(parents=True,exist_ok=True)
            digest=hashlib.sha256()
            with z.open(arc) as src, target.open('wb') as dst:
                for part in iter(lambda:src.read(4*1024*1024),b''):
                    digest.update(part)
                    dst.write(part)
            assert digest.hexdigest()==expected, ('ZIP member SHA256 mismatch',arc)
            sources[arc]=target
        elif target.is_file():
            assert sha256_file(target)==expected, ('Extracted handoff SHA256 mismatch',arc)
            sources[arc]=target
        elif arc.startswith('cache/') and (
            arc.endswith('.json.gz') or arc.endswith('.json.gz.sha256')
        ) and len(p.parts)==3 and p.parts[1] in ('train','dev'):
            # The original Kaggle dataset conversion may unpack/drop nested GZIPs.
            # A locally recovered exact copy is allowed; NEVER substitute a different pair.
            local=CACHE.joinpath(*p.parts[1:])
            if local.is_file():
                assert sha256_file(local)==expected, ('Local cache differs from handoff',arc)
                sources[arc]=local
            else:
                # If the previous run downloaded the blob successfully but failed
                # during Git restore, reuse that same locally HF-cached blob.
                hf_cache_root=(ROOT/'e02_handoff_hf_download_cache'
                               /'models--AhmedLabib1--rtc-sdd-research-checkpoints'
                               /'snapshots')
                reused=None
                if hf_cache_root.is_dir():
                    sub='preprocessing/e02_pcl_cache/v1/'+arc[len('cache/'):]
                    for snapshot in hf_cache_root.iterdir():
                        candidate=snapshot/sub
                        if candidate.is_file() and sha256_file(candidate)==expected:
                            reused=candidate
                            break
                if reused is not None:
                    sources[arc]=reused
                else:
                    missing_cache.append((arc,expected))
        else:
            raise FileNotFoundError(
                f'Handoff missing NON-CACHE file: {arc}. '
                'Re-attach the original private handoff dataset.'
            )

    print('[VERIFY] Handoff files:',len(all_entries),flush=True)
    print('[VERIFY] Matching local/ZIP files:',len(sources),flush=True)
    print('[VERIFY] Missing cache artifacts:',len(missing_cache),flush=True)

    if missing_cache:
        # Use the existing private HF cache produced during the ORIGINAL six-hour
        # preprocessing. No phoneme recognition or reprocessing is performed.
        try:
            from huggingface_hub import HfApi, hf_hub_download
        except ImportError:
            subprocess.run([sys.executable,'-m','pip','install','-q',
                            'huggingface_hub>=0.30,<1.0'],check=True)
            from huggingface_hub import HfApi, hf_hub_download
        token=os.environ.get('HF_TOKEN','').strip()
        if not token:
            from kaggle_secrets import UserSecretsClient
            token=(UserSecretsClient().get_secret('HF_TOKEN') or '').strip()
        assert token, 'Kaggle secret HF_TOKEN required to restore missing private cache files'
        api=HfApi(token=token)
        info=api.repo_info(repo_id=HF_REPO,repo_type='model',token=token)
        assert info.private is True, 'Expected a private checkpoint repository'
        remote_files=set(api.list_repo_files(repo_id=HF_REPO,repo_type='model',token=token))
        prefix='preprocessing/e02_pcl_cache/v1/'
        missing_remote=[arc for arc,_ in missing_cache if prefix+arc[len('cache/'):] not in remote_files]
        assert not missing_remote, (
            'Some missing Kaggle cache files also lack HF backups',missing_remote[:10],
            'Total missing remote',len(missing_remote)
        )
        print('[HF] Recovering',len(missing_cache),
              'missing cache files from PRIVATE HF. No preprocessing.',flush=True)
        download_cache=ROOT/'e02_handoff_hf_download_cache'
        def fetch_one(item):
            arc,expected=item
            remote=prefix+arc[len('cache/'):]
            path=Path(hf_hub_download(
                repo_id=HF_REPO,repo_type='model',filename=remote,
                token=token,cache_dir=str(download_cache)
            ))
            actual=sha256_file(path)
            if actual!=expected:
                raise RuntimeError(
                    f'HF cache SHA256 differs from the original handoff: {arc}; '
                    'STOP: do not train on a mismatched alignment cache.'
                )
            return arc,path
        with ThreadPoolExecutor(max_workers=6) as pool:
            futures={pool.submit(fetch_one,item):item[0] for item in missing_cache}
            n=0
            for future in as_completed(futures):
                arc,path=future.result()
                sources[arc]=path
                n+=1
                if n%25==0 or n==len(missing_cache):
                    print(f'[HF] {n}/{len(missing_cache)} recovered + SHA256 verified',flush=True)

    assert len(sources)==len(all_entries), 'Incomplete handoff source map'
    print('[PASS] EVERY original handoff file SHA256 verified;',
          'HF restores=',len(missing_cache),flush=True)

    # Recover the exact tracked source from the verified portable Git bundle.
    # A git bundle made in a shallow Kaggle clone may have the tip commit,
    # its complete tree/blobs, but omit the older parent commits.  A normal
    # `git clone bundle` wrongly expects full history and fails.  Recover
    # the contained objects into a FRESH shallow repo, then validate the
    # commit+tree+blobs and the resulting clean working tree.
    # No GitHub / GITHUB_TOKEN access is required.
    bundle=sources['repo/e02_repository.bundle']
    ORIGIN_URL='https://github.com/AhmedLabib1/RTC-SDD-Research.git'

    def git_run(args, *, timeout=900, cwd=None, stdin=None):
        return subprocess.run(
            ['git', *map(str,args)],
            cwd=cwd, stdin=stdin, check=False,
            stdout=subprocess.PIPE, stderr=subprocess.PIPE,
            timeout=timeout,
        )

    def git_text(result):
        return result.stdout.decode('utf-8',errors='replace').strip()

    def git_error(result):
        return result.stderr.decode('utf-8',errors='replace').strip()[-1300:]

    def git_ok(args, **kwargs):
        result=git_run(args, **kwargs)
        if result.returncode:
            raise RuntimeError(
                f'Git command failed: {args}\n{git_error(result)}'
            )
        return git_text(result)

    required=(
        'scripts/setup/bootstrap_official.py',
        'scripts/setup/install_training_env.py',
        'src/rtcsdd/data/paired_batches.py',
        'src/rtcsdd/data/phoneme_cache.py',
        'src/rtcsdd/training/trainer.py',
        'src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json',
    )

    def ensure_exact(repo_path):
        head=git_run(['-C',repo_path,'rev-parse','HEAD'])
        if head.returncode or git_text(head)!=EXPECTED_COMMIT:
            return False
        typed=git_run(['-C',repo_path,'cat-file','-t',EXPECTED_COMMIT])
        if typed.returncode or git_text(typed)!='commit':
            return False
        return all((Path(repo_path)/name).is_file() for name in required)

    if REPO.exists():
        assert (REPO/'.git').is_dir() and ensure_exact(REPO), (
            'Existing checkout is incomplete or not the original E02 commit. '
            'Refusing to overwrite anything: '+str(REPO)
        )
        print('[PASS] Existing exact E02 repository checkout',flush=True)
    else:
        with tempfile.TemporaryDirectory(prefix='e02_offline_git_',dir=str(ROOT)) as git_stage:
            temp=Path(git_stage)
            candidate=temp/'reconstructed'
            print('[GIT] Restoring exact E02 source OFFLINE from portable bundle',flush=True)
            heads=git_ok(['bundle','list-heads',bundle]).splitlines()
            matching=[line for line in heads if line.split() and line.split()[0]==EXPECTED_COMMIT]
            assert matching, (
                'Portable bundle lacks the EXPECTED_COMMIT reference. '
                'Do not use a different revision.'
            )
            git_ok(['init','-q',candidate])
            git_ok(['-C',candidate,'symbolic-ref','HEAD',
                    'refs/heads/experiment/e02-pcl'])
            git_ok(['-C',candidate,'config','core.autocrlf','false'])

            # Read bundle header until blank line; the binary PACK follows.
            # Stream it to git unpack-objects with normal STRICT error handling.
            with Path(bundle).open('rb', buffering=0) as f:
                signature=f.readline()
                assert signature.startswith((b'# v2 git bundle',b'# v3 git bundle')), (
                    'Unsupported Git bundle header'
                )
                for header_line in iter(f.readline,b''):
                    if header_line in (b'\n',b'\r\n'):
                        break
                else:
                    raise ValueError('Git bundle header terminator missing')
                assert f.read(4)==b'PACK', 'Git bundle missing PACK data'
                f.seek(-4,1)
                result=git_run(['-C',candidate,'unpack-objects','-q'],stdin=f)
                if result.returncode:
                    raise RuntimeError(
                        'The E02 bundle does not contain a complete, '
                        'unpackable object pack. Exact offline restore is '
                        'not possible. Use a complete handoff export or '
                        'authorized GitHub access.\n'+git_error(result)
                    )

            assert git_ok(['-C',candidate,'cat-file','-t',EXPECTED_COMMIT])=='commit', (
                'Expected pinned E02 commit object is absent from bundle'
            )
            commit_tree=git_ok(['-C',candidate,'rev-parse',EXPECTED_COMMIT+'^{tree}'])
            assert git_ok(['-C',candidate,'cat-file','-t',commit_tree])=='tree'

            # Check the entire tracked tree, not just handpicked source files.
            ls=git_run(['-C',candidate,'ls-tree','-r','-z',EXPECTED_COMMIT])
            assert ls.returncode==0, 'Git commit tree is incomplete: '+git_error(ls)
            entries=[entry for entry in ls.stdout.split(b'\0') if entry]
            assert len(entries)>0, 'The pinned commit contains no tracked files'
            blob_ids=set()
            for entry in entries:
                meta_line,relative=entry.split(b'\t',1)
                mode,kind,obj_id=meta_line.split()
                if kind!=b'blob':
                    # Gitlink entries require their own separate checkout;
                    # official RTC-SDD is bootstrapped in Cell 03.
                    assert kind==b'commit' and mode==b'160000', (
                        'Unexpected tracked Git object kind',relative
                    )
                else:
                    blob_ids.add(obj_id.decode('ascii'))
            for obj in sorted(blob_ids):
                check=git_run(['-C',candidate,'cat-file','-e',obj+'^{blob}'])
                if check.returncode:
                    raise RuntimeError(
                        'Portable bundle lacks a blob required by the pinned '
                        f'E02 source tree: {obj}. Cannot safely restore.'
                    )
            print(f'[PASS] Original Git commit/tree; {len(entries)} tracked entries '
                  f'and {len(blob_ids)} unique blobs verified',flush=True)

            # Mark the EXACT tip shallow: its parent objects are unnecessary to
            # extract and execute the intact pinned source snapshot. This makes
            # `git fsck` respect the missing history boundary without modifying
            # the cryptographic content or SHA of the original E02 commit.
            (candidate/'.git/shallow').write_text(
                EXPECTED_COMMIT+'\n',encoding='ascii'
            )
            git_ok(['-C',candidate,'read-tree',EXPECTED_COMMIT+'^{tree}'])
            git_ok(['-C',candidate,'checkout-index','-a','-f'])
            git_ok(['-C',candidate,'update-ref',
                    'refs/heads/experiment/e02-pcl',EXPECTED_COMMIT])

            fsck=git_run(['-C',candidate,'fsck','--full','--connectivity-only',
                          '--no-reflogs'])
            assert fsck.returncode==0, (
                'Git connectivity check failed even with correct shallow boundary',
                git_error(fsck)
            )
            status=git_ok(['-C',candidate,'status','--porcelain'])
            assert not status, ('Restored tracked files do not match the pinned commit',status)
            assert ensure_exact(candidate), (
                'The bundle lacks required pinned E02 training source files. '
                'Offline checkout must not be used.'
            )
            git_ok(['-C',candidate,'remote','add','origin',ORIGIN_URL])
            # Publish only once all checks passed; no prior REPO is overwritten.
            assert not REPO.exists(), 'Unexpected research directory appeared'
            shutil.move(str(candidate),str(REPO))
            print('[PASS] Reconstructed exact pinned E02 Git checkout offline',flush=True)
            print('[INFO] Git history is intentionally SHALLOW; commit SHA '
                  'and complete source files are unchanged',flush=True)
    assert ensure_exact(REPO), 'Final pinned repository identity failed'
    print('[PASS] Pinned E02 Git commit:',EXPECTED_COMMIT,flush=True)

    source_map={
        'source/e02_pcl_trainer.py':REPO/'src/rtcsdd/training/e02_pcl_trainer.py',
        'source/phoneme_contrastive.py':REPO/'src/rtcsdd/training/phoneme_contrastive.py',
        'experiment/config.json':OUT/'config.json',
    }
    for arc,target in source_map.items():
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():
            assert sha256_file(target)==meta['file_hashes'][arc], ('Existing source differs',str(target))
        else:
            shutil.copy2(sources[arc],target)
        assert sha256_file(target)==meta['file_hashes'][arc]
    assert sha256_file(source_map['source/e02_pcl_trainer.py'])==EXPECTED_TRAINER
    assert sha256_file(source_map['source/phoneme_contrastive.py'])==EXPECTED_PCL

    for arc,expected in all_entries:
        if arc.startswith('cache/'):
            target=CACHE.joinpath(*PurePosixPath(arc).parts[1:])
        elif arc.startswith('reports/'):
            target=ROOT.joinpath(PurePosixPath(arc).name)
        else:
            continue
        target.parent.mkdir(parents=True,exist_ok=True)
        if target.exists():
            assert sha256_file(target)==expected, ('Existing output differs',str(target))
        else:
            shutil.copy2(sources[arc],target)
        assert sha256_file(target)==expected

HANDOFF_META_PATH.write_text(json.dumps(meta,indent=2)+'\n',encoding='utf-8')
for split,expected in (('train',146),('dev',30)):
    found=sorted((CACHE/split).glob('*.json.gz'))
    assert len(found)==expected, ('Unexpected shard count',split,len(found))
    for path in found:
        sidecar=Path(str(path)+'.sha256')
        assert sidecar.is_file(), ('Missing shard SHA sidecar',path)
        assert sha256_file(path)==sidecar.read_text(encoding='utf-8').strip().split()[0], path
manifest=json.loads((OUT/'config.json').read_text(encoding='utf-8'))
assert manifest['config_fingerprint']==EXPECTED_FINGERPRINT
if (OUT/'last.pt').exists():
    assert sha256_file(OUT/'last.pt')==meta['checkpoint']['sha256'], (
        'Existing local checkpoint differs from handoff. Stop before resume.'
    )
print('[PASS] Two pinned E02 source modules + config.json restored',flush=True)
print('[PASS] All 176 cache shards + sidecars restored and validated',flush=True)
print('[PASS] No training / optimizer updates; Kaggle GPU remains enabled',flush=True)


### 03 — Pinned official RTC-SDD model and isolated CUDA+Fairseq training runtime
**KEEP · GPU stays ON · dependency setup only.** Uses the repository's own `bootstrap_official.py` and `install_training_env.py`. No phoneme preprocessing or new model training.


In [ ]:
import subprocess, sys, shutil, os

BOOTSTRAP=REPO/'scripts/setup/bootstrap_official.py'
INSTALLER=REPO/'scripts/setup/install_training_env.py'
assert BOOTSTRAP.is_file() and INSTALLER.is_file()
setup_env=dict(os.environ)
setup_env['UV_CACHE_DIR']='/tmp/rtcsdd-e02-uv-cache'
setup_env['UV_HTTP_TIMEOUT']='300'
setup_env['PYTHONUNBUFFERED']='1'
setup_env['PYTHONPATH']=str(REPO/'src')+os.pathsep+setup_env.get('PYTHONPATH','')
print('[SETUP] Restore pinned official model repository',flush=True)
subprocess.run([sys.executable,str(BOOTSTRAP)],cwd=REPO,env=setup_env,check=True)
subprocess.run([sys.executable,str(BOOTSTRAP),'--verify-only'],cwd=REPO,env=setup_env,check=True)

uv=shutil.which('uv')
if uv is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    uv=shutil.which('uv')
assert uv,'Missing uv binary after installation'
if not PY310.is_file():
    subprocess.run([uv,'python','install','3.10'],env=setup_env,check=True)
    subprocess.run([uv,'venv','--python','3.10',str(PY310.parent.parent)],
                   env=setup_env,check=True)

probe=r"""
import sys,numpy as np
for k,v in {'float':float,'int':int,'complex':complex,'bool':bool,'object':object,'str':str}.items():
 if k not in np.__dict__:setattr(np,k,v)
import torch,fairseq,rtcsdd
assert sys.version_info[:2]==(3,10)
assert torch.__version__=='2.5.1+cu121',torch.__version__
assert torch.cuda.is_available()
print('[PASS] CUDA',torch.__version__,torch.cuda.get_device_name(0))
"""

def probe_runtime():
    return subprocess.run([str(PY310),'-c',probe],cwd=REPO,env=setup_env,
                          text=True,capture_output=True)
first=probe_runtime()
if first.returncode!=0:
    print('[SETUP] Install source-pinned CUDA PyTorch + Fairseq',flush=True)
    subprocess.run([sys.executable,str(INSTALLER),'--python',str(PY310),'--install'],
                   cwd=REPO,env=setup_env,check=True)
final=probe_runtime()
print(final.stdout,flush=True)
if final.returncode:
    print(final.stderr[-5000:],flush=True)
    raise RuntimeError('Pinned Fairseq+CUDA environment validation failed')
print('[PASS] GPU still enabled; model training not started',flush=True)


### 04 — Download original XLS-R 300M weights and match OLD SHA-256
**KEEP · GPU stays ON · no training.** Only the original Fairseq XLS-R checkpoint is acceptable. File is downloaded from the same official source used in the old notebook, and checked against the SHA computed in the old session and stored in the portable ZIP.


In [ ]:
from pathlib import Path
import requests, re, os, shutil

official_url='https://dl.fbaipublicfiles.com/fairseq/wav2vec/xlsr2_300m.pt'
meta=json.loads(HANDOFF_META_PATH.read_text(encoding='utf-8'))
ref=meta['pretrained']
PRETRAINED.parent.mkdir(parents=True,exist_ok=True)
PART=PRETRAINED.with_name(PRETRAINED.name+'.part')
if not PRETRAINED.is_file():
    print('[DOWNLOAD] Original XLS-R 300M, no E01 fine-tuned weights',flush=True)
    offset=PART.stat().st_size if PART.is_file() else 0
    headers={'Range':f'bytes={offset}-'} if offset else {}
    with requests.get(official_url,stream=True,timeout=(30,180),headers=headers) as resp:
        resp.raise_for_status()
        assert resp.status_code in (200,206)
        assert 'text/html' not in resp.headers.get('Content-Type','').lower()
        if resp.status_code==206:
            match=re.fullmatch(r'bytes\s+(\d+)-(\d+)/(\d+|\*)',resp.headers.get('Content-Range',''))
            if not match or int(match.group(1))!=offset:
                raise RuntimeError('Bad resumed checkpoint HTTP range')
            mode='ab'
            total=int(match.group(3)) if match.group(3)!='*' else None
        else:
            mode='wb'
            total=int(resp.headers['Content-Length']) if resp.headers.get('Content-Length','').isdigit() else None
        with PART.open(mode) as f:
            for chunk in resp.iter_content(chunk_size=8*1024*1024):
                if chunk: f.write(chunk)
            f.flush();os.fsync(f.fileno())
    if total is not None and PART.stat().st_size!=total:
        raise RuntimeError('Original XLS-R incomplete; .part retained for retry')
    assert PART.stat().st_size==ref['size_bytes'], 'Original XLS-R byte size differs from old session'
    assert sha256_file(PART)==ref['sha256'], 'Original XLS-R SHA mismatch; refusing to train'
    os.replace(PART,PRETRAINED)
else:
    print('[FOUND] Existing original XLS-R checkpoint; re-checking',flush=True)
assert PRETRAINED.stat().st_size==ref['size_bytes']
assert sha256_file(PRETRAINED)==ref['sha256'], 'Original XLS-R hash differs from old session'
print('[PASS] Original pretrained XLS-R identical to old session',flush=True)
print('[PASS] GPU remains enabled',flush=True)


### 05 — Restore **E02** step-1 full Adam/RNG checkpoint from PRIVATE HF
**KEEP · GPU stays ON · recovery only.** Verifies the private HF checkpoint SHA256 receipt and resume fingerprint. Starts from step 1 on first launch, or the *newest verified* E02 checkpoint if a later Kaggle session is needed; never restores E01 weights.


In [ ]:
from pathlib import Path
import subprocess,os,json,shutil

try:
    from huggingface_hub import HfApi, hf_hub_download
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30,<1.0'],check=True)
    from huggingface_hub import HfApi,hf_hub_download

from kaggle_secrets import UserSecretsClient
TOKEN=os.environ.get('HF_TOKEN') or UserSecretsClient().get_secret('HF_TOKEN')
assert TOKEN, 'HF_TOKEN Secret missing or Notebook access not enabled'
api=HfApi(token=TOKEN)
assert api.repo_info(HF_REPO,repo_type='model',token=TOKEN).private is True
print('[PASS] Private Hugging Face checkpoint repository accessible',flush=True)

remote=HF_PREFIX+'/last.pt'
remote_receipt=remote+'.sha256.json'
remote_files=set(api.list_repo_files(repo_id=HF_REPO,repo_type='model',token=TOKEN))
assert {remote,remote_receipt}.issubset(remote_files), 'E02 step-1 HF backup is incomplete'
receipt_file=hf_hub_download(repo_id=HF_REPO,repo_type='model',
                             filename=remote_receipt,token=TOKEN,force_download=True)
receipt=json.loads(Path(receipt_file).read_text(encoding='utf-8'))
expected=meta['checkpoint']
assert expected['included_in_zip'] is False
assert isinstance(receipt.get('sha256'),str) and len(receipt['sha256'])==64
assert isinstance(receipt.get('size_bytes'),int) and receipt['size_bytes']>3*1024**3
if receipt['sha256']!=expected['sha256']:
    print('[INFO] HF has a newer checkpoint than the original step-1 handoff; validating continuity',flush=True)
    assert receipt['size_bytes']>=expected['size_bytes']*0.95, 'Remote checkpoint unexpected size'

LAST=OUT/'last.pt'
if LAST.exists():
    assert LAST.stat().st_size==receipt['size_bytes']
    assert sha256_file(LAST)==receipt['sha256']
    print('[PASS] Identical latest HF checkpoint already local',flush=True)
else:
    recovery=ROOT/'e02_hf_recovery'
    recovered=Path(hf_hub_download(repo_id=HF_REPO,repo_type='model',
                                   filename=remote,token=TOKEN,
                                   local_dir=str(recovery)))
    assert recovered.stat().st_size==receipt['size_bytes']
    assert sha256_file(recovered)==receipt['sha256'], 'Downloaded checkpoint SHA mismatch'
    OUT.mkdir(parents=True,exist_ok=True)
    os.replace(recovered,LAST)
    print('[PASS] Restored newest verified E02 checkpoint (Adam + RNG)',flush=True)

inspector=r"""
import torch,sys,json
c=torch.load(sys.argv[1],map_location='cpu',mmap=True,weights_only=False)
print(json.dumps({
 'version':c['format_version'],'epoch':c['epoch'],
 'next_batch':c['next_batch'],'global_step':c['global_step'],
 'config_fingerprint':c['config_fingerprint'],
 'optimizer_present':bool(c['optimizer']),
 'rng_present':bool(c['rng']),
 'accumulated_samples':c['accumulated_samples'],
}))
"""
check=subprocess.run([str(PY310),'-c',inspector,str(LAST)],
                     cwd=REPO,env=setup_env,text=True,capture_output=True,check=True)
loaded=json.loads(check.stdout.strip())
assert loaded['version']==2
assert loaded['config_fingerprint']==EXPECTED_FINGERPRINT
assert loaded['global_step']>=1 and loaded['epoch'] in (0,1)
if loaded['global_step']==1:
    assert loaded['epoch']==0 and loaded['next_batch']==8
assert loaded['optimizer_present'] and loaded['rng_present'] and loaded['accumulated_samples']==0
START_STEP=loaded['global_step']
START_BATCH=loaded['next_batch']
EXPECTED_RESUME_SHA=receipt['sha256']
print('[PASS] E02 checkpoint cursor: epoch_index',loaded['epoch'],
      'global_step',START_STEP,'next_batch',START_BATCH,flush=True)
print('[PASS] No optimizer steps repeated and no weights changed',flush=True)


### 06 — Strict all-pair preflight and GPU/disk readiness
**KEEP · GPU stays ON · no training yet.** Rechecks all 44,590 cached pair hashes and the original E02 config fingerprint. This check runs in the isolated CUDA-compatible Python 3.10 environment.


In [ ]:
import subprocess,shutil,json
TRAINER=REPO/'src/rtcsdd/training/e02_pcl_trainer.py'
PCL=REPO/'src/rtcsdd/training/phoneme_contrastive.py'
BASE_CONFIG=REPO/'src/rtcsdd/configs/p01_xlsr_aasist_official_like_fp32.json'
assert sha256_file(TRAINER)==EXPECTED_TRAINER
assert sha256_file(PCL)==EXPECTED_PCL
assert PRETRAINED.exists() and LAST.is_file() and (OUT/'config.json').is_file()

trainer_args=[
 str(PY310),'-u','-m','rtcsdd.training.e02_pcl_trainer',
 '--repo',str(REPO),'--data',str(DATA),'--cache',str(CACHE),
 '--pretrained',str(PRETRAINED),'--config',str(BASE_CONFIG),
 '--out',str(OUT),'--epochs','1','--lambda-pcl','0.1',
 '--temperature','0.1','--max-matches','96',
]
assert torch.cuda.is_available()
free=shutil.disk_usage(ROOT).free
print(f'[DISK] Available: {free/1024**3:.2f} GiB',flush=True)
assert free > 5.0*1024**3, 'Need >5 GiB free for atomic last.pt replacement'
subprocess.run(trainer_args+['--max-minutes','0','--preflight-only'],
               cwd=REPO,env=setup_env,check=True)
print('[PASS] All 44,590 pair hashes validated by unchanged E02 trainer',flush=True)
print('[PASS] Original exact E02 resume fingerprint is:',EXPECTED_FINGERPRINT,flush=True)
print('[PASS] Ready for one invocation of full Epoch-1 resume on cuda:0',flush=True)


### 07 — Historical Epoch 1 completion · CE + PCL · FP32
**KEEP · GPU training · one trainer invocation.** Do **not** add a one-step smoke cell. Launches exactly one real `--resume` invocation for the remaining microbatches of Epoch 1, then the *full* clean Dev evaluation. The original trainer saves/backs up checkpoints at its original cadence and at completion.

The runtime computes **`SAFE_TRAINING_LIMIT_MINUTES` dynamically** to use nearly the whole fresh Kaggle session, while reserving about 50 minutes for Dev evaluation, checkpoint uploads, and an unknown head start to the Kaggle session clock. This is a **last-resort safeguard**, not a one-step or 100-minute cap. Set it to `0` ONLY if you deliberately want no safety stop (Kaggle can terminate without a final save). **No code here can extend Kaggle's hard session limit.**


In [ ]:
import subprocess,os,json,time,shutil

# One-shot: use as much of the fresh 12-hour Kaggle session as is safe.
# The Kaggle clock may begin before the first cell; reserve 50 minutes for
# this uncertainty, full Dev evaluation, and a final HF upload.
SETUP_ELAPSED_MINUTES=(time.monotonic()-RUN_STARTED_MONOTONIC)/60
SAFE_TRAINING_LIMIT_MINUTES=max(1.0,720.0-SETUP_ELAPSED_MINUTES-50.0)
assert SAFE_TRAINING_LIMIT_MINUTES>0
assert torch.cuda.is_available()
assert sha256_file(TRAINER)==EXPECTED_TRAINER
assert sha256_file(PCL)==EXPECTED_PCL
assert sha256_file(LAST)==EXPECTED_RESUME_SHA, 'Checkpoint changed since restore; inspect before restarting'

# The existing training backup manager reads this token. Never write it to a file.
train_env=dict(setup_env)
train_env['HF_TOKEN']=TOKEN
train_env['PYTHONUNBUFFERED']='1'
LOG=ROOT/'e02_epoch1_one_shot.log'
if LOG.exists():
    raise FileExistsError('This launch cell was already run. Inspect checkpoint before relaunching.')
cmd=trainer_args+[
    '--max-minutes',str(SAFE_TRAINING_LIMIT_MINUTES),
    '--resume',
]
print('='*74,flush=True)
print('E02 — RESUME STEP 1 -> END OF EPOCH 1 (CE + 0.1*PCL)',flush=True)
print('GPU:',torch.cuda.get_device_name(0),'| single-GPU cuda:0',flush=True)
print('Training source SHA256:',EXPECTED_TRAINER,flush=True)
print('Checkpoint starting cursor: step=',START_STEP,'next_batch=',START_BATCH,flush=True)
print('Safe wall-time guard (training minutes):',SAFE_TRAINING_LIMIT_MINUTES,flush=True)
print('Periodic HF backup: configured by the UNCHANGED E02 trainer',flush=True)
print('='*74,flush=True)
started=time.monotonic()
with LOG.open('w',encoding='utf-8') as sink:
    p=subprocess.Popen(cmd,cwd=REPO,env=train_env,text=True,
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
    assert p.stdout is not None
    for line in p.stdout:
        print(line,end='',flush=True)
        sink.write(line)
        sink.flush()
    status=p.wait()
if status!=0:
    raise RuntimeError(f'E02 trainer failed with code {status}. Log: {LOG}. '
                       'Do not delete existing last.pt or restart without --resume.')
print('[PASS] Trainer exited normally; verifying status and HF backups next',flush=True)
print('[INFO] Trainer elapsed training+evaluation minutes:',round((time.monotonic()-started)/60,2),flush=True)


### 08 — Verify completed epoch, full Dev metrics and private HF backup
**KEEP · GPU remains ON · verification.** Reports **COMPLETED** only when the checkpoint has advanced to epoch index 1 and `dev_epoch_001.json` exists. Otherwise reports **SAFELY PAUSED**, not a finished epoch.


In [ ]:
import json,subprocess,os

verify_code=r"""
import json,sys,torch
c=torch.load(sys.argv[1],map_location='cpu',mmap=True,weights_only=False)
print(json.dumps({
 'epoch_index':c['epoch'],'global_step':c['global_step'],
 'next_batch':c['next_batch'],'fingerprint':c['config_fingerprint'],
 'optimizer_present':bool(c['optimizer']),'rng_present':bool(c['rng']),
 'accumulated_samples':c['accumulated_samples'],
}))
"""
result=subprocess.run([str(PY310),'-c',verify_code,str(LAST)],
                      cwd=REPO,env=setup_env,text=True,capture_output=True,check=True)
latest=json.loads(result.stdout.strip())
assert latest['fingerprint']==EXPECTED_FINGERPRINT
assert latest['global_step']>=START_STEP and latest['optimizer_present'] and latest['rng_present']
assert latest['accumulated_samples']==0
completed=(latest['epoch_index']>=1)
metrics_file=OUT/'dev_epoch_001.json'
if completed:
    assert metrics_file.is_file(),'Checkpoint shows epoch done but Dev metrics missing'
    metrics=json.loads(metrics_file.read_text())
    print('[PASS] FULL E02 EPOCH 1 COMPLETED',flush=True)
    print('[DEV] weighted loss:',metrics['dev_weighted_loss'],flush=True)
    print('[DEV] online macro-F1:',metrics['dev_online_macro_f1'],flush=True)
    print('[PASS] Best checkpoint files:',
          (OUT/'best_dev_loss.pt').is_file(),(OUT/'best_online_f1.pt').is_file(),flush=True)
else:
    print('[SAFE PAUSE] E02 Epoch 1 was not finished within time budget',flush=True)

remote_receipt_path=HF_PREFIX+'/last.pt.sha256.json'
rp=hf_hub_download(repo_id=HF_REPO,repo_type='model',filename=remote_receipt_path,
                   token=TOKEN,force_download=True)
rec=json.loads(Path(rp).read_text(encoding='utf-8'))
local_sha=sha256_file(LAST)
backup_verified=(rec.get('sha256')==local_sha and
                 rec.get('size_bytes')==LAST.stat().st_size)
print('Latest global step:',latest['global_step'],flush=True)
print('Next microbatch:',latest['next_batch'],flush=True)
print('Epoch 1 complete:',completed,flush=True)
print('Latest last.pt HF receipt VERIFIED:',backup_verified,flush=True)
summary={
 'epoch1_completed':completed,'resume_cursor':latest,
 'hf_latest_last_pt_verified':backup_verified,
 'local_sha256':local_sha,'remote_receipt_sha256':rec.get('sha256'),
 'dev_metrics_path':str(metrics_file) if completed else None,
 'training_log':str(LOG),
}
(ROOT/'e02_epoch1_one_shot_summary.json').write_text(json.dumps(summary,indent=2)+'\n')
if not backup_verified:
    raise RuntimeError('Latest local checkpoint is NOT verified on HF. '
                       'Do not close this session before recovering backup.')
if not completed:
    print('[NEXT SESSION] Use --resume with the newest HF checkpoint; DO NOT start fresh.',flush=True)
print('[REPORT]',ROOT/'e02_epoch1_one_shot_summary.json',flush=True)


---

## Final research audit — completed, no further inference

The follow-up read-only audit independently recomputed precision, recall,
per-class F1, accuracy and Macro-F1 from the **saved confusion counts** for all
nine Dev subgroups. All arithmetic checks and group-partition sums passed.
This audits the report's arithmetic; it is not independent model inference.

### Full Dev confusion matrix (threshold `0.5`)

| True / predicted | Spoof | Bonafide |
|---|---:|---:|
| Spoof | 5,595 | 6,760 |
| Bonafide | 1,197 | 1,652 |

- Total Dev: **15,204**; accuracy: **0.476651**; Macro-F1: **0.438914**.
- Spoof precision/recall/F1: **0.823763 / 0.452853 / 0.584426**.
- Bonafide precision/recall/F1: **0.196386 / 0.579853 / 0.293402**.
- Balanced accuracy, computed from class recalls: **0.516353**.
- Official hidden-set weighted F1: **not available**, and must not be inferred from Dev.

### Next planned controlled experiment

**E03 — CE + PCL + Noise Augmentation** is *planned*, **not run**. Hold the
E02 architecture, initialization, data, paired sampling, loss weight, seed,
precision and evaluation procedure fixed, while adding a documented permitted
non-speech noise augmentation. Record augmentation source, SNR distribution,
probability and RNG seed. Verify the competition's current dataset-use rules
before enabling any external noise input. Keep E03 checkpoint and outputs in
a distinct directory and HF prefix.

### GitHub hygiene

Before pushing source, check whether the two handoff-restored modules
`src/rtcsdd/training/e02_pcl_trainer.py` and
`src/rtcsdd/training/phoneme_contrastive.py` are untracked.
Also review the E02 resolved `config.json` and a small aggregate Dev summary.
Do not push private audio, per-ID caches, GPU environments, or `.pt` files.
Update `docs/experiments.md` to track **E01, E02 and planned E03 only**.

This archived notebook is intentionally **output-free**. For complete experiment
evidence, use the recorded SHA-verified private artifacts and Git commit IDs.
